# A10: Data Understanding on the unified dataset (Tanner Edwards)

Personal notebook. Re-runs Matt's Data Understanding (`Sorensen-A10.ipynb` §2–§3, plan steps 8–9) on the **unified dataset** built by `00_build_dataset.ipynb`, which adds four sources and grows the usable population from 5,766 to about 6,800 films. Same helpers, same review flags, same privacy rule.

**Run `00_build_dataset.ipynb` first.** This notebook only reads `data/clean/films_unified.csv` and the raw files; it writes nothing.

| Section | What | Matt's equivalent |
|---|---|---|
| 1 | Setup and Matt's profile helpers | §1, §3 helpers |
| 2 | Data Receipt for the **new** sources | §2 |
| 3a | Field inventory: all columns, role at the financing decision | §3a |
| 3b | The label on the expanded population | §3b |
| 3c–3f | Numeric, categorical, identifier/text and date fields | §3c–§3h |
| 3g | **New:** do the sources agree with each other? | — |
| 3h | **New:** who has which data (coverage by period and budget tier) | §4a idea |
| 3i | Flag register | §3i |
| 3j–3k | Findings and proposed dispositions (need Tanner + Matt sign-off) | §3j–§3k |

This section **observes and records; it doesn't clean.**

## 1. Setup

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
YEARS = (2000, 2023)

films = pd.read_csv(CLEAN / "films_unified.csv", low_memory=False)
for c in ["release_dt", "tn_Release Date", "bom_first_date", "sl_original_release_date",
          "rt_releaseDateTheaters", "rt_releaseDateStreaming", "rtrev_first_review", "slrev_first_review"]:
    films[c] = pd.to_datetime(films[c], errors="coerce")
print(f"films_unified.csv: {len(films):,} films x {films.shape[1]} columns; {films['usable'].sum():,} usable")

Matt's chart style and profile helpers, copied unchanged from `Sorensen-A10.ipynb` §3 so the two profiles are directly comparable.

In [ ]:
SERIES = {"train 2000-15": "#2a78d6", "test 2016-19": "#eb6834", "stress 2020-23": "#1baf7a"}
BLUE, BLUE_LIGHT, INK, INK2 = "#2a78d6", "#cde2fb", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ad", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "axes.titlesize": 10, "axes.titlelocation": "left", "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
})

SKEW_FLAG, RARE_FLAG = 0.5, 0.05   # Ch 6 review flags
FLAGS = []

def flag(file, field, issue, evidence):
    FLAGS.append({"file": file, "field": field, "issue": issue, "evidence": evidence})


def numeric_profile(df, cols, zero_as_missing=()):
    rows = []
    for c in cols:
        s = df[c].astype(float)
        if c in zero_as_missing:
            s = s.mask(s == 0)
        x = s.dropna()
        q1, q3 = x.quantile([0.25, 0.75])
        iqr = q3 - q1
        rows.append({
            "field": c, "n": len(x), "% missing": round(100 * s.isna().mean(), 1),
            "mean": x.mean(), "median": x.median(), "std": x.std(),
            "min": x.min(), "Q1": q1, "Q3": q3, "max": x.max(), "IQR": iqr, "skew": x.skew(),
            "# above Q3+1.5 IQR": int((x > q3 + 1.5 * iqr).sum()),
            "# below Q1-1.5 IQR": int((x < q1 - 1.5 * iqr).sum()),
        })
    out = pd.DataFrame(rows).set_index("field")
    out["skew flag"] = out["skew"].abs() > SKEW_FLAG
    return out


def category_profile(s, name, multi=False, names_hidden=False):
    n = s.notna().sum()
    vals = s.dropna().astype(str)
    if multi:
        vals = vals.str.split(",").explode().str.strip()
        vals = vals[vals.ne("")]
    vc = vals.value_counts()
    share = vc / n
    row = {
        "field": name, "% missing": round(100 * s.isna().mean(), 1), "multi-label": multi,
        "levels": len(vc),
        "top level": "(person, withheld)" if names_hidden else (vc.index[0] if len(vc) else None),
        "top level % of rows": round(100 * share.iloc[0], 1) if len(vc) else None,
        "levels < 5% of rows": int((share < RARE_FLAG).sum()),
        "% of rows in a level < 5%": None if multi else round(100 * share[share < RARE_FLAG].sum(), 1),
    }
    return row, vc


def id_text_profile(df, cols):
    rows = []
    for c in cols:
        s = df[c]
        nn = s.dropna().astype(str)
        rows.append({
            "field": c, "% missing": round(100 * s.isna().mean(), 1),
            "unique values": nn.nunique(),
            "rows sharing a value": int(nn.duplicated(keep=False).sum()),
            "median length (chars)": nn.str.len().median(),
        })
    return pd.DataFrame(rows).set_index("field")


def box_hist(ax_box, ax_hist, x, title, xlabel, ylabel, log=False):
    x = x.dropna().astype(float)
    if log and (x <= 0).any():
        title += f"\n({(x <= 0).sum():,} values <= 0 not shown on the log scale)"
        x = x[x > 0]
    bins = int(np.ceil(len(x) ** (1 / 3)))
    edges = np.logspace(np.log10(x.min()), np.log10(x.max()), bins + 1) if log else bins
    ax_hist.hist(x, bins=edges, color=BLUE, edgecolor="white", linewidth=0.4)
    ax_box.boxplot(x, orientation="horizontal", widths=0.6, patch_artist=True,
                   boxprops=dict(facecolor=BLUE_LIGHT, edgecolor=BLUE),
                   medianprops=dict(color=INK, linewidth=1.5),
                   whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE),
                   flierprops=dict(marker="o", markersize=2.5, markerfacecolor=BLUE,
                                   markeredgecolor="none", alpha=0.35))
    ax_hist.axvline(x.median(), color=INK, lw=1, label=f"median {x.median():,.3g}")
    ax_hist.axvline(x.mean(), color=INK, lw=1, ls="--", label=f"mean {x.mean():,.3g}")
    if log:
        ax_hist.set_xscale("log")
    ax_box.set_yticks([])
    ax_box.grid(False)
    ax_box.set_title(title)
    ax_hist.set_xlabel(xlabel)
    ax_hist.set_ylabel(ylabel)
    ax_hist.legend(fontsize=7, frameon=False)


def profile_grid(specs, ylabel, ncols=3):
    nrows = int(np.ceil(len(specs) / ncols))
    fig = plt.figure(figsize=(4.3 * ncols, 3.1 * nrows), layout="constrained")
    subfigs = np.atleast_1d(fig.subfigures(nrows, ncols)).ravel()
    for sf, (x, title, xlabel, log) in zip(subfigs, specs):
        ax_box, ax_hist = sf.subplots(2, 1, sharex=True, height_ratios=[1, 4])
        box_hist(ax_box, ax_hist, x, title, xlabel, ylabel, log)
    for sf in subfigs[len(specs):]:
        sf.set_visible(False)
    plt.show()


def count_plot(ax, vc, n_rows, title, top=10, xlabel="% of rows"):
    shares = (100 * vc / n_rows).head(top)[::-1]
    ax.barh(shares.index.astype(str).str.slice(0, 35), shares.values, color=BLUE, height=0.7)
    ax.axvline(100 * RARE_FLAG, color=INK2, ls=":", lw=1)
    for y, v in enumerate(shares.values):
        ax.text(v, y, f" {v:.1f}%", va="center", fontsize=7, color=INK2)
    ax.set_title(title + (f"  (top {top} of {len(vc):,})" if len(vc) > top else ""))
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)

print("Profile helpers defined.")

## 2. Data Receipt for the new sources (plan step 8)

Matt's §2 covers TMDB, RT movies (andrezaza) and RT reviews. These are the sources added since. Licenses were read from the Kaggle API (`licenseName`) and the GitHub API on 2026-11-23.

| Field | The Numbers | Box Office Mojo | Wikidata snapshot | RT movies (stefanoleone992) | RT critic reviews (stefanoleone992) |
|---|---|---|---|---|---|
| **File** | `the_numbers/Top Movies (Cleaned Data).csv` | `boxofficemojo/revenues_per_day.csv.gz` | `wikidata/wikidata_film_financials_2026-11-21.csv` | `stefanoleone992/rotten_tomatoes_movies.csv` | `stefanoleone992/rotten_tomatoes_critic_reviews.csv` |
| **Publisher** | michaelmatta0 (Kaggle) | tjwaterman99 (GitHub release) | Our own SPARQL query (`tools/wikidata_films.rq`) | stefanoleone992 (Kaggle) | same |
| **Original source** | the-numbers.com (Nash Information Services) | boxofficemojo.com (IMDb) | Wikidata (Wikimedia, crowd-edited) | Rotten Tomatoes | Rotten Tomatoes |
| **URL** | kaggle.com/datasets/michaelmatta0/movies-ultimate-metrics-features-and-metadata | github.com/tjwaterman99/boxofficemojo-scraper | query.wikidata.org | kaggle.com/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset | same |
| **Access date** | 2026-11-21 | 2026-11-20 | 2026-11-21 (snapshot committed to git) | 2026-11-20 | 2026-11-20 |
| **License (as stated)** | MIT | **None stated** | CC0 1.0 | CC0 1.0 | CC0 1.0 |
| **Claimed grain** | One row per film | One row per film **per day** | One row per film with a USD budget and a TMDB id | One row per RT film (link) | One row per critic review |
| **Population** | ~6.6k "top" films by budget, all years | Films on the US daily chart, 2000–2025 | Wikidata films 2000–2023 with a USD cost | ~17.7k RT films (2020 scrape) | Reviews of those films |
| **Sensitive fields** | None | None | None | `directors`, `authors`, `actors`: public professionals | `critic_name`, `review_content`: named people and copyrighted text |
| **Rights caveats** | Scraped; The Numbers' terms aren't overridden by the uploader's MIT | Scraped; no license, so academic fair use only | — | Scores belong to RT | Review text is copyrighted; not copied into the unified file |
| **Permitted use here** | Academic, non-commercial. Cite the-numbers.com. | Same, and cite Box Office Mojo | Cite Wikidata | Same as Matt's RT files | Same; counts only |

**Also note:** `get_data.py` downloads `rotten_tomatoes_movie_reviews.csv` from **priyamchoksi** (license MIT), while Matt's receipt cites the **andrezaza** upload (CC0). Same row count (1,444,963). The receipt should cite one; open question for Matt.

### 2a. File fingerprints
SHA-256 for every raw file, Matt's originals included, so all of them can be checked as unchanged at submission.

In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

FILES = sorted(p for p in RAW.rglob("*") if p.is_file() and p.name != ".gitkeep")
fingerprints = pd.DataFrame([{"file": str(p.relative_to(RAW)), "MB": round(p.stat().st_size / 1e6, 1),
                              "sha256": sha256(p)} for p in FILES])
with pd.option_context("display.max_colwidth", 70):
    display(fingerprints)

### 2b. Grain check (new sources)
Does one row equal what the source claims? Box Office Mojo's `id` turned out to be per row, not per film (it was collapsed to one row per title in the build).

In [ ]:
tn_raw = pd.read_csv(RAW / "the_numbers/Top Movies (Cleaned Data).csv", low_memory=False)
bom_raw = pd.read_csv(RAW / "boxofficemojo/revenues_per_day.csv.gz")
wd_raw = pd.read_csv(RAW / "wikidata/wikidata_film_financials_2026-11-21.csv")
sl_raw = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_movies.csv", low_memory=False)
slr_raw = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_critic_reviews.csv", low_memory=False,
                      usecols=["rotten_tomatoes_link", "critic_name", "review_date", "review_type", "top_critic"])

def grain_check(df, key):
    return {"rows": len(df), "key": key, "unique keys": df[key].nunique(),
            "rows with a repeated key": int(df[key].duplicated(keep=False).sum()),
            "exact duplicate rows": int(df.duplicated().sum())}

grain = pd.DataFrame({
    "The Numbers (Movie URL)": grain_check(tn_raw, "Movie URL"),
    "Box Office Mojo (id)": grain_check(bom_raw, "id"),
    "Box Office Mojo (title + date)": grain_check(bom_raw.assign(k=bom_raw["title"] + "|" + bom_raw["date"]), "k"),
    "Wikidata (tmdb)": grain_check(wd_raw, "tmdb"),
    "RT sl movies (link)": grain_check(sl_raw, "rotten_tomatoes_link"),
    "RT sl reviews (link + critic + date)": grain_check(
        slr_raw.assign(k=slr_raw["rotten_tomatoes_link"] + "|" + slr_raw["critic_name"].fillna("?") + "|" + slr_raw["review_date"].astype(str)), "k"),
})
grain

### 2c. Date coverage and placeholder dates (new sources)

In [ ]:
dates = {
    "The Numbers Release Date": pd.to_datetime(tn_raw["Release Date"], errors="coerce"),
    "Box Office Mojo date": pd.to_datetime(bom_raw["date"], errors="coerce"),
    "RT sl original_release_date": pd.to_datetime(sl_raw["original_release_date"], errors="coerce"),
    "RT sl review_date": pd.to_datetime(slr_raw["review_date"], errors="coerce"),
}
ACCESS = pd.Timestamp("2026-11-21")
pd.DataFrame({n: {"min": d.min().date(), "max": d.max().date(), "% missing": round(100 * d.isna().mean(), 1),
                  "before 1900": int((d.dt.year < 1900).sum()), "after 2023": int((d.dt.year > 2023).sum()),
                  "after access date": int((d > ACCESS).sum())} for n, d in dates.items()}).T

### 2d. Population: where the usable films come from

Same funnel as Matt's §2f, but on the unified table, with the source of each film's budget and revenue. The label period split is the contract's.

In [ ]:
f = films.copy()
f["period"] = pd.cut(f["year"], [1999, 2015, 2019, 2023], labels=list(SERIES))
steps = [
    ("2000-2023 film with any budget or revenue (unified table)", pd.Series(True, index=f.index)),
    ("budget >= $1M (any source)", f["fin_budget"].ge(1_000_000)),
    ("revenue > 0 (any source)", f["fin_revenue"].gt(0)),
]
mask = pd.Series(True, index=f.index)
funnel = []
for label, cond in steps:
    mask &= cond
    funnel.append({"filter (cumulative)": label, "films remaining": int(mask.sum())})
display(pd.DataFrame(funnel))
assert mask.sum() == f["usable"].sum()

u = f[f["usable"]].copy()
display(pd.crosstab(u["fin_budget_src"], u["fin_revenue_src"], margins=True).rename_axis(index="budget from", columns="revenue from"))
display(pd.crosstab(u["period"], u["in_tmdb"].map({True: "TMDB film", False: "The Numbers only"}), margins=True))

# S4 restated on the unified table: budget >= $1M but no revenue from any source
b1 = f["fin_budget"].ge(1_000_000)
s4 = pd.DataFrame({p: {"budget >= $1M": int((b1 & (f["period"] == p)).sum()),
                       "with revenue": int((b1 & f["fin_revenue"].gt(0) & (f["period"] == p)).sum())}
                   for p in SERIES}).T
s4["% missing revenue"] = (100 * (1 - s4["with revenue"] / s4["budget >= $1M"])).round(1)
display(s4)
flag("Unified", "fin_revenue", "missing (S4)",
     f"{100 * (1 - (b1 & f['fin_revenue'].gt(0)).sum() / b1.sum()):.1f}% of budget >= $1M films have no revenue "
     f"from any source (Matt's TMDB-only figure: 43.6%)")

## 3. Univariate profile of every field (plan step 9)

**Which rows:** the usable films (2000–2023, budget ≥ $1M, revenue > 0 from any source), the population the screen will score. The label is shown by period. **Review flags (Ch 6):** |skew| > 0.5; any category level held by < 5% of non-missing rows. **Privacy:** director, writer, actor and critic names are profiled by counts only.

In [ ]:
pop = u
pop["multiple"] = pop["fin_revenue"] / pop["fin_budget"]
pop["log10_multiple"] = np.log10(pop["multiple"])
for k in [1, 2, 2.5, 3]:
    pop[f"loser_{k}x"] = pop["multiple"] < k
pop["money_loser"] = pop["loser_2.5x"]
pop["release_month"] = pop["release_dt"].dt.month
pop["n_genres"] = pop["genres"].str.split(",").str.len()
print(f"Population: {len(pop):,} films")
print(pop["period"].value_counts().sort_index().to_string())

### 3a. Field inventory
Every column in the unified file, with its source and what the contract lets it do at the financing decision.
- **allowed input**: on the contract's input list (budget, genre, director/company track record, runtime, release month/quarter, language, country)
- **candidate input (needs amendment)**: known before production, but not on the contract's list (e.g. MPAA rating, source material, franchise, writer, cast). Using it needs a dated amendment.
- **prohibited (post-release)**, **outcome part**, **key / join**, **build metadata**, **not used**

In [ ]:
A, CAND, P, K, O, B, N = ("allowed input", "candidate input (needs amendment)", "prohibited (post-release)",
                         "key / join", "outcome part (post-release)", "build metadata", "not used")

def source_of(c):
    for pre, name in [("tn_", "The Numbers"), ("bom_", "Box Office Mojo"), ("sl_", "RT sl movies"),
                      ("wd_", "Wikidata"), ("rtrev_", "RT reviews (rollup)"), ("slrev_", "RT sl reviews (rollup)"),
                      ("rt_", "RT movies"), ("fin_", "Combined")]:
        if c.startswith(pre):
            return name
    return "Build" if c in ("film_uid", "usable", "in_tmdb", "merged_ids", "release_dt", "year") else "TMDB"

ROLE = {
    # TMDB
    "film_uid": K, "id": K, "title": K, "original_title": K, "imdb_id": K, "release_date": A, "release_dt": A, "year": A,
    "budget": A, "revenue": O, "runtime": A, "genres": A, "production_companies": A, "production_countries": A,
    "original_language": A, "spoken_languages": N, "keywords": P, "vote_average": P, "vote_count": P,
    "popularity": P, "status": B, "adult": B, "overview": N, "tagline": N, "homepage": N, "poster_path": N,
    "backdrop_path": N,
    # combined / build
    "fin_budget": A, "fin_revenue": O, "fin_budget_src": B, "fin_revenue_src": B, "usable": B, "in_tmdb": B,
    "merged_ids": B,
    # The Numbers
    "tn_Movie Name": K, "tn_Release Date": A, "tn_Production Budget (USD)": A, "tn_Running Time (minutes)": A,
    "tn_Genre": A, "tn_Production/Financing Companies": A, "tn_Production Countries": A, "tn_Languages": A,
    "tn_MPAA Rating": CAND, "tn_Source": CAND, "tn_Creative Type": CAND, "tn_Production Method": CAND,
    "tn_Franchise": CAND, "tn_Keywords": P, "tn_Movie URL": K, "tn_year_diff": B,
    "tn_Domestic Releases": P, "tn_International Releases": P, "tn_Theater counts": P, "tn_Video Release": P,
    "tn_Legs": P, "tn_Opening Weekend (USD)": P, "tn_Domestic Share Percentage": P,
    "tn_Est. Domestic DVD Sales (USD)": P, "tn_Est. Domestic Blu-ray Sales (USD)": P,
    "tn_Total Est. Domestic Video Sales (USD)": P,
    # Box Office Mojo
    "bom_title": K, "bom_year_diff": B, "bom_first_date": P, "bom_distributor": CAND,
    # RT sl movies
    "sl_rotten_tomatoes_link": K, "sl_movie_title": K, "sl_year_diff": B, "sl_genres": A, "sl_runtime": A,
    "sl_production_company": A, "sl_directors": A, "sl_authors": CAND, "sl_actors": CAND,
    "sl_content_rating": CAND, "sl_original_release_date": A, "sl_movie_info": N, "sl_critics_consensus": P,
    "sl_streaming_release_date": P,
    # RT movies (andrezaza)
    "rt_id": K, "rt_title": K, "rt_matched_on": B, "rt_rule": B, "rt_year_diff": B, "rt_runtime_diff": B,
    "rt_director": A, "rt_writer": CAND, "rt_rating": CAND, "rt_ratingContents": CAND, "rt_genre": A,
    "rt_originalLanguage": A, "rt_runtimeMinutes": A, "rt_releaseDateTheaters": A,
    "rt_releaseDateStreaming": P, "rt_distributor": CAND, "rt_soundMix": N, "rt_boxOffice": O,
    "rt_boxOffice_usd": O,
    # Wikidata
    "wd_item": K, "wd_title": K, "wd_year": A, "wd_wd_budget_usd": A, "wd_wd_box_office_usd": O,
}
def role_of(c):
    if c in ROLE:
        return ROLE[c]
    if c.startswith(("tn_Domestic", "tn_International", "tn_Worldwide", "tn_Infl", "bom_domestic", "bom_days", "bom_max")):
        return O if "Gross" in c or "Box Office" in c or "gross" in c else P
    if c.startswith(("sl_tomatometer", "sl_audience", "rt_audience", "rt_tomato", "rtrev_", "slrev_")):
        return P
    return None

inventory = pd.DataFrame({"field": films.columns})
inventory["source"] = inventory["field"].map(source_of)
inventory["dtype"] = inventory["field"].map(lambda c: str(films[c].dtype))
inventory["% filled (usable films)"] = inventory["field"].map(lambda c: round(100 * pop[c].notna().mean(), 1))
inventory["role at financing decision"] = inventory["field"].map(role_of)
assert inventory["role at financing decision"].notna().all(), inventory.loc[inventory["role at financing decision"].isna(), "field"].tolist()
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 60):
    display(inventory)
pd.crosstab(inventory["source"], inventory["role at financing decision"], margins=True)

### 3b. The label on the expanded population
Money-loser = combined revenue < 2.5 × combined budget, with 1×, 2×, 3× sensitivity. Compared with Matt's TMDB-only figures (64.6% losers overall), and split by where the financials came from.

In [ ]:
def label_summary(g):
    return pd.Series({
        "films": len(g), "median multiple": g["multiple"].median(),
        "Q1": g["multiple"].quantile(0.25), "Q3": g["multiple"].quantile(0.75),
        "skew (log10 multiple)": g["log10_multiple"].skew(),
        **{f"% losers at {k}x": 100 * g[f"loser_{k}x"].mean() for k in [1, 2, 2.5, 3]},
    })

label_tbl = pd.DataFrame({p: label_summary(g) for p, g in pop.groupby("period", observed=True)}
                         | {"all 2000-23": label_summary(pop)}).T
display(label_tbl.round(2))

pop["fin_source"] = np.where(pop["fin_budget_src"].eq("tmdb") & pop["fin_revenue_src"].eq("tmdb"),
                             "TMDB only (Matt's population)", "uses The Numbers / Wikidata")
display(pd.DataFrame({k: label_summary(g) for k, g in pop.groupby("fin_source")}).T.round(2))

extremes = pd.Series({
    "multiple < 0.01": (pop["multiple"] < 0.01).sum(), "multiple < 0.1": (pop["multiple"] < 0.1).sum(),
    "multiple > 20": (pop["multiple"] > 20).sum(), "multiple > 100": (pop["multiple"] > 100).sum(),
}, name="films")
display(extremes.to_frame())
cols = ["title", "year", "fin_budget", "fin_revenue", "fin_budget_src", "fin_revenue_src", "multiple"]
print("Lowest 5 multiples:"); display(pop.nsmallest(5, "multiple")[cols])
print("Highest 5 multiples:"); display(pop.nlargest(5, "multiple")[cols])
loser_rate = label_tbl.loc["all 2000-23", "% losers at 2.5x"]
flag("Unified", "multiple", "extreme values",
     f"{extremes.iloc[0]} films below 0.01x and {extremes.iloc[3]} above 100x: check for unit errors")
flag("Unified", "money_loser", "majority class",
     f"{loser_rate:.1f}% losers at 2.5x: a 20% flag can catch at most {20 / loser_rate:.0%} of them")

edges = np.logspace(-3, 3, 37)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), layout="constrained", sharey=True)
for ax, (p, g) in zip(axes, pop.groupby("period", observed=True)):
    x = g["multiple"].clip(edges[0], edges[-1])
    ax.hist(x, bins=edges, weights=np.full(len(x), 100 / len(x)), color=SERIES[p], edgecolor="white", linewidth=0.4)
    ax.axvline(2.5, color=INK, lw=1, ls="--")
    ax.set_xscale("log")
    ax.set_title(f"{p}: n = {len(g):,}, {100 * g['money_loser'].mean():.0f}% losers")
    ax.set_xlabel("revenue / budget (log scale)")
axes[0].set_ylabel("% of films in period")
fig.suptitle("Label on the unified population (dashed line = 2.5x cut-off)", x=0.01, ha="left", fontsize=11)
plt.show()

### 3c. Numeric fields
Money and count fields on log scales. Runtime 0 = unknown. Post-release fields are profiled for quality only.

In [ ]:
num_cols = [c for c in films.columns if pd.api.types.is_numeric_dtype(films[c])
            and c not in ("id",) and not c.endswith("_year_diff") and c not in ("rt_runtime_diff",)
            and films[c].dtype != bool]
zero_missing = ("runtime", "tn_Running Time (minutes)", "sl_runtime", "rt_runtimeMinutes", "budget", "revenue")
prof_num = numeric_profile(pop, num_cols + ["multiple", "log10_multiple", "n_genres"], zero_as_missing=zero_missing)
prof_num.insert(0, "role", prof_num.index.map(lambda c: role_of(c) or "derived"))
with pd.option_context("display.float_format", "{:,.2f}".format, "display.max_rows", 80):
    display(prof_num)

checks = pd.Series({
    "runtime (TMDB) = 0 or missing": int(pop["runtime"].fillna(0).eq(0).sum()),
    "runtime < 60 min": int(pop["runtime"].between(1, 59).sum()),
    "runtime > 240 min": int((pop["runtime"] > 240).sum()),
    "fin_budget > $400M": int((pop["fin_budget"] > 400e6).sum()),
    "fin_budget is a round $1M multiple": int((pop["fin_budget"] % 1_000_000 == 0).sum()),
    "fin_revenue < $1,000": int((pop["fin_revenue"] < 1_000).sum()),
    "fin_revenue < $100,000": int((pop["fin_revenue"] < 100_000).sum()),
    "scores outside 0-100 (RT / sl)": int(sum((~pop[c].dropna().between(0, 100)).sum() for c in
                                              ["rt_audienceScore", "rt_tomatoMeter", "sl_tomatometer_rating", "sl_audience_rating"])),
}, name="films")
display(checks.to_frame())
flag("Unified", "fin_revenue", "implausible range",
     f"{checks['fin_revenue < $1,000']} films under $1,000; {checks['fin_revenue < $100,000']} under $100,000")
flag("Unified", "fin_budget", "estimate, not audited",
     f"{checks['fin_budget is a round $1M multiple']:,} of {len(pop):,} budgets are round $1M figures")
flag("Unified", "runtime", "implausible range",
     f"{checks['runtime (TMDB) = 0 or missing']} zero/missing, {checks['runtime < 60 min']} under 60 min, "
     f"{checks['runtime > 240 min']} over 240 min")

profile_grid([
    (pop["fin_budget"], "fin_budget (combined)", "US$ (log scale)", True),
    (pop["fin_revenue"], "fin_revenue (combined; outcome part)", "US$ (log scale)", True),
    (pop["runtime"].mask(pop["runtime"].eq(0)), "runtime (TMDB; 0 excluded)", "minutes", False),
    (pop["rt_tomatoMeter"], "rt_tomatoMeter (post-release)", "score, 0-100", False),
    (pop["rt_audienceScore"], "rt_audienceScore (post-release)", "score, 0-100", False),
    (pop["rtrev_n_reviews"], "rtrev_n_reviews (post-release)", "reviews (log scale)", True),
], ylabel="films")

### 3d. Categorical fields
Multi-label fields are split on commas, so shares add to more than 100%. Person fields (director, writer, actors) show counts only.

In [ ]:
cat_specs = [
    # (column, multi-label, names hidden)
    ("original_language", False, False), ("genres", True, False), ("production_countries", True, False),
    ("production_companies", True, False), ("spoken_languages", True, False), ("keywords", True, False),
    ("tn_MPAA Rating", False, False), ("tn_Source", False, False), ("tn_Genre", False, False),
    ("tn_Creative Type", False, False), ("tn_Production Method", True, False), ("tn_Franchise", False, False),
    ("tn_Production/Financing Companies", True, False), ("tn_Production Countries", True, False),
    ("tn_Languages", True, False), ("tn_Keywords", True, False),
    ("bom_distributor", False, False),
    ("sl_content_rating", False, False), ("sl_genres", True, False), ("sl_production_company", False, False),
    ("sl_tomatometer_status", False, False), ("sl_audience_status", False, False),
    ("rt_rating", False, False), ("rt_genre", True, False), ("rt_originalLanguage", False, False),
    ("rt_distributor", False, False), ("rt_soundMix", True, False),
    ("rt_director", True, True), ("rt_writer", True, True), ("sl_directors", True, True),
    ("sl_authors", True, True), ("sl_actors", True, True),
    ("fin_budget_src", False, False), ("fin_revenue_src", False, False),
]
rows, counts = [], {}
for col, multi, hidden in cat_specs:
    row, vc = category_profile(pop[col], col, multi, names_hidden=hidden)
    row["role"] = role_of(col)
    rows.append(row)
    counts[col] = vc
prof_cat = pd.DataFrame(rows).set_index("field")
with pd.option_context("display.max_rows", 60, "display.max_colwidth", 40):
    display(prof_cat)

# Track-record sparsity (counts only): directors and companies with a single film in the population
for col, label in [("rt_director", "RT directors"), ("production_companies", "TMDB production companies")]:
    vc = counts[col]
    print(f"{label}: {len(vc):,} distinct; {(vc == 1).mean():.0%} appear in only one usable film")

# The Numbers 'Franchise' looks like several names run together (e.g. 'BatmanDark Knight Trilogy')
fr = pop["tn_Franchise"].dropna()
glued = fr.str.contains(r"[a-z][A-Z]").sum()
print(f"tn_Franchise: {len(fr):,} films; {glued:,} values look like names glued together (lowercase then uppercase)")
flag("The Numbers", "tn_Franchise", "parse error", f"{glued:,} of {len(fr):,} values look like several franchise names run together")

fig, axes = plt.subplots(2, 3, figsize=(15, 7), layout="constrained")
for ax, col in zip(axes.ravel(), ["genres", "tn_MPAA Rating", "tn_Source", "tn_Creative Type", "original_language", "fin_budget_src"]):
    count_plot(ax, counts[col], pop[col].notna().sum(), col, xlabel="% of films with the level")
fig.suptitle("Selected categories, usable films (dotted line = 5% rare-level flag)", x=0.01, ha="left", fontsize=11)
plt.show()

### 3e. Identifier and free-text fields
Uniqueness, duplicates and length only. After the build's one-row-per-film step, no key should repeat.

In [ ]:
id_cols = ["id", "imdb_id", "title", "original_title", "tn_Movie URL", "rt_id", "sl_rotten_tomatoes_link",
           "wd_item", "overview", "tagline", "sl_movie_info", "sl_critics_consensus", "merged_ids"]
prof_id = id_text_profile(pop, id_cols)
display(prof_id)
for key in ["id", "imdb_id", "tn_Movie URL", "rt_id", "sl_rotten_tomatoes_link", "wd_item"]:
    shared = prof_id.loc[key, "rows sharing a value"]
    if shared:
        flag("Unified", key, "duplicate key", f"{shared} usable films share a {key}")
same_ty = pop.duplicated(["title", "year"], keep=False)
print(f"usable films sharing title and year with another film: {same_ty.sum()}")
if same_ty.any():
    display(pop.loc[same_ty, ["id", "title", "year", "fin_budget", "fin_revenue", "runtime", "in_tmdb"]].sort_values("title"))

### 3f. Release dates
TMDB's date drives the year split. Release month is an allowed input. Jan 1 is checked as a placeholder.

In [ ]:
d = pop["release_dt"]
jan1 = ((d.dt.month == 1) & (d.dt.day == 1)).sum()
print(f"release_dt: {d.min().date()} to {d.max().date()}, missing {d.isna().sum()}, Jan 1 dates {jan1} ({jan1 / len(d):.1%})")
if jan1 / len(d) > 0.01:
    flag("Unified", "release_dt", "placeholder date?", f"{jan1} films dated Jan 1")

month_share = pop["release_month"].value_counts(normalize=True).sort_index() * 100
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.2), width_ratios=[2, 1], layout="constrained")
for p, g in pop.groupby("period", observed=True):
    yc = g["year"].value_counts().sort_index()
    a1.bar(yc.index, yc.values, color=SERIES[p], width=0.8, label=f"{p} ({len(g):,})")
a1.set_title("Usable films per release year"); a1.set_xlabel("release year"); a1.set_ylabel("films")
a1.legend(fontsize=7, frameon=False); a1.grid(axis="x", visible=False)
a2.bar(month_share.index, month_share.values, color=BLUE, width=0.7)
a2.axhline(100 / 12, color=INK2, ls=":", lw=1)
a2.set_xticks(range(1, 13), list("JFMAMJJASOND"))
a2.set_title("Release month (dotted line = even 8.3%)"); a2.set_ylabel("% of films"); a2.grid(axis="x", visible=False)
plt.show()

### 3g. Do the sources agree? (new)

Where two sources report the same thing, they should agree. Disagreement is either a wrong match or a data error, and it tells Phase 3 which source to trust. Agreement = within 15% for money, 5 minutes for runtime, 7 days for release dates, same label for MPAA.

In [ ]:
def agree_num(a, b, tol):
    m = pop[a].gt(0) & pop[b].gt(0)
    r = (pop.loc[m, a] / pop.loc[m, b] - 1).abs()
    return {"films with both": int(m.sum()), "% agree": round(100 * (r <= tol).mean(), 1),
            "median |diff| %": round(100 * r.median(), 1)}

def agree_abs(a, b, tol):
    x, y = pop[a], pop[b]
    m = x.gt(0) & y.gt(0) if pd.api.types.is_numeric_dtype(x) else x.notna() & y.notna()
    diff = (x[m] - y[m]).abs()
    if hasattr(diff, "dt"):
        diff = diff.dt.days
    return {"films with both": int(m.sum()), "% agree": round(100 * (diff <= tol).mean(), 1),
            "median |diff| %": None}

agreement = pd.DataFrame({
    "budget: TMDB vs The Numbers": agree_num("budget", "tn_Production Budget (USD)", 0.15),
    "budget: TMDB vs Wikidata": agree_num("budget", "wd_wd_budget_usd", 0.15),
    "budget: The Numbers vs Wikidata": agree_num("tn_Production Budget (USD)", "wd_wd_budget_usd", 0.15),
    "revenue: TMDB vs The Numbers worldwide": agree_num("revenue", "tn_Worldwide Gross (USD)", 0.15),
    "revenue: TMDB vs Wikidata box office": agree_num("revenue", "wd_wd_box_office_usd", 0.15),
    "domestic: The Numbers vs Box Office Mojo": agree_num("tn_Domestic Gross (USD)", "bom_domestic_gross", 0.15),
    "domestic: The Numbers vs RT box office": agree_num("tn_Domestic Gross (USD)", "rt_boxOffice_usd", 0.15),
    "runtime: TMDB vs The Numbers (min)": agree_abs("runtime", "tn_Running Time (minutes)", 5),
    "runtime: TMDB vs RT (min)": agree_abs("runtime", "rt_runtimeMinutes", 5),
    "release: TMDB vs The Numbers (days)": agree_abs("release_dt", "tn_Release Date", 7),
    "release: TMDB vs Box Office Mojo first day (days)": agree_abs("release_dt", "bom_first_date", 7),
}).T
display(agreement)
for k, r in agreement.iterrows():
    if r["% agree"] < 80:
        flag("Unified", k, "sources disagree", f"{r['% agree']}% agree across {r['films with both']:,} films")

mpaa = pop[["tn_MPAA Rating", "rt_rating", "sl_content_rating"]].dropna()
print(f"MPAA rating, The Numbers vs RT vs RT sl (films with all three: {len(mpaa):,}): "
      f"all three agree on {(mpaa.nunique(axis=1) == 1).mean():.0%}")

# Matt's label risk: TMDB "worldwide" revenue that equals RT domestic box office
dom = pop["rt_boxOffice_usd"].gt(0) & pop["revenue"].gt(0)
same_dom = dom & (pop["revenue"] / pop["rt_boxOffice_usd"]).between(0.98, 1.02)
fixable = same_dom & pop["tn_Worldwide Gross (USD)"].gt(pop["revenue"] * 1.02)
print(f"TMDB revenue within 2% of RT domestic: {same_dom.sum():,} films (Matt: 396). "
      f"The Numbers has a higher worldwide gross for {fixable.sum():,} of them.")
flag("Unified", "fin_revenue", "domestic-only revenue?",
     f"{same_dom.sum():,} films have TMDB revenue = RT domestic; The Numbers worldwide is higher for {fixable.sum():,}")

ratio = (pop["budget"] / pop["tn_Production Budget (USD)"]).where(pop["budget"].gt(0) & pop["tn_Production Budget (USD)"].gt(0))
fig, ax = plt.subplots(figsize=(6, 3.2), layout="constrained")
ax.hist(np.log10(ratio.dropna()).clip(-1.5, 1.5), bins=60, color=BLUE, edgecolor="white", linewidth=0.4)
ax.axvspan(np.log10(1 / 1.15), np.log10(1.15), color=BLUE_LIGHT, zorder=0)
ax.set_title(f"Budget, TMDB / The Numbers (n = {ratio.notna().sum():,}; shaded = within 15%)")
ax.set_xlabel("log10 ratio (0 = identical)"); ax.set_ylabel("films")
plt.show()

### 3h. Who has which data? (new)
Matt found the RT join covers big films better than small ones (58% of $1–5M vs 91% of $60M+). The same check for every source and key input, by period and budget tier. Uneven coverage means missing values aren't random, so Phase 3 needs "unknown" indicators, not dropped rows.

In [ ]:
pop["budget_tier"] = pd.cut(pop["fin_budget"], [1e6, 5e6, 20e6, 60e6, np.inf], right=False,
                           labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"])
cov_cols = {"TMDB genres": "genres", "The Numbers match": "tn_Movie Name", "Wikidata": "wd_item",
            "Box Office Mojo": "bom_title", "RT movies (director)": "rt_director", "RT writer": "rt_writer",
            "RT sl (actors)": "sl_actors", "MPAA (The Numbers)": "tn_MPAA Rating", "Source (The Numbers)": "tn_Source",
            "Franchise (The Numbers)": "tn_Franchise", "Critic reviews": "rtrev_n_reviews"}
cov = pd.concat({
    "period": pop.groupby("period", observed=True).agg(**{k: (v, lambda s: s.notna().mean()) for k, v in cov_cols.items()}),
    "budget tier": pop.groupby("budget_tier", observed=True).agg(**{k: (v, lambda s: s.notna().mean()) for k, v in cov_cols.items()}),
})
display((100 * cov).round(0).astype(int).astype(str) + "%")
gap = (cov.loc["budget tier"].max() - cov.loc["budget tier"].min()) * 100
for k, g in gap.items():
    if g > 20:
        flag("Unified", cov_cols[k], "coverage skews by budget", f"{k}: coverage gap of {g:.0f} points between budget tiers")

### 3i. Flag register
Every flag from 2d–3h plus the skew, rare-level and missingness flags from the profile tables. Hand-off to the readiness gate (step 12) and the cleaning ledger (step 13).

In [ ]:
reg = list(FLAGS)
for f_, r in prof_num.iterrows():
    if r["role"] in (A, CAND, O, "derived") and r["skew flag"]:
        reg.append({"file": "Unified", "field": f_, "issue": "skew > 0.5",
                    "evidence": f"skew {r['skew']:.2f}; median {r['median']:,.3g} vs mean {r['mean']:,.3g}"})
    if r["role"] in (A, CAND) and r["% missing"] > 0:
        reg.append({"file": "Unified", "field": f_, "issue": "missing", "evidence": f"{r['% missing']}% missing"})
for f_, r in prof_cat.iterrows():
    if r["role"] not in (A, CAND):
        continue
    if r["levels < 5% of rows"] > 0:
        reg.append({"file": "Unified", "field": f_, "issue": "rare levels",
                    "evidence": f"{r['levels < 5% of rows']:,} of {r['levels']:,} levels under 5% of rows"})
    if r["% missing"] > 0:
        reg.append({"file": "Unified", "field": f_, "issue": "missing", "evidence": f"{r['% missing']}% missing"})
register = pd.DataFrame(reg).drop_duplicates(["file", "field", "issue"]).sort_values(["issue", "field"])
with pd.option_context("display.max_rows", 150, "display.max_colwidth", 100):
    display(register.reset_index(drop=True))
print(f"{len(register)} flags across {register['field'].nunique()} fields")
register.groupby("issue").size().rename("flags").to_frame()

### 3j. What the profile shows (facts from the outputs above, run 2026-11-23)

**Population and the label**
1. **6,838 usable films** (train 4,952 · test 1,264 · stress 622), up from Matt's 5,766. The test set grows 16% (1,086 → 1,264), which tightens the bootstrap CI on the evidence standard.
2. **The added films are mostly losers.** Films whose budget or revenue came from The Numbers or Wikidata are 83% losers at 2.5×, against 65% for Matt's TMDB-only films. Overall: 67.5% (Matt 64.6%). This is the S4 worry in reverse: the extra sources recover some of the flops TMDB was missing. The 20%-capacity ceiling drops slightly, from 31% to 30% of losers.
3. **S4 is still triggered.** 39.0% of films with a budget ≥ $1M still have no revenue from any source (Matt: 43.6%), and 52% in the stress years.
4. **Junk values grew with the population.** 269 films earn < 1% of budget (Matt 179), 35 report revenue < $1,000, 380 < $100,000. The lowest are the same junk Matt found (*Aquajots*: $100M budget, $1 revenue). Six films above 100×, including *Vacancy* (2018, $903M on $1M) and a new one, *Space Station 3D*, where a Wikidata budget meets a TMDB revenue.

**Do the sources agree?** (new)
5. **Money mostly agrees.** Budget: TMDB vs The Numbers 85%, TMDB vs Wikidata 90% (within 15%). Worldwide revenue: 81–87%. Domestic gross across The Numbers, Box Office Mojo and RT: 92–97%. Runtime: 92–93% within 5 minutes. MPAA rating: the three sources agree on 99%.
6. **Release dates often don't.** TMDB's date is within 7 days of The Numbers / Box Office Mojo for only 73–75% of films. TMDB often records the first release anywhere (a festival or a foreign market); the others record the US release. Release month is an allowed input, so Phase 3 has to pick one definition.
7. **Matt's domestic-as-worldwide risk is larger than first counted:** 434 films (Matt: 396) have TMDB revenue equal to RT domestic. **The Numbers has a higher worldwide figure for 220 of them**, a direct repair.

**Coverage is uneven** (new)
8. **Every extra source covers big films far better than small ones.** The Numbers matches 37% of $1–5M films vs 91% of $60M+; MPAA and source material 35% vs 91%; director 62% vs 90%; franchise 3% vs 44%. Small films are where most losers are, so a missing value is not random. RT stefanoleone992 (scraped in 2020) has actors for only 5% of stress-period films.
9. **Franchise is thin and partly garbled:** 15% coverage; 92 of 1,006 values look like several franchise names run together (e.g. "BatmanDark Knight Trilogy").
10. **Track records are sparse:** 63% of directors and 67% of production companies appear in only one usable film.

**Keys and duplicates**
11. **Duplicates (fixed in the build, 2026-11-23).** The first profile found title+year pairs that were the same film entered twice (*Spider-Man: No Way Home*, *Timber Falls*): the build's rule required runtime within 5 minutes, and one copy had runtime 0. The rule now treats 0 as unknown, which also caught *WALL·E* and *Black Panther: Wakanda Forever*. 3 pairs remain and are **not** duplicates: *Underdogs* (two different 2013 films), *Uncharted 2* (a video game, entered twice) and a junk *Blue Beetle* entry ($400M revenue, runtime 0). Those are cleaning issues. 2 usable films share a Wikidata item.
12. **Missed The Numbers matches (fixed in the build, 2026-11-23).** About 120 The Numbers films had been added as separate "not in TMDB" rows although TMDB had them under a differently written title (*Star Wars **Ep. VII**: The Force Awakens*, *Kill Bill: **Volume** 1*, *Harry Potter and the **Sorcerer's** Stone*). A second matching pass (build §4b) now joins them, which removed about 90 double-counted usable films (6,928 → 6,838). About 10 remain where the titles share almost nothing (*X-Men 2* vs *X2*, *F9: The Fast Saga* vs *F9*).
13. **Review copies (fixed in the build).** The stefanoleone992 review file had 120,945 rows identical in every column, and the andrezaza file 12,394 repeated review ids (each repeat identical in every column). Only those copies are dropped; every distinct review is kept. The individual reviews are now saved as their own table, `reviews_unified.csv` (one row per review, keyed by `film_uid`).

**Raw-file notes:** The Numbers has 150 release dates after the access date (up to 2068); none reach the usable population. RT sl reviews have 83 placeholder dates (1800-01-01). The Box Office Mojo scraper has no license.

### 3k. Proposed dispositions and phase gate (DRAFT, pending Tanner + Matt sign-off)

Matt's §3k rows 1–12 still apply. These are the additions for the unified dataset. **None are applied here.**

| # | Flag | Evidence | Proposed disposition | Treatment | Verification |
|---|---|---|---|---|---|
| U1 | Population now multi-source | 6,838 films; 1,078 use The Numbers / Wikidata financials | Requires a contract amendment | Dated amendment: population = 2000–2023, budget ≥ $1M, revenue > 0 **from TMDB, The Numbers or Wikidata**, with source precedence stated | Amendment recorded in the contract file |
| U2 | Added films skew to losers | 83% vs 65% losers | Usable; report it | Report results on the full population and on Matt's TMDB-only population as a sensitivity check | Capture on both populations |
| U3 | Domestic-as-worldwide revenue | 434 films; The Numbers higher for 220 | Requires repair | Replace TMDB revenue with The Numbers worldwide where TMDB = RT domestic and The Numbers is higher. Flag the other 214. | Loser rate before / after |
| U4 | Source disagreement on budget / revenue | 81–90% agree within 15% | Requires a rule | Keep precedence TMDB → The Numbers → Wikidata, but hand-check films where they disagree by > 2× | Hand-check sample error rate |
| U5 | Release date definitions differ | 74% agree within 7 days | Requires a rule | Use the US release date (The Numbers / Box Office Mojo) where available, TMDB otherwise; keep the source | Month distribution before / after |
| U6 | Coverage skews by budget | The Numbers 37% vs 91%; director 62% vs 90% | Usable with a limitation | Every optional input gets an "unknown" indicator; no dropped rows | Loser rate by known / unknown |
| U7 | Leftover duplicates | Runtime 0 blocked the merge rule | **Done** (build, 2026-11-23) | Runtime 0 counts as unknown; 4 more films merged | 3 same-title pairs left, none duplicates. Second pass (build §4b) joined ~120 missed The Numbers matches; ~10 remain (*X-Men 2* / *X2*) |
| U8 | Review copies | 120,945 + 12,394 copies of the same review | **Done** (build, 2026-11-23) | Only identical copies dropped; all reviews saved to `reviews_unified.csv` | Counts reported in the build |
| U11 | Junk entries | *Uncharted 2* (video game), *Blue Beetle* copy with $400M revenue and runtime 0 | Requires repair | Cleaning rule for runtime-0 TMDB entries that collide with a real film | Pair count after cleaning |
| U9 | Franchise field | 14% coverage, 92 garbled values | Investigate | See `FEATURE_IDEAS.md` (sequel / franchise); likely needs Wikidata series data | Coverage after |
| U10 | Candidate inputs not in the contract | MPAA, source, creative type, writer, cast, franchise | Needs a decision | Decide which to add by amendment before Phase 3 features | Decision recorded |

**Proposed phase gate: CONTINUE**, after U1 (amendment). Build fixes U7 and U8 are done. The S4 readiness verdict still belongs to step 12.

## 4. Missing values and repeated columns (step toward cleaning)

**Goal:** many facts appear in several columns, one per source (runtime is in four). Before combining them, measure what's actually missing in each, so a fake value (TMDB's `0`) never overrides a real one, and so we know whether a second source fills the gaps or misses the same films.

**Which rows:** the usable films. **This section only measures; nothing is changed.**

**What counts as missing** (found by scanning every column for placeholder values):

| Rule | Columns | Why |
|---|---|---|
| Blank (NaN) | all | — |
| `0` | TMDB `budget`, `revenue`, `runtime` | TMDB writes 0 for unknown (634 / 692 / 55 usable films) |
| `-` | `bom_distributor` | placeholder (57 films) |
| **Not missing** | "Not Rated" / "NR" ratings, 0% Tomatometer, 0 reviews before release | real values |

In [ ]:
ZERO_MISSING = {"budget", "revenue", "runtime"}
DASH_MISSING = {"bom_distributor"}

def present(c):
    """True where column c has a real value on the usable films (missing rules above)."""
    s = pop[c]
    ok = s.notna()
    if c in ZERO_MISSING:
        ok &= s.ne(0)
    if c in DASH_MISSING:
        ok &= s.astype(str).str.strip().ne("-")
    if s.dtype == object or str(s.dtype) == "str":
        ok &= s.astype(str).str.strip().ne("")
    return ok

# One fact, several source columns (first column listed = the source used today, if any)
CONCEPTS = {
    "budget":              ("allowed input",  ["budget", "tn_Production Budget (USD)", "wd_wd_budget_usd"]),
    "worldwide revenue":   ("label",          ["revenue", "tn_Worldwide Gross (USD)", "wd_wd_box_office_usd"]),
    "domestic gross":      ("post-release",   ["tn_Domestic Gross (USD)", "bom_domestic_gross", "rt_boxOffice_usd"]),
    "runtime":             ("allowed input",  ["runtime", "tn_Running Time (minutes)", "sl_runtime", "rt_runtimeMinutes"]),
    "release date":        ("allowed input",  ["release_date", "tn_Release Date", "bom_first_date", "sl_original_release_date", "rt_releaseDateTheaters"]),
    "genre":               ("allowed input",  ["genres", "tn_Genre", "sl_genres", "rt_genre"]),
    "MPAA rating":         ("candidate",      ["tn_MPAA Rating", "sl_content_rating", "rt_rating"]),
    "director":            ("allowed input",  ["rt_director", "sl_directors"]),
    "writer":              ("candidate",      ["rt_writer", "sl_authors"]),
    "cast":                ("candidate",      ["sl_actors"]),
    "production companies":("allowed input",  ["production_companies", "tn_Production/Financing Companies", "sl_production_company"]),
    "distributor":         ("candidate",      ["rt_distributor", "bom_distributor"]),
    "country":             ("allowed input",  ["production_countries", "tn_Production Countries"]),
    "language":            ("allowed input",  ["original_language", "rt_originalLanguage", "tn_Languages"]),
    "critic score":        ("post-release",   ["rt_tomatoMeter", "sl_tomatometer_rating", "rtrev_pct_fresh", "slrev_pct_fresh"]),
    "audience score":      ("post-release",   ["vote_average", "rt_audienceScore", "sl_audience_rating"]),
    "franchise":           ("candidate",      ["tn_Franchise"]),
    "source material":     ("candidate",      ["tn_Source"]),
    "creative type":       ("candidate",      ["tn_Creative Type"]),
}

rows = []
for concept, (role, cols) in CONCEPTS.items():
    P = pd.DataFrame({c: present(c) for c in cols})
    first = P.iloc[:, 0]
    rows.append({
        "concept": concept, "role": role, "columns": len(cols),
        "best single source": f"{P.mean().idxmax()} ({P.mean().max():.0%})",
        "first column": f"{first.mean():.0%}",
        "any source": f"{P.any(axis=1).mean():.0%}",
        "gain from combining": f"+{P.any(axis=1).mean() - P.mean().max():.0%}",
        "missing everywhere": int((~P.any(axis=1)).sum()),
    })
coverage = pd.DataFrame(rows).set_index("concept")
print(f"{len(pop):,} usable films; {sum(len(c) for _, c in CONCEPTS.values())} columns in {len(CONCEPTS)} concepts")
coverage

### 4a. Fill rate of every source column

Per concept: how full each source column is on the usable films, and how many films only that column covers (`only here`). A column that is never the only source adds nothing once the others are combined.

In [ ]:
detail = []
for concept, (role, cols) in CONCEPTS.items():
    P = pd.DataFrame({c: present(c) for c in cols})
    for c in cols:
        others = P.drop(columns=c).any(axis=1) if len(cols) > 1 else pd.Series(False, index=P.index)
        detail.append({"concept": concept, "column": c, "filled": f"{P[c].mean():.0%}",
                       "only here": int((P[c] & ~others).sum())})
pd.DataFrame(detail).set_index(["concept", "column"])

### 4b. Combine rules (reviewed by Tanner 2026-11-24; applied in `01_clean_dataset.ipynb`)

**General approach (agreed 2026-11-24):** each concept becomes **one combined column**. It takes the fullest source first, then fills its gaps from the others in the order listed. A value only counts if it's real under the §4 missing rules, so a TMDB `0` never wins. Each combined column gets a `_src` column recording which source supplied it. The original source columns move to a **source-detail** group: kept in the file for audits, but never model inputs. Applied in `01_clean_dataset.ipynb`, which writes `films_clean.csv`.

Fill rates are on the 6,838 usable films (§4 tables).

| # | Combined column | Fill order | Formatting before combining | Fill | Role | Notes |
|---|---|---|---|---|---|---|
| C1 | `fin_budget` | TMDB → The Numbers → Wikidata | none | 100% | allowed input | **Already built.** Keep. |
| C2 | `fin_revenue` | TMDB → The Numbers → Wikidata | none | 100% | label only | **Already built.** U3 (domestic-as-worldwide repair) is a separate cleaning step. |
| C3 | `domestic_gross` | The Numbers → Box Office Mojo → RT | none | 76% | post-release: checks only | Used for the S4 and U3 cross-checks, never as an input. |
| C4 | `runtime_min` | TMDB → RT → The Numbers → RT sl | none | 99% | allowed input | Values agree 92–93% within 5 min. 86 films left with none. |
| C5 | `release_date_us` | The Numbers → RT theatrical → Box Office Mojo first day | none | 78% | allowed input | **Decided (Tanner, 2026-11-24): use the US theatrical date**, because that's what a financier plans around. |
| C6 | `release_date_first` | TMDB | none | 98% | allowed input (fallback) | Release month = month of `release_date_us`, or `release_date_first` if that's missing; `_src` records which. |
| C7 | `genres` | TMDB → RT → The Numbers | map RT / The Numbers genre names onto TMDB's 19 genres (e.g. "Sci-fi" → "Science Fiction", "Thriller/Suspense" → "Thriller") | 100% | allowed input | TMDB already covers 97%; the others only fill 31 + 109 films. Multi-label, kept as a list. |
| C8 | `mpaa_rating` | The Numbers → RT → RT sl | standardize to G / PG / PG-13 / R / NC-17 / NR. The Numbers' run-on text ("NC-17for explicit…") becomes its leading rating; "Not Rated" and "NR" → NR; TV ratings and "Open" → missing | 76% | candidate input (A4) | Sources agree 99%. ⚠️ RT has **no NR level**, so a blank RT rating may really mean "not rated". "Unknown" and "NR" stay separate. |
| C9 | `director` | RT → RT sl | none (same name format) | 82% | allowed input (as **track record from earlier films only**) | 1,209 films have no director in any source. |
| C10 | `writer` | RT → RT sl | none | 81% | candidate input (A4) | Same track-record treatment as director. |
| C11 | ~~`cast`~~ | — | — | 60% | **dropped as an input** (Tanner, 2026-11-24) | Only 5% coverage in 2020–23 (2020 scrape). `sl_actors` stays in source detail. |
| C12 | `production_companies` | TMDB → The Numbers → RT sl | none: only fills TMDB's blanks, no name matching | 96% | allowed input (as track record) | Names differ by source ("Warner Bros." vs "Warner Bros. Pictures"). Standardizing names is a later cleaning step. |
| C13 | `distributor` | RT → Box Office Mojo | `-` → missing | 70% | candidate input (A4) | Same name-format issue as companies. |
| C14 | `countries` | TMDB → The Numbers | "United States" → "United States of America" (TMDB's names) | 98% | allowed input | Multi-label. |
| C15 | `original_language` | TMDB → RT | RT language name → ISO code ("English" → `en`) | 100% | allowed input | `tn_Languages` lists every spoken language, not the original one: source detail only. |
| C16 | `critic_pct_fresh` | RT reviews (`rtrev_pct_fresh`) → RT Tomatometer → RT sl Tomatometer | none (all 0–100) | 79% | **post-release: description only** | The one review-consensus column for the films table. `rtrev_n_reviews` stays next to it, as does `sl_critics_consensus` (text). |
| C17 | `audience_score` | TMDB `vote_average` × 10 → RT → RT sl | put on a 0–100 scale | 98% | **post-release: description only** | **Kept** (Tanner, 2026-11-24) to show how audience reception relates to money-losers. Never an input. |
| C18 | `franchise`, `source_material`, `creative_type` | The Numbers only | split the 92 run-together franchise names (U9) | 15% / 63% / 62% | candidate inputs (A4) | Nothing to combine. Franchise likely needs Wikidata series data (`FEATURE_IDEAS.md`). |

**Kept as is:** `film_uid`, `id`, `imdb_id`, `title`, `original_title`, `year`, `in_tmdb`, `usable`, `merged_ids`, all `fin_*`.

**Moves to source detail** (about 50 columns): every source's own copy of the facts above (`tn_Running Time (minutes)`, `sl_genres`, `bom_first_date`, …), match metadata (`*_year_diff`, `rt_rule`, `rt_matched_on`, source titles), and post-release columns not used for checks (DVD sales, opening weekend, theater counts, popularity, vote counts).

**"Unknown" indicators (U6):** every candidate or allowed input that is below 100% gets a missing flag (e.g. `mpaa_rating_missing`). Coverage depends on budget (§3h), so blanks aren't random and shouldn't be filled with a guess or dropped.

**Review decisions (Tanner, 2026-11-24):**
1. C5/C6: use the US theatrical date, with TMDB's date only as the fallback.
2. C11: cast is dropped as an input.
3. C17: audience score is kept, for correlation and description only.
All other rules are accepted as drafted.

## 5. The cleaned columns that could go into a model

From `films_clean.csv` (built by `01_clean_dataset.ipynb`), on the usable films: the budget and the combined input columns. Not shown: keys and ids (`film_uid`, `id`, `imdb_id`, titles), population flags, the label, review/description columns, `_src` / `_missing` helpers and the source-detail columns. They stay in the file for joins and audits but aren't model inputs.

In [ ]:
clean = pd.read_csv("../data/clean/films_clean.csv", low_memory=False)
cols = pd.read_csv("../data/clean/films_clean_columns.csv")
cu = clean[clean["usable"]]

def kind(c):
    s = cu[c].dropna()
    if len(s) and s.isin([True, False]).all():
        return "bool"
    if pd.api.types.is_numeric_dtype(cu[c]):
        return "number"
    if len(s) and pd.to_datetime(s.head(50), errors="coerce", format="mixed").notna().all():
        return "date"
    return "text"

cols["type"] = cols["column"].map(kind)
cols["filled"] = cols["column"].map(lambda c: int(cu[c].notna().sum()))
cols["blank"] = len(cu) - cols["filled"]
cols["fill %"] = (100 * cols["filled"] / len(cu)).round(0).astype(int)
cols["distinct"] = cols["column"].map(lambda c: cu[c].nunique())
core = cols[(cols["group"].eq("input") | cols["column"].eq("fin_budget"))
            & ~cols["column"].str.endswith(("_src", "_missing"))]
print(f"{len(core)} columns that could be model inputs (of {len(cols)} in the file)")
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 45):
    display(core.drop(columns="% filled (usable films)").set_index(["group", "column"]))

## 6. A closer look at six inputs

What the values are, and how they spread, for `runtime_min`, `genres_all`, `production_companies`, `distributor`, `countries` and `creative_type`. Usable films only, from `films_clean.csv`. **Observe only; nothing is changed.** Reviews come later.

For list columns (genres, companies, countries), a film can have several values, so shares add up to more than 100%.

In [ ]:
def items(col):
    """One row per (film, value) for a comma-separated list column."""
    s = cu[col].dropna().astype(str).str.split(",").explode().str.strip()
    return s[s.ne("")]


def per_film(col):
    """How many values each film has (0 = blank)."""
    return cu[col].fillna("").astype(str).map(lambda x: len([t for t in x.split(",") if t.strip()]))


def films_per_value(col):
    """For each distinct value, how many usable films it appears in, grouped into buckets."""
    vc = items(col).value_counts()
    b = pd.cut(vc, [0, 1, 4, 9, np.inf], labels=["1 film", "2-4 films", "5-9 films", "10+ films"])
    out = pd.DataFrame({"values": b.value_counts().sort_index(),
                        "films covered": vc.groupby(b, observed=False).sum()})
    out["% of values"] = (100 * out["values"] / len(vc)).round(1)
    return out


n_u = len(cu)
print(f"{n_u:,} usable films")

### 6a. `runtime_min` (minutes)

TMDB → RT → The Numbers → RT sl. Shown: summary statistics, the distribution, runtime bands, and the shortest and longest films (to spot errors).

In [ ]:
rt = cu["runtime_min"]
display(numeric_profile(cu, ["runtime_min"]))
profile_grid([(rt, "runtime_min", "minutes", False)], "films", ncols=1)
bands = pd.cut(rt, [0, 60, 80, 90, 100, 110, 120, 150, 180, np.inf],
               labels=["< 60", "60-79", "80-89", "90-99", "100-109", "110-119", "120-149", "150-179", "180+"], right=False)
display(bands.value_counts(dropna=False).sort_index().rename("films").to_frame()
        .assign(**{"% of films": lambda d: (100 * d["films"] / n_u).round(1)}))
show = ["title", "year", "runtime_min", "runtime_min_src", "fin_budget"]
print("Shortest:"); display(cu.nsmallest(8, "runtime_min")[show])
print("Longest:"); display(cu.nlargest(8, "runtime_min")[show])

### 6b. `genres_all`

TMDB's 19 genres, with RT / The Numbers filling TMDB's blanks.

In [ ]:
g = items("genres_all")
gc = g.value_counts()
display(pd.DataFrame({"films": gc, "% of films": (100 * gc / n_u).round(1)}))
print("Genres per film:"); display(per_film("genres_all").value_counts().sort_index().rename("films").to_frame())
print("Most common genre combinations:")
combo = cu["genres_all"].dropna().map(lambda x: ", ".join(sorted(t.strip() for t in x.split(","))))
display(combo.value_counts().head(15).rename("films").to_frame())
fig, ax = plt.subplots(figsize=(6, 5))
count_plot(ax, gc, n_u, "genres_all", top=19)
plt.show()

### 6c. `production_companies`

TMDB → The Numbers → RT sl. Companies only work as an input as a **track record** (how a company's earlier films did), so the key question is how many films each company has.

In [ ]:
pc = items("production_companies")
pcc = pc.value_counts()
print(f"{len(pcc):,} distinct companies across {cu['production_companies'].notna().sum():,} films")
print("Companies per film:"); display(per_film("production_companies").value_counts().sort_index().head(12).rename("films").to_frame())
print("How many films each company appears in:"); display(films_per_value("production_companies"))
print("Top 25 companies:"); display(pd.DataFrame({"films": pcc.head(25), "% of films": (100 * pcc.head(25) / n_u).round(1)}))
print("Same company, different spellings (example: names containing 'Warner' or 'Universal'):")
display(pcc[pcc.index.str.contains("Warner|Universal", case=False)].head(15).rename("films").to_frame())

### 6d. `distributor`

RT → Box Office Mojo. One value per film. Like companies, it would be used as a track record, or grouped into major studio vs independent.

In [ ]:
dc = cu["distributor"].value_counts()
print(f"{len(dc):,} distinct distributors across {cu['distributor'].notna().sum():,} films ({cu['distributor'].isna().mean():.0%} blank)")
print("How many films each distributor appears in:"); display(films_per_value("distributor"))
print("Top 25 distributors:"); display(pd.DataFrame({"films": dc.head(25), "% of films": (100 * dc.head(25) / n_u).round(1)}))
print("Same distributor, different spellings (examples):")
display(dc[dc.index.str.contains("Warner|Universal|Sony|Disney|Buena|Fox|Lionsgate|Lions Gate", case=False)].rename("films").to_frame())

### 6e. `countries`

TMDB → The Numbers. A film can have several production countries (co-productions).

In [ ]:
cc = items("countries").value_counts()
print(f"{len(cc):,} distinct countries")
print("Top 20:"); display(pd.DataFrame({"films": cc.head(20), "% of films": (100 * cc.head(20) / n_u).round(1)}))
n_c = per_film("countries")
has_us = cu["countries"].fillna("").str.contains("United States of America", regex=False)
kind_c = np.select([n_c.eq(0), has_us & n_c.eq(1), has_us, n_c.eq(1)],
                   ["blank", "US only", "US co-production", "one non-US country"], "non-US co-production")
display(pd.Series(kind_c).value_counts().rename("films").to_frame()
        .assign(**{"% of films": lambda d: (100 * d["films"] / n_u).round(1)}))
print("Countries per film:"); display(n_c.value_counts().sort_index().head(10).rename("films").to_frame())

### 6f. `creative_type`

The Numbers only (blank when the film isn't in The Numbers). Since The Numbers covers big films much better, the blank rate is shown by budget tier.

In [ ]:
ct = cu["creative_type"].value_counts(dropna=False)
display(pd.DataFrame({"films": ct, "% of films": (100 * ct / n_u).round(1)}))
tier = pd.cut(cu["fin_budget"], [0, 5e6, 20e6, 60e6, np.inf], labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"])
print("Blank by budget tier:")
display(cu["creative_type"].isna().groupby(tier, observed=True).mean().map("{:.0%}".format).rename("% blank").to_frame())
print("Creative type by budget tier (films with a value):")
display(pd.crosstab(cu["creative_type"], tier, normalize="columns").mul(100).round(1))

### 6g. What the six inputs show (run 2026-11-24)

1. **Runtime exposes non-films in the usable population.** 75 "films" run 180+ minutes, and the longest are TV series and video games with a runtime of 999 (*God of War*, *The Bear*, *Avatar: The Last Airbender*, *What If...?*, *Dragon Ball Z Kai*). Under 60 minutes are junk entries (*Anne Hathaway and Emily Blunt Hosting the Oscars*, 1 minute, "$30M"). These are cleaning items (extends U11): not feature films, so they don't belong in the population. Typical films: median 105 min, middle half 95–120.
2. **Genre is usable as is.** 19 genres, 99.5% filled, mostly 2–3 per film. Drama (48%) and Comedy (35%) dominate. *TV Movie* (14 films) is another sign of non-theatrical entries. *Documentary*, *Western* and *TV Movie* are under 2% each.
3. **Production companies: track records will be thin.** 8,393 companies; two-thirds appear in only one usable film, and only 382 appear in 10+. Name variants split the same studio ("Warner Bros. Pictures", "Warner Bros. Pictures Animation", "Warner Bros. Entertainment France"…). A track record needs either name standardization or a "major studio" grouping.
4. **Distributor is concentrated but messy.** 548 names, but 77 distributors with 10+ films cover 4,397 of the 4,811 films that have one. Same studio, several spellings ("Warner Bros." vs "Warner Bros. Pictures"; "Fox" vs "20th Century Fox"), and some values list two companies. 30% blank. A major / mini-major / independent grouping looks more useful than individual names.
5. **Countries: mostly US.** 65% of films include the US (42% US only, 23% US co-productions); 33% are non-US. A small set of flags (US / UK / France / India / co-production) would capture most of it; 108 countries are too many to use individually.
6. **Creative type is informative but its blanks aren't random.** 38% blank overall, but 64% for $1–5M films vs 9% for $60M+. Where filled, it tracks budget strongly (Super Hero and Kids Fiction are big-budget; Factual is small-budget). Using it needs the `creative_type_missing` flag. One stray value ("\tScience Fiction", a tab character) is fixed in `01_clean_dataset.ipynb`.

### 6h. Tanner's review of the six inputs (2026-11-24)

**Runtime**
- Use runtime to find and drop non-films: outliers (999 min, under 60 min) and TV shows.
- Identify TV shows with runtime **plus** other signals, e.g. the *TV Movie* genre. Then remove them in cleaning.

**Genres**
- Look at *Music*: take a sample to see whether these are music videos/concerts (drop) or feature films about musicians, like biopics (keep).
- Films in several genres are fine. How to represent combinations is still open.

**Production companies**
- Group name variants into one studio in cleaning (all Warner Bros. variants → Warner Bros.; same for Universal, etc.).
- Then categorize: studio groups, foreign studios, animation vs live-action arms.

**Distributor**
- Same as companies: standardize spelling in cleaning, then group.
- Some films list several distributors. Open: how to link multiple distributors to one film (like genres).

**Countries**
- Keep for now, but likely the first input to drop: most big-budget films are US.
- Better as a **US / non-US** category than 108 countries.

**Creative type**
- Interesting values, but 38% blank and overlaps genre (a finer-grained genre).
- Don't go find data to fill it. Usefulness doubtful; decide at modeling.

**Cleaning to-do from this review** (for `01_clean_dataset.ipynb`)
1. Drop runtime outliers and TV shows / non-films.
2. Sample *Music* genre; drop non-films.
3. Standardize production company and distributor names, then group.
4. Decide how to handle multiple distributors per film.


## 7. The remaining inputs

Same review as §6 for the other input columns: `fin_budget`, release timing, `mpaa_rating`, `director`, `writer`, `original_language`, `franchise`, `source_material`. Values and spread only; no outcome yet.

### 7a. `fin_budget`

The production budget (the contract allows it as an input). Shown: spread, budget tiers, round-number budgets, and the smallest and largest.

In [ ]:
b = cu["fin_budget"]
display(numeric_profile(cu, ["fin_budget"]))
profile_grid([(np.log10(b), "log10 fin_budget", "log10 $ (6 = $1M, 8 = $100M)", False)], "films", ncols=1)
btier = pd.cut(b, [0, 5e6, 20e6, 60e6, 150e6, np.inf], labels=["$1-5M", "$5-20M", "$20-60M", "$60-150M", "$150M+"])
display(btier.value_counts().sort_index().rename("films").to_frame().assign(**{"% of films": lambda d: (100 * d["films"] / n_u).round(1)}))
print(f"Budgets that are a round $1M: {(b % 1e6 == 0).mean():.0%}")
print("Source:"); display(cu["fin_budget_src"].value_counts().rename("films").to_frame())
show = ["title", "year", "fin_budget", "fin_budget_src", "runtime_min"]
print("Largest:"); display(cu.nlargest(8, "fin_budget")[show])

### 7b. Release timing: `release_date`, `release_month`, `release_quarter`

US release date, falling back to TMDB's first release. Shown: films by month and year, and how often the fallback is used.

In [ ]:
display(cu["release_date_src"].value_counts().rename("films").to_frame())
rm = cu["release_month"].value_counts().sort_index()
display(pd.DataFrame({"films": rm, "% of films": (100 * rm / n_u).round(1)}).T)
yr = cu["year"].value_counts().sort_index()
display(yr.rename("films").to_frame().T)
rd = pd.to_datetime(cu["release_date"], format="mixed", errors="coerce")
print(f"Release year differs from film year: {(rd.dt.year != cu['year']).mean():.1%} of films")
print("Day of week:"); display(rd.dt.day_name().value_counts().rename("films").to_frame().T)

### 7c. `mpaa_rating`

The Numbers → RT → RT sl, standardized. "NR" (not rated) is a real value; blank means no source had one.

In [ ]:
mr = cu["mpaa_rating"].value_counts(dropna=False)
display(pd.DataFrame({"films": mr, "% of films": (100 * mr / n_u).round(1)}))
print("Blank by budget tier:")
display(cu["mpaa_rating"].isna().groupby(btier, observed=True).mean().map("{:.0%}".format).rename("% blank").to_frame())
print("Blank by original language (top 6):")
top_lang = cu["original_language"].value_counts().head(6).index
display(cu[cu["original_language"].isin(top_lang)].groupby("original_language")["mpaa_rating"].apply(lambda s: f"{s.isna().mean():.0%}").rename("% blank").to_frame())

### 7d. `director` and `writer`

People only work as an input as a **track record** (how their earlier films did), so the key question is how many films each person has.

In [ ]:
for col in ["director", "writer"]:
    v = items(col).value_counts()
    print(f"--- {col}: {len(v):,} distinct people across {cu[col].notna().sum():,} films ---")
    print("People per film:"); display(per_film(col).value_counts().sort_index().head(8).rename("films").to_frame().T)
    print("How many films each person appears in:"); display(films_per_value(col))
    print("Top 10:"); display(v.head(10).rename("films").to_frame().T)
both = cu[["director", "writer"]].dropna()
same = both.apply(lambda r: bool(set(t.strip() for t in r["director"].split(",")) & set(t.strip() for t in r["writer"].split(","))), axis=1)
print(f"Director also credited as writer: {same.mean():.0%} of films with both")

### 7e. `original_language`

TMDB → RT, ISO codes.

In [ ]:
ol = cu["original_language"].value_counts(dropna=False)
display(pd.DataFrame({"films": ol, "% of films": (100 * ol / n_u).round(1)}).head(15))
print(f"{cu['original_language'].nunique()} distinct languages")
print("English vs countries (does the US flag already cover it?):")
us = cu["countries"].fillna("").str.contains("United States")
display(pd.crosstab(cu["original_language"].eq("en").map({True: "English", False: "Not English"}), us.map({True: "includes US", False: "no US"}), margins=True))

### 7f. `franchise`

The Numbers only, split into one value per franchise (`|`-separated). Blank means either "not in a franchise" or "not in The Numbers", and the two can't be told apart.

In [ ]:
fr = cu["franchise"].dropna().str.split("|").explode().str.strip()
frc = fr.value_counts()
print(f"{cu['franchise'].notna().mean():.1%} of films have a franchise; {len(frc):,} distinct franchises")
print("Films per franchise:")
display(pd.cut(frc, [0, 1, 2, 4, np.inf], labels=["1", "2", "3-4", "5+"]).value_counts().sort_index().rename("franchises").to_frame().T)
display(frc.head(15).rename("films").to_frame().T)
print("Filled by budget tier:")
display(cu["franchise"].notna().groupby(btier, observed=True).mean().map("{:.0%}".format).rename("% with franchise").to_frame().T)
print("Franchise vs source_material:")
display(pd.crosstab(cu["source_material"].fillna("(blank)"), cu["franchise"].notna().map({True: "franchise", False: "no franchise"})))

### 7g. `source_material`

The Numbers only.

In [ ]:
sm = cu["source_material"].value_counts(dropna=False)
display(pd.DataFrame({"films": sm, "% of films": (100 * sm / n_u).round(1)}))
print("Blank by budget tier:")
display(cu["source_material"].isna().groupby(btier, observed=True).mean().map("{:.0%}".format).rename("% blank").to_frame().T)
print(f"source_material and creative_type both blank: {(cu['source_material'].isna() & cu['creative_type'].isna()).mean():.0%}; "
      f"only one blank: {(cu['source_material'].isna() ^ cu['creative_type'].isna()).mean():.0%}")

### 7h. What the remaining inputs show (run 2026-11-24), for Tanner's review

1. **Budget has bad values at the top.** Most budgets look right: median $14M, 74% round millions. But the largest are errors: *Batman v Superman … Ultimate Edition* ($874M, a re-release copy), *Adventures in Bora Bora* ($800M, 5 min), *M4's The Hobbit Book Edit* ($700M, a fan edit), *Diggy and The Bandits* ($447M). *Lost in the Stars* ($418M, Chinese) is probably in yuan, not dollars, so **some non-US budgets may be in local currency**. Cleaning item: check large budgets and non-English films' budgets against a second source.
2. **Release timing is clean, but the timing rule needs a decision.** Releases are spread evenly, with October and December highest. 22% of dates are the TMDB fallback (not US), which explains the Wednesday/Thursday releases (most US releases are Friday). Films per year drop by half in 2020–23 (COVID). ⚠️ The release month is usually picked well after financing. Decision: allow it as "planned release window," or drop it.
3. **MPAA rating blanks follow language.** 24% blank overall: 6% for English films, 56–93% for non-English films. A blank mostly means "not released in the US with a rating." Keep `mpaa_rating_missing` as a flag, or merge blank with NR.
4. **Directors and writers: track records will be thin, like companies.** 63% of directors and 67% of writers appear in only one usable film; only about 30 people in each have 10+ films. 48% of films are directed by one of their writers. Better as "number of earlier films" (experience) than as names. Track records must only count films released **before** this one.
5. **Language mostly repeats country.** 72% English. English lines up with "includes US" for 89% of films. With a US / non-US flag, language adds little; an English / non-English flag at most. Note: TMDB has both `cn` (Cantonese) and `zh` (Mandarin).
6. **Franchise is useful but blank is ambiguous.** 15% of films have one; 530 franchises, MCU largest (33). Strongly tied to budget: 4% of $1–5M films vs 65% of $150M+. Blank can mean "not a franchise" or "not in The Numbers"; since nearly all usable films are in The Numbers, blank is close to "no franchise." Good base for the sequel idea in `FEATURE_IDEAS.md`. 478 franchise films are tagged "Original Screenplay," so the two columns measure different things.
7. **Source material repeats creative type's gaps.** 37% blank, 63% for $1–5M films. When one is blank, the other almost always is too (only 1% differ), so both come from the same missing The Numbers record. Too many small categories (23); group into Original / Book / Real life / Comic / Remake-sequel / Other.

**Cleaning to-do from this review** (adds to §6h)
5. Check and fix extreme budgets and possible local-currency budgets.
6. Drop re-release editions and fan edits (e.g. "Ultimate Edition", "Book Edit").

**Decisions for Tanner**
- Release month as an input: allow (planned window) or drop?
- MPAA blank: keep as its own value, or merge with NR?
- Language: keep alongside the US flag, or drop?


### 7i. Tanner's review of the remaining inputs (2026-11-24)

**Budget**
- Must be cleaned. Find bad budgets by comparing **budget vs runtime**: films far outside the normal band are the suspects.
- Compare budget by **country / region** to find budgets in local currency (e.g. *Lost in the Stars*).

**MPAA rating**
- Fill blanks with NR.
- But keep the blank flag: a blank/NR may signal an entry that isn't a theatrical film (leakage of non-films into the data). Check this during non-film cleanup.

**Director and writer**
- Not enough data to rank "valuable" directors, but we do have film counts.
- Every person listed has at least one film. Count each person's films, then aggregate across a film's directors and writers. (Cleaning step; counts must only use earlier films when used as a model input.)
- Possible smaller feature: the writer + director combination.

**Language and country**
- They serve the same purpose. Combine them: **country is primary**; when country is blank, fill it from language.

**Franchise**
- Turn into a category. Blanks are real ("not a franchise"), since most films aren't franchises. **Don't drop for incompleteness.**
- Franchise × source material could be a sub-predictor (e.g. franchise based on a comic vs original). The pool gets small, but worth trying as a sub-level.

**Release timing**
- Keep. The outcome is revenue, and the release month is known before revenue.
- The frozen contract already allows the planned release month. Tanner also proposes moving the timing rule to "anything controllable before release" (amendment A6).

**Cleaning to-do from this review** (adds to §6h and §7h)
7. Flag suspect budgets with budget vs runtime and budget by country/region; fix or drop.
8. Fill blank MPAA with NR; keep `mpaa_rating_missing`.
9. Build film counts per director and writer.
10. Fill blank country from language.
11. Franchise: blank = "no franchise".


## 8. Outliers and non-films (candidate flags)

Each rule below is a **candidate**. Nothing is dropped here. Each rule shows how many usable films it catches and a sample to check. Once Tanner approves a rule, it moves into `01_clean_dataset.ipynb` as an `exclude_reason` (the film stays in the file but is no longer `usable`).

In [ ]:
pd.set_option("display.max_colwidth", 60)
SHOW = ["title", "year", "runtime_min", "fin_budget", "fin_revenue", "genres_all", "countries", "original_language", "mpaa_rating"]
flags = pd.DataFrame(index=cu.index)

# Outside sources: a real theatrical film is almost always in at least one of these.
cu = cu.assign(n_sources=cu["tn_Movie URL"].notna().astype(int) + cu["rt_id"].notna() + cu["bom_title"].notna() + cu["sl_rotten_tomatoes_link"].notna())

def check(rule, mask, note, n=15, sort=None, seed=0):
    flags[rule] = mask.reindex(cu.index, fill_value=False)
    hit = cu[flags[rule]]
    print(f"{rule}: {len(hit):,} usable films ({len(hit) / n_u:.1%}). {note}")
    hit = hit.sort_values(sort, ascending=False) if sort else hit.sample(min(n, len(hit)), random_state=seed)
    display(hit[SHOW + ["n_sources"]].head(n))

### 8a. Non-film signals

In [ ]:
check("runtime_400plus", cu["runtime_min"] >= 400, "TV series and video games (runtime 999 = 'unknown/very long').", sort="runtime_min")

In [ ]:
check("runtime_under_60", cu["runtime_min"] < 60, "Shorts, clips and junk entries.", n=25)

In [ ]:
check("tv_movie_genre", cu["genres_all"].fillna("").str.contains("TV Movie"), "TMDB's TV Movie genre.", n=20)

In [ ]:
pat = (r"\b(?:edition|extended|ultimate cut|director'?s cut|re-?release|original video|ufc \d|wwe|live (?:at|with|from)|live in \w+|"
       r"anniversary tour|confessions tour|in concert|concert|broadway|the play|soundstage|making of|behind the scenes|season \d)\b")
check("title_pattern", cu["title"].str.contains(pat, case=False, regex=True),
      "Title says it's an edition, event, concert, stage play or episode.", n=30)

In [ ]:
music = cu[cu["genres_all"].fillna("").str.contains("Music")]
print(f"Music genre: {len(music):,} usable films; Music as the only genre: {(music['genres_all'].str.strip() == 'Music').sum()}")
print("Random sample of 30 (Tanner's question: music videos/concerts, or films about musicians?):")
display(music.sample(30, random_state=1)[SHOW + ["n_sources"]])
check("music_only_genre", cu["genres_all"].fillna("").str.strip().eq("Music"), "Music is the only genre (likely concerts).", n=20)

In [ ]:
print("Usable films by number of outside sources (The Numbers, RT, RT sl, Box Office Mojo):")
display(cu["n_sources"].value_counts().sort_index().rename("films").to_frame().T)
check("no_outside_source", cu["n_sources"].eq(0), "Only TMDB knows this film; budget and revenue are unverified.", n=30, seed=2)
print("Original language of films with no outside source:")
display(cu.loc[flags["no_outside_source"], "original_language"].value_counts().head(8).rename("films").to_frame().T)

### 8b. Budget and revenue errors

Tanner's idea: compare **budget vs runtime** (a long, cheap film or a short, expensive one stands out) and **budget by country** (non-US budgets that look too large may be in local currency).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ok = cu["n_sources"] > 0
ax.scatter(cu.loc[ok, "runtime_min"], cu.loc[ok, "fin_budget"], s=4, alpha=.3, label="in an outside source")
ax.scatter(cu.loc[~ok, "runtime_min"], cu.loc[~ok, "fin_budget"], s=6, alpha=.6, color="C3", label="TMDB only")
ax.set(xscale="log", yscale="log", xlabel="runtime (min, log)", ylabel="budget ($, log)", title="Budget vs runtime, usable films")
ax.legend(); plt.show()
cu = cu.assign(budget_per_min=cu["fin_budget"] / cu["runtime_min"])
print("Budget per minute, by runtime band (typical films):")
bpm = cu.groupby(bands.reindex(cu.index), observed=True)["budget_per_min"].describe(percentiles=[.5, .95])[["count", "50%", "95%", "max"]]
display(bpm.astype({"count": int}).round({"50%": -3, "95%": -3, "max": -3}))
check("budget_per_min_extreme", (cu["budget_per_min"] > 3e6) | ((cu["runtime_min"] < 70) & (cu["fin_budget"] >= 20e6)),
      "Over $3M per minute, or under 70 min with a $20M+ budget.", n=25, sort="budget_per_min")

In [ ]:
country1 = cu["countries"].fillna("(blank)").str.split(",").str[0].str.strip()
by = cu.groupby(country1)["fin_budget"].agg(films="size", median="median", p90=lambda s: s.quantile(.9), max="max")
print("Budget by first-listed country (countries with 40+ films):")
display(by[by["films"] >= 40].sort_values("median", ascending=False).round({"median": -5, "p90": -5, "max": -5}))
non_us = ~cu["countries"].fillna("").str.contains("United States") & cu["original_language"].ne("en")
check("non_us_budget_100m", non_us & (cu["fin_budget"] >= 100e6), "Non-US, non-English films with a $100M+ budget: possibly local currency.", n=30, sort="fin_budget")

In [ ]:
check("revenue_equals_budget", cu["fin_revenue"].eq(cu["fin_budget"]), "Revenue exactly equals budget: a copied value, not a real figure.", n=20)
check("revenue_under_10k", cu["fin_revenue"] < 1e4, "Revenue under $10K (e.g. 5 or 10 dollars).", n=20, sort="fin_budget")

### 8c. Summary of candidate flags

In [ ]:
summary = pd.DataFrame({"films": flags.sum(), "% of usable": (100 * flags.sum() / n_u).round(1),
                        "also TMDB-only": flags.apply(lambda c: (c & flags["no_outside_source"]).sum())})
display(summary)
hard = flags.drop(columns="no_outside_source").any(axis=1)
print(f"Caught by at least one rule other than 'no outside source': {hard.sum():,} ({hard.mean():.1%})")
print(f"Plus 'no outside source': {flags.any(axis=1).sum():,} ({flags.any(axis=1).mean():.1%})")

### 8d. What the outlier checks show (run 2026-11-24), for Tanner's review

**The biggest finding: 929 usable films (14%) exist only in TMDB.** None of The Numbers, Rotten Tomatoes or Box Office Mojo has them, so their budget and revenue (and so the label) come only from TMDB's user-edited data. Almost every other rule mostly catches these same films. They are a mix:
- Junk: *Skibidi Toilet: The Movie*, *Adventures in Bora Bora* ($800M, 5 min), *😤 insect james 😤*, fan edits, concert videos.
- Real films, mostly non-English (80%): *Deep Sea* (China), *Death Note: The Last Name* (Japan), *Solomon Kane*.

| Rule | Films | Proposed |
|---|---|---|
| Runtime 400+ min | 12 | **Exclude.** All TV series or video games (*God of War*, *The Bear*, *What If...?*). |
| Runtime under 60 min | 46 | **Exclude.** Anime episodes, clips, junk. A few real IMAX documentaries (*Sea Rex*), which aren't feature films either. |
| TV Movie genre | 14 | **Exclude.** Not theatrical (*High School Musical*, *Stargate: Continuum*); their revenue figures are made up ($5,940). |
| Title says edition / cut / concert / play | 26 | **Exclude**, except theatrical concert films (*Hannah Montana … Concert*, *Glee: The Concert Movie*). Re-release editions duplicate the real film. |
| Music as the only genre | 15 | **Exclude** if TMDB-only (concert videos). |
| No outside source | 929 | **Decision needed** (below). |
| Budget per minute extreme | 25 | **Exclude.** Junk entries; also shows wrong runtimes on real titles. |
| Non-US $100M+ budget | 12 | **Check by hand.** Some are real US dollars (*The Battle at Lake Changjin*, *Detective Chinatown 3*); *Lost in the Stars* and *Chasing the Dragon* look like local currency. |
| Revenue = budget | 18 | **Exclude.** Copied value, not a real revenue. |
| Revenue under $10K | 101 | Keep if an outside source has the film (real tiny releases, e.g. *The Fanatic*); exclude if TMDB-only. |

**Tanner's Music question:** the Music genre is mostly real films about music (*A Star Is Born*, *Crazy Heart*, *Step Up*, *Footloose*). Concert films that played in theaters (*This Is It*, *Justin Bieber: Believe*) are real releases with real box office. Only the TMDB-only concert videos are junk. **Keep the genre; drop the videos.**

**Budget vs runtime:** works for junk at the edges (5-minute films with $100M budgets) but not in the middle. Budget per minute is roughly flat from 80 to 180 minutes.

**Budget by country:** doesn't show currency errors as a group. China has the highest median ($25M). Currency mistakes are individual films, so they need a hand check, not a country rule.

**New matching problem found:** in the TMDB snapshot, the real *Wonka* and *Wish* (2023) are still "Post Production", so the build skipped them. The Numbers' figures for them then matched to junk TMDB entries with the same title and year: *WONKA* (a 11-minute "Documentary") and *Wish* (a 66-minute drama). The rows have the real budget and revenue but the wrong runtime, genre and metadata. Fix in `00_build_dataset.ipynb`: don't match to TMDB entries whose runtime is far from the other source's, and consider late-2023 "Post Production" entries. Other late-2023 releases may have the same problem.

**Decision for Tanner: the 929 TMDB-only films**
1. **Exclude all** (recommended). The label can't be checked against anything. Cost: loses some real foreign films and makes the data more US-heavy.
2. **Exclude only those caught by another rule** (about 120). Keeps real foreign films, but leaves unverified revenue in the label.
3. **Keep all**, and add `n_sources` as a data-quality flag for a sensitivity check.

**After this:** §9 goes through missing values one column at a time.


### 8e. Tanner's decisions on the outlier rules (2026-11-24)

| Rule | Decision | Left after dropping TMDB-only |
|---|---|---|
| No outside source (929) | **Drop all.** "They do not feel like real movies." | 0 |
| *The Bear* | **Drop everywhere** (`tmdb-1233925`, `tmdb-1769662`). A TV series with odd numbers. | 0 |
| Runtime 400+ | **Drop** TV series and video games (confirmed 2026-11-24). Tanner first thought *Silent Hill 2* and *10 Things I Hate About You* were films; these entries are the 2001 video game and the 2009 TV series (the films are 2006 / 1999). All are TMDB-only, so dropped anyway. | 1 (*God of War*) |
| TV Movie genre | **Drop** (revised 2026-11-24). A TV movie has no box office, so its revenue and loser label can't be trusted. | 0 |
| Concerts / music events | **Keep if they had a theatrical run.** If people paid to see it, it's a movie (e.g. *Taylor Swift: The Eras Tour*). Having an outside source counts as a theatrical run. | 2 (*Hannah Montana*, *Glee*) |
| Extended editions / director's cuts | **Drop.** They come out after the original, usually without a theatrical run. | 1 (*Napoléon* director's cut) |
| Music genre | **Keep.** Mostly real films. | all |
| Budget per minute extreme | **Drop**, except *Winnie the Pooh* (2011) and *Pooh's Heffalump Movie* (real, double-check). | 7 |
| Revenue = budget | **Drop.** Data-entry errors; can't trust the numbers. Only *High School Musical 2* gave pause. | 9 |
| Runtime under 60 | **Drop all** (2026-11-24). | 0 |
| Non-US $100M+ budget | Hand check, not reviewed yet. | 6 |
| Revenue under $10K | Not reviewed yet (proposed keep). | 75 |

**TV movies:** Tanner first said keep, then agreed to drop after seeing that their "revenue" isn't box office (*High School Musical*: $5,940; *HSM 2*: revenue = budget).

**Build fix needed (`00_build_dataset.ipynb`):** the TMDB snapshot marks 46 films with 2023 release dates as "Post Production" or "In Production", so the build skipped them. Among them: *The Marvels* ($275M), *Wish*, *Wonka*, *Rebel Moon*, and *Taylor Swift: The Eras Tour*. Their The Numbers figures then matched junk TMDB entries with the same title (*WONKA*, *Wish*, likely *Five Nights at Freddy's*, 7 min). Fix: include those films, and don't match when runtimes disagree badly. The Numbers already has 5 of the 46 (*The Marvels*, *Wish*, *Wonka*…). The rest, including *Eras Tour*, are in no source. Tanner wants *The Marvels* (flop) in the data. Decision: recover the 5 The Numbers covers; no hand-entered data, so *Eras Tour* stays out.


### 8f. Budget fixes, currency check, and tiny revenues (agreed 2026-11-24)

Works on the films that survive the drops agreed in §8e (`keep`). Tanner agreed:
1. When TMDB's budget and The Numbers' (or Wikidata's) differ by more than 2×, use The Numbers.
2. For non-US films with only one budget source: chart budget vs revenue, and for the extreme ones, test "what if this budget is in the local currency?" using the country of origin. Worst case, drop them.
3. Revenue under $10K: keep films with a real MPAA rating; drop NR and blank.

In [ ]:
keep = (cu["n_sources"] > 0) & (cu["runtime_min"].fillna(999) >= 60) & (cu["runtime_min"].fillna(0) < 400) \
       & ~flags["tv_movie_genre"] & ~flags["revenue_equals_budget"] \
       & ~(flags["title_pattern"] & ~cu["title"].str.contains("concert", case=False)) \
       & ~(flags["budget_per_min_extreme"] & ~cu["title"].isin(["Winnie the Pooh", "Pooh's Heffalump Movie"]))
k = cu[keep].copy()
print(f"Usable films after the §8e drops: {len(k):,} (dropped {(~keep).sum():,})")

# 1. Two budget sources that disagree
tmdb_b = k["budget"].where(k["budget"] > 0)
other_b = k["tn_Production Budget (USD)"].fillna(k["wd_wd_budget_usd"])
ratio = tmdb_b / other_b
k["budget_fixed"] = np.where((ratio > 2) | (ratio < 0.5), other_b, k["fin_budget"])
fixed = k["budget_fixed"].ne(k["fin_budget"])
print(f"\n1. Films with two budgets: {ratio.notna().sum():,}; differ by more than 2x: {fixed.sum()} -> use The Numbers / Wikidata")
display(k.loc[fixed, ["title", "year", "countries", "fin_budget", "budget_fixed"]].assign(ratio=ratio[fixed].round(1))
        .sort_values("ratio", ascending=False).head(15))

In [ ]:
# 2. Non-US films with only one budget source
non_us = ~k["countries"].fillna("").str.contains("United States")
single = non_us & (tmdb_b.isna() | other_b.isna())
country1 = k["countries"].fillna("(blank)").str.split(",").str[0].str.strip()
print(f"Non-US films: {non_us.sum():,}; with only one budget source: {single.sum():,}")

fig, ax = plt.subplots(figsize=(8, 5))
for m, lab, col in [(~non_us, "US", "C0"), (non_us & ~single, "non-US, two budget sources", "C2"), (single, "non-US, one budget source", "C3")]:
    ax.scatter(k.loc[m, "budget_fixed"], k.loc[m, "fin_revenue"], s=5, alpha=.35, color=col, label=lab)
ax.plot([1e6, 1e9], [2.5e6, 2.5e9], "k--", lw=1, label="revenue = 2.5x budget")
ax.set(xscale="log", yscale="log", xlabel="budget ($, log)", ylabel="revenue ($, log)", title="Budget vs revenue")
ax.legend(fontsize=8); plt.show()

# Extreme = above twice the 95th percentile of verified budgets from the same country (or $100M+ if the country has none)
verified = k[non_us & ~single]
p95 = verified.groupby(country1[verified.index])["budget_fixed"].quantile(.95)
limit = country1.map(p95).fillna(50e6) * 2
extreme = single & (k["budget_fixed"] > limit) & (k["budget_fixed"] >= 20e6)

# Approximate average exchange rates, local currency per US$ (2000-2023, rounded). Only used for this test.
RATE = {"China": ("CNY", 6.8), "Hong Kong": ("HKD", 7.8), "Japan": ("JPY", 110), "South Korea": ("KRW", 1150),
        "India": ("INR", 65), "Russia": ("RUB", 50), "United Kingdom": ("GBP", 0.7), "Canada": ("CAD", 1.25),
        "Australia": ("AUD", 1.35), "France": ("EUR", 0.88), "Germany": ("EUR", 0.88), "Italy": ("EUR", 0.88),
        "Spain": ("EUR", 0.88), "Belgium": ("EUR", 0.88), "Mexico": ("MXN", 17), "Brazil": ("BRL", 3.5),
        "Turkey": ("TRY", 5), "Norway": ("NOK", 8), "Sweden": ("SEK", 8.5), "Denmark": ("DKK", 6.5),
        "Thailand": ("THB", 33), "Taiwan": ("TWD", 31), "Argentina": ("ARS", 20), "Poland": ("PLN", 3.6)}
x = k[extreme].assign(country=country1[extreme])
x["currency"] = x["country"].map(lambda c: RATE.get(c, ("?", np.nan))[0])
x["if_local_usd"] = x["budget_fixed"] / x["country"].map(lambda c: RATE.get(c, ("?", np.nan))[1])
x["country_p95"] = x["country"].map(p95)
x["local_plausible"] = x["if_local_usd"].between(1e6, x["country_p95"].fillna(50e6) * 2)
print(f"Extreme one-source budgets: {len(x)}")
display(x[["title", "year", "country", "budget_fixed", "fin_revenue", "country_p95", "currency", "if_local_usd", "local_plausible"]]
        .sort_values("budget_fixed", ascending=False))

In [ ]:
# 3. Revenue under $10K: keep only films with a real MPAA rating
tiny = k["fin_revenue"] < 1e4
rated = k["mpaa_rating"].isin(["G", "PG", "PG-13", "R", "NC-17"])
print(f"Revenue under $10K: {tiny.sum()} films; keep (rated) {(tiny & rated).sum()}, drop (NR or blank) {(tiny & ~rated).sum()}")

# New finding: "worldwide" revenue that is really the US gross
dom = k["domestic_gross"]
us_only = dom.notna() & (k["fin_revenue"] - dom).abs().le(0.01 * k["fin_revenue"])
loser = k["fin_revenue"] < 2.5 * k["budget_fixed"]
print("\nWorldwide revenue exactly equal to the US gross:")
display(pd.DataFrame({"films": [us_only[~non_us].sum(), us_only[non_us].sum()],
                      "% of group": [f"{us_only[~non_us].mean():.0%}", f"{us_only[non_us].mean():.0%}"],
                      "loser rate, these films": [f"{loser[us_only & ~non_us].mean():.0%}", f"{loser[us_only & non_us].mean():.0%}"],
                      "loser rate, everyone else": [f"{loser[~us_only & ~non_us].mean():.0%}", f"{loser[~us_only & non_us].mean():.0%}"]},
                     index=["US films", "non-US films"]))
display(k[us_only & non_us].sort_values("budget_fixed", ascending=False)[["title", "year", "countries", "budget_fixed", "fin_revenue", "fin_revenue_src"]].head(12))

### 8g. What 8f shows (run 2026-11-24)

1. **Budget disagreements:** 103 of 3,915 films with two budgets differ by more than 2×, and now use The Numbers. Most are extra-zero typos in TMDB (*Awake* $86M → $8.6M), in US and foreign films alike.
2. **The currency test doesn't work.** Only 11 one-source foreign budgets are extreme. Converting them "as local currency" gives impossible results for India (*RRR* would cost $1M; it cost about $70M). Most of the 11 are real dollar budgets of big hits (*RRR*, *Pathaan*, *2.0*, *Saaho*, *The Battle at Lake Changjin*). Currency errors look rare; typos are the real problem, and rule 1 handles those where a second source exists.
3. **Revenue under $10K:** 71 films left. Keep the 49 with a real MPAA rating; drop the 22 NR or blank.
4. **New problem: "worldwide" revenue that is only the US gross.** 618 films (515 US, 103 non-US) have revenue exactly equal to their US box office, so foreign box office is missing. They're labeled losers far more often (89% vs 62% for US films; 98% vs 70% non-US). For foreign films this is clearly wrong: *Nomad* shows $79K, *Kung Fu Jungle* $129K. For small US films it may be true (US-only releases), but probably not always.

**Decisions for Tanner**
- The 11 extreme foreign budgets: keep the known hits, drop the rest, or drop all 11?
- Revenue = US gross: drop the 103 foreign films? For the 515 US films, keep them but flag them for a sensitivity check?


### 8h. Tanner's decisions on 8g (2026-11-24)

- **Units:** all money is already in US dollars. Display in **$ thousands**.
- **Foreign currency:** none of the 11 extreme budgets is confirmed foreign currency. Drop the 4 that look wrong: *Chasing the Dragon*, *A Aa*, *Pink*, *Burnout*. Keep the real big-budget hits (*RRR*, *Pathaan*, *2.0*, *Saaho*, *The Battle at Lake Changjin*…).
- **Label:** change to **US box office < 1.25× budget** (about 67% losers). Population becomes films released in the US. This also fixes the 618 "worldwide = US gross" films. Drafted as amendment **A7** in `A10_contract_amendments_PROPOSED.md` (needs Matt).


## 9. Missing values, column by column

Population: films left after the §8 drops, released in the US (have a US box office), minus the 4 wrong foreign budgets (§8h). Money shown in **$ thousands**.

For each input: how many films are missing it, the share of the population, and (except for columns that already have a fill rule) a sample of the missing films. Tanner's hypothesis: many of these can be filled by looking the film up.

In [ ]:
pop = k[k["domestic_gross"].notna() & ~k["title"].isin(["Chasing the Dragon", "A Aa", "Pink", "Burnout"])].copy()
pop["budget_$k"] = (pop["budget_fixed"] / 1e3).round(0)
pop["us_gross_$k"] = (pop["domestic_gross"] / 1e3).round(0)
n_pop = len(pop)
print(f"Population: {n_pop:,} films")

INPUTS = ["runtime_min", "release_date", "genres_all", "mpaa_rating", "director", "writer", "production_companies",
          "distributor", "countries", "original_language", "franchise", "source_material", "creative_type"]
RULE = {"franchise": "blank = no franchise", "mpaa_rating": "blank -> NR, keep flag", "countries": "fill from original_language"}
miss = pd.DataFrame({"missing": pop[INPUTS].isna().sum()})
miss["% of population"] = (100 * miss["missing"] / n_pop).round(1)
miss["fill rule"] = miss.index.map(lambda c: RULE.get(c, ""))
display(miss.sort_values("missing", ascending=False))

BASE = ["title", "year", "budget_$k", "us_gross_$k", "countries"]
def show_missing(col, also=(), n=20, seed=0):
    m = pop[col].isna()
    print(f"{col}: {m.sum():,} missing ({m.mean():.1%} of {n_pop:,})")
    if col in RULE:
        print(f"  Fill rule: {RULE[col]}. No sample.")
        return
    cols = list(dict.fromkeys(BASE + list(also)))
    display(pop.loc[m, cols].sample(min(n, m.sum()), random_state=seed).sort_values("budget_$k", ascending=False))

### 9a. `creative_type`

In [ ]:
show_missing("creative_type", also=['genres_all', 'source_material'])

### 9b. `source_material`

In [ ]:
show_missing("source_material", also=['genres_all', 'creative_type'])

### 9c. `distributor`

In [ ]:
show_missing("distributor", also=['production_companies'])

### 9d. `writer`

In [ ]:
show_missing("writer", also=['director'])

### 9e. `director`

In [ ]:
show_missing("director", also=['writer'])

### 9f. `production_companies`

In [ ]:
show_missing("production_companies", also=['distributor'])

### 9g. `original_language`

In [ ]:
show_missing("original_language", also=[])

### 9h. `runtime_min`

In [ ]:
show_missing("runtime_min", also=['genres_all'])

### 9i. `mpaa_rating`

In [ ]:
show_missing("mpaa_rating", also=[])

### 9j. `countries`

In [ ]:
show_missing("countries", also=[])

### 9k. `franchise`

In [ ]:
show_missing("franchise", also=[])

### 9l. Filling from Wikidata (agreed 2026-11-24)

Snapshot `data/raw/wikidata/wikidata_film_people_2026-11-24.csv` (from `tools/get_wikidata_people.py`, matched by TMDB id). A feasibility test on 200 films showed director and writer agree with ours 96–97% of the time and runtime within 5 min 96%. Distributor was rejected (Wikidata lists worldwide and streaming distributors). Plan:
1. Fill blank `director`, `writer`, `runtime_min`.
2. Fill blank `source_material` only where Wikidata says what the film is based on.
3. Flag films where both have a director and they don't share one: a possible wrong match in the build.

In [ ]:
wd = pd.read_csv("../data/raw/wikidata/wikidata_film_people_2026-11-24.csv").set_index("tmdb_id")
w = wd.reindex(pop["id"].astype("Int64")).set_index(pop.index)
print(f"Population films found in Wikidata: {w['item'].notna().sum():,} of {n_pop:,}")

def names(s):
    return {t.strip().lower() for t in str(s).replace("|", ",").split(",") if t.strip() and str(s) != "nan"}

before = pop[["director", "writer", "runtime_min", "source_material"]].isna().sum()
pop["director"] = pop["director"].fillna(w["wd_director"].str.replace("|", ", "))
pop["writer"] = pop["writer"].fillna(w["wd_writer"].str.replace("|", ", "))
pop["runtime_min"] = pop["runtime_min"].fillna(w["wd_runtime_min"])

def source_from_type(t):
    t = str(t).lower()
    for keys, val in [(("comic", "manga", "graphic novel"), "Based on Comic/Graphic Novel"),
                      (("video game",), "Based on Game"),
                      (("television series", "tv series", "television program"), "Based on TV"),
                      (("musical",), "Based on Musical or Opera"), (("play",), "Based on Play"),
                      (("fairy tale", "folk", "legend", "myth"), "Based on Folk Tale/Legend/Fairytale"),
                      (("film",), "Remake"),
                      (("novel", "short story", "novella", "book", "literary work", "written work"), "Based on Fiction Book/Short Story")]:
        if any(k in t for k in keys):
            return val
    return np.nan
pop["source_material"] = pop["source_material"].fillna(w["wd_based_on_type"].map(source_from_type))
after = pop[["director", "writer", "runtime_min", "source_material"]].isna().sum()
display(pd.DataFrame({"missing before": before, "filled from Wikidata": before - after, "missing after": after,
                      "% missing after": (100 * after / n_pop).round(1)}))

In [ ]:
# Possible wrong matches: both sources name a director, and none overlap
ours = cu.loc[pop.index, "director"]          # our original value, before filling
both = ours.notna() & w["wd_director"].notna()
import unicodedata, difflib
def norm(n):
    n = unicodedata.normalize("NFKD", n).encode("ascii", "ignore").decode()
    return "".join(ch for ch in n.lower() if ch.isalpha() or ch == " ").split()
def same_person(a, b):
    # ignoring spaces, punctuation, accents and name order: same name, one name inside the other
    # ("Tarsem" / "Tarsem Singh"), same last name, or a near-identical spelling
    A, B = norm(a), norm(b)
    if not A or not B:
        return False
    if "".join(A) == "".join(B) or set(A) <= set(B) or set(B) <= set(A) or A[-1] == B[-1]:
        return True
    return difflib.SequenceMatcher(None, " ".join(sorted(A)), " ".join(sorted(B))).ratio() >= 0.8
def overlap(a, b):
    return any(same_person(x, y) for x in names(a) for y in names(b))
mismatch = both & pd.Series([not overlap(a, b) for a, b in zip(ours, w["wd_director"])], index=pop.index)
pop["director_mismatch"] = mismatch
print(f"Director disagrees with Wikidata: {mismatch.sum()} of {both.sum():,} films with both ({mismatch.mean():.1%} of population)")
display(pop.loc[mismatch, ["title", "year", "countries", "budget_$k", "us_gross_$k"]]
        .assign(our_director=ours[mismatch], wikidata_director=w.loc[mismatch, "wd_director"]).sort_values("budget_$k", ascending=False).head(30))

### 9m. What the Wikidata fill shows (run 2026-11-24)

| Column | Missing before | Filled | Missing after |
|---|---|---|---|
| director | 208 | 127 | 81 (1.6%) |
| writer | 266 | 136 | 130 (2.6%) |
| source_material | 891 | 158 | 733 (14.8%) |
| runtime_min | 28 | 0 | 28 (0.6%) |

**Wrong matches found:** 36 films (0.7%) have a director that disagrees with Wikidata after allowing for spelling, accents and name order. Most are real errors where our row picked up another film's data with the same title, mostly from Rotten Tomatoes: *A Christmas Carol* (2009) lists the director of a different *Christmas Carol* (it's Robert Zemeckis), *Elemental*, *Monsters vs Aliens*, *The Great Wall*, *Truth or Dare*. A few are spelling only (*Kim Gwang-sik*). **Proposed:** for these 36, use Wikidata's director, and check that their other Rotten Tomatoes columns (rating, writer, distributor) came from the right film.

**Still open:** creative type (no second source), distributor, remaining source material blanks.


### 9n. `source_group`: grouped source material, filled from TMDB keywords (agreed 2026-11-24)

Tanner's hypothesis: comic-book films vs original screenplays vs remakes is a key driver. The Numbers' 23 categories are grouped into 6: **Comic, Book, Real life, Remake, Other adaptation, Original**. Blanks are filled in order: The Numbers → Wikidata "based on" (§9l) → TMDB keywords.

Keywords are banned as a model *input* (crowd-edited after release), but here they only look up a fact that is known before production. A test on 3,801 films with both showed 74% agreement overall (Book 89%, Comic 72% but finds 93% of comic films, Original 72%, Remake 50%). `source_group_src` records where each value came from, so the model can be rerun without the keyword guesses.

In [ ]:
def kw_group(s):
    if pd.isna(s) or not str(s).strip():
        return np.nan
    t = {x.strip() for x in str(s).lower().split(",")}
    has = lambda *ks: any(k in x for x in t for k in ks)
    if has("based on comic", "superhero", "based on graphic novel", "based on manga", "comic book", "based on webcomic", "based on cartoon"):
        return "Comic"
    if has("remake", "reboot", "based on movie", "based on short film"):
        return "Remake"
    if has("based on true story", "biography", "biographical", "based on memoir", "autobiograph"):
        return "Real life"
    if has("based on novel", "based on young adult novel", "based on children's book", "based on short story", "based on magazine", "literary adaptation"):
        return "Book"
    if has("based on video game", "based on tv series", "based on play", "based on toy", "based on fairy tale", "based on myths",
           "based on theme park", "based on song", "based on anime"):
        return "Other adaptation"
    return "Original"

def tn_group(s):
    if pd.isna(s):
        return np.nan
    if "Comic" in s: return "Comic"
    if s == "Remake": return "Remake"
    if "Real Life" in s or "Factual" in s: return "Real life"
    if "Fiction Book" in s: return "Book"
    if s == "Original Screenplay": return "Original"
    return "Other adaptation"   # TV, game, play, musical, folk tale, toy, spin-off...

from_tn = cu.loc[pop.index, "source_material"].map(tn_group)        # The Numbers' own value
from_wd = pop["source_material"].map(tn_group).where(from_tn.isna())  # added by Wikidata in 9l
from_kw = pop["keywords"].map(kw_group)
pop["source_group"] = from_tn.fillna(from_wd).fillna(from_kw)
pop["source_group_src"] = np.select([from_tn.notna(), from_wd.notna(), from_kw.notna()], ["the_numbers", "wikidata", "tmdb_keywords"], "missing")

print("Where each film's source_group came from:")
display(pop["source_group_src"].value_counts().rename("films").to_frame().assign(**{"%": lambda d: (100 * d["films"] / n_pop).round(1)}))
print("source_group by origin:")
display(pd.crosstab(pop["source_group"].fillna("(missing)"), pop["source_group_src"], margins=True))

## 10. Each input vs return (training years only)

**Outcome shown:** each film's **return multiple** = US box office ÷ budget (`us_multiple`). The label (A7) cuts this at **1.25×**: at or above = winner, below = loser. Shown as a multiple, not dollars, because dollars mostly track budget (big films earn more dollars whether or not they profit). Tanner, 2026-11-24.

**Rows:** `in_population` films from **2000–2015 only** (the training years), so the 2016–19 test films stay unseen until evaluation. Data: `films_clean.csv` from the current pipeline.

**How to read a chart:** each row is a group. The dot is the group's **median** multiple; the bar spans the middle half of its films (25th–75th percentile). The dashed line is the 1.25× winner line: groups whose dot sits right of it have a typical film that's a winner. The axis is a log scale (each step doubles). Groups under 30 films are folded into "Other" or left out.

**Release year is not a model input** (Tanner): a new film's year is always later than any training year, so the model can't learn anything usable from it. It only defines the train / test / stress split.

People and companies can't be compared by name (most appear once), so §10d uses **experience**: how many earlier films the person or company had, counted from release dates (earlier films only).

In [ ]:
clean_now = pd.read_csv("../data/clean/films_clean.csv", low_memory=False)
clean_now["release_date"] = pd.to_datetime(clean_now["release_date"], format="mixed", errors="coerce")
tr = clean_now[clean_now["in_population"] & clean_now["year"].le(2015)].copy()
tr["loser"] = tr["loser"].astype(bool)
MED = tr["us_multiple"].median()
print(f"Training films: {len(tr):,}; median return {MED:.2f}x; winners (>= 1.25x) {(~tr['loser']).mean():.1%}")

SIGNAL = []   # spread between the best and worst group medians (groups with 30+ films)
TICKS = [0.05, 0.1, 0.25, 0.5, 1.25, 2, 4, 8]


def mult_table(groups, min_n=30, order=None):
    df = pd.DataFrame({"g": groups, "m": tr.loc[groups.index, "us_multiple"]})
    t = df.groupby("g", observed=True)["m"].agg(films="size", median="median",
                                                 q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75))
    t = t[t["films"] >= min_n]
    if order is not None:
        t = t.reindex([o for o in order if o in t.index])
    return t


def mult_plot(ax, groups, title, min_n=30, order=None, sort=False, record=None):
    """Median return multiple per group (dot) with the middle half of films (bar), top to bottom."""
    t = mult_table(groups, min_n, order)
    if sort:
        t = t.sort_values("median", ascending=False)
    y = np.arange(len(t))[::-1]
    ax.hlines(y, t["q25"], t["q75"], color=BLUE_LIGHT, lw=7)
    ax.plot(t["median"], y, "o", color=BLUE, ms=6)
    ax.axvline(1.25, color="#9a9890", lw=1, ls="--")
    ax.set_xscale("log"); ax.set_xlim(0.02, 10)
    ax.set_xticks(TICKS, [f"{x:g}x" for x in TICKS]); ax.minorticks_off()
    ax.set_yticks(y, [f"{i}  (n={n:,})" for i, n in zip(t.index, t["films"])])
    ax.set_xlabel("US box office / budget (median; bar = middle half)")
    ax.grid(False); ax.grid(axis="x", color="#e6e5e0", lw=0.6); ax.set_axisbelow(True)
    ax.set_title(title)
    if record:
        SIGNAL.append({"input": record, "groups": len(t),
                       "best group (median)": f"{t['median'].idxmax()} ({t['median'].max():.2f}x)",
                       "worst group (median)": f"{t['median'].idxmin()} ({t['median'].min():.2f}x)",
                       "best / worst": round(t["median"].max() / t["median"].min(), 1)})
    return t

### 10a. Money and timing: budget, runtime, release month, quarter, day of week

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
btier = pd.cut(tr["fin_budget"], [0, 5e6, 10e6, 20e6, 40e6, 80e6, np.inf], labels=["$1-5M", "$5-10M", "$10-20M", "$20-40M", "$40-80M", "$80M+"])
mult_plot(ax[0, 0], btier, "Budget tier", order=list(btier.cat.categories), record="budget")
rbin = pd.cut(tr["runtime_min"], [0, 85, 95, 105, 115, 130, np.inf], labels=["<85", "85-94", "95-104", "105-114", "115-129", "130+"], right=False)
mult_plot(ax[0, 1], rbin, "Runtime (minutes)", order=list(rbin.cat.categories), record="runtime_min")
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
mult_plot(ax[0, 2], tr["release_month"].astype(int).map(lambda m: MONTHS[m - 1]), "Release month", order=MONTHS, record="release_month")
mult_plot(ax[1, 0], tr["release_quarter"].astype(int).map(lambda q: f"Q{q}"), "Release quarter", order=["Q1", "Q2", "Q3", "Q4"], record="release_quarter")
dow = tr["release_date"].dt.day_name().where(lambda s: s.isin(["Wednesday", "Friday"]), "Other day")
mult_plot(ax[1, 1], dow, "Release day of week", order=["Wednesday", "Friday", "Other day"], record="release day of week")
ax[1, 2].axis("off")
plt.tight_layout(); plt.show()

### 10b. Content: MPAA rating, genre, source, creative type, franchise

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 11))
mult_plot(ax[0, 0], tr["mpaa_rating"].where(~tr["mpaa_rating_missing"], "blank (filled NR)"), "MPAA rating",
          order=["G", "PG", "PG-13", "R", "NC-17", "NR", "blank (filled NR)"], record="mpaa_rating")
gen = tr["genres_all"].str.split(",").explode().str.strip().dropna()
mult_plot(ax[0, 1], gen, "Genre (a film counts in each of its genres)", sort=True, record="genres_all")
mult_plot(ax[1, 0], tr["source_group"].fillna("(blank)"), "Source material group (Tanner's hypothesis)", sort=True, record="source_group")
mult_plot(ax[1, 1], tr["creative_type"].fillna("(blank)"), "Creative type", sort=True, record="creative_type")
plt.tight_layout(); plt.show()

In [ ]:
# Franchise: as tagged, and the version known before release (an earlier film in the same franchise was already out)
fr = tr["franchise"].ne("(none)")
allf = clean_now[clean_now["usable"] & clean_now["exclude_reason"].isna() & clean_now["franchise"].ne("(none)")][["franchise", "release_date"]]
first = allf.assign(f=allf["franchise"].str.split("|")).explode("f").groupby("f")["release_date"].min()
prior = pd.Series([fs != "(none)" and any(first.get(f, pd.NaT) < d for f in str(fs).split("|"))
                   for fs, d in zip(tr["franchise"], tr["release_date"])], index=tr.index)
known = pd.Series(np.select([prior, fr], ["sequel / later film", "first film of a franchise"], "no franchise"), index=tr.index)
fig, ax = plt.subplots(1, 2, figsize=(15, 3.2))
mult_plot(ax[0], fr.map({True: "tagged franchise", False: "no franchise"}), "Franchise (as tagged; tags first films in hindsight)")
mult_plot(ax[1], known, "Franchise, known before release", order=["no franchise", "first film of a franchise", "sequel / later film"],
          record="franchise (earlier film exists)")
plt.tight_layout(); plt.show()

### 10c. Origin: country and language

In [ ]:
us = tr["countries"].fillna("")
cgroup = np.select([us.eq("United States of America"), us.str.contains("United States")], ["US only", "US co-production"], "non-US")
fig, ax = plt.subplots(1, 2, figsize=(15, 3.2))
mult_plot(ax[0], pd.Series(cgroup, index=tr.index), "Production country", order=["US only", "US co-production", "non-US"], record="countries (US group)")
mult_plot(ax[1], tr["original_language"].eq("en").map({True: "English", False: "not English"}), "Original language", record="original_language")
plt.tight_layout(); plt.show()

### 10d. People and companies: experience from earlier films

For each training film, how many **earlier** released films (any year from 2000, in the cleaned data) its director, writer or production company already had. Names alone are too thin: most appear once.

In [ ]:
hist = clean_now[clean_now["usable"] & clean_now["exclude_reason"].isna()][["film_uid", "release_date", "director", "writer", "production_companies", "distributor"]]


def prior_count(col, top_only=False):
    """For each training film: the most earlier films any one of its listed people/companies had."""
    h = hist[["film_uid", "release_date", col]].dropna()
    h = h.assign(name=h[col].str.split(",")).explode("name")
    h["name"] = h["name"].str.strip()
    h = h.sort_values("release_date")
    h["prior"] = h.groupby("name").cumcount()          # films by this name released earlier
    best = h.groupby("film_uid")["prior"].max()
    return tr["film_uid"].map(best)


EXP = ["0 (first film)", "1", "2-4", "5-9", "10+"]
exp_bins = lambda s: pd.cut(s, [-1, 0, 1, 4, 9, np.inf], labels=EXP)
fig, ax = plt.subplots(1, 3, figsize=(17, 3.6))
for a, col in zip(ax, ["director", "writer", "production_companies"]):
    mult_plot(a, exp_bins(prior_count(col)).astype(object).fillna("(blank)"), f"{col}: earlier films",
              order=EXP + ["(blank)"], record=f"{col} experience")
plt.tight_layout(); plt.show()

In [ ]:
# Distributor: the biggest names vs everyone else (names are not standardized yet, §6h)
d = tr["distributor"].fillna("(blank)").str.split(",").str[0].str.strip()
top = d.value_counts().head(14).index
fig, ax = plt.subplots(figsize=(9, 6))
mult_plot(ax, d.where(d.isin(top), "Other"), "Distributor (first listed; top 14 by films)", sort=True, record="distributor")
plt.tight_layout(); plt.show()

### 10e. Do the "missing" flags carry signal?

Blanks aren't random (they cluster in small films), so a blank may itself predict the outcome.

In [ ]:
rows = []
for c in [c for c in tr.columns if c.endswith("_missing")]:
    blank = tr[c].astype(bool)
    if blank.sum() >= 30:
        mb, mf = tr.loc[blank, "us_multiple"].median(), tr.loc[~blank, "us_multiple"].median()
        rows.append({"input": c.removesuffix("_missing"), "films blank": int(blank.sum()),
                     "median return if blank": f"{mb:.2f}x", "median return if filled": f"{mf:.2f}x", "filled / blank": round(mf / mb, 1)})
display(pd.DataFrame(rows).sort_values("filled / blank", ascending=False).set_index("input"))

### 10f. Which inputs separate returns most?

Best / worst = the best group's median return divided by the worst group's (groups with 30+ films). A rough first ranking only: it ignores group size and overlap between inputs.

In [ ]:
display(pd.DataFrame(SIGNAL).sort_values("best / worst", ascending=False).set_index("input"))

### 10g. What the return comparisons show (run 2026-11-24, training years only)

Overall: 3,708 training films. **Median return 0.72×** (US box office ÷ budget); 33.1% reach the 1.25× winner line. Numbers below are group medians.

**Strong signals**
1. **MPAA rating:** G 1.31× → PG 1.08× → PG-13 0.96× → R 0.56× → **NR 0.05×**. 80% of NR films are non-US; NR mostly means a foreign or limited release.
2. **Origin:** US-only 1.02×, US co-productions 0.70×, **non-US 0.11×**. Language says the same (English 0.85×, not English 0.12×). Under a US box-office outcome, foreign films rarely earn back their budget in the US.
3. **Budget (clearer than with the winner rate):** films under $10M have a median of only 0.24–0.27×, vs 0.78–0.99× above $10M. Small films usually don't earn their budget back in US theaters, though their spread is the widest (a few break out).
4. **Sequels:** films with an earlier franchise film already released: 1.45×, vs 0.54× for non-franchise films.
5. **Genre:** Family 1.17×, Documentary 1.05×, Animation 1.00× at the top; Drama 0.50×, History 0.36×, Western and War 0.32× at the bottom.
6. **Creative type:** Super Hero 1.20×, Kids Fiction 1.18× at the top; Dramatization 0.57×, Historical Fiction 0.54× at the bottom.
7. **Distributor:** Warner Bros. 1.20× at the top, IFC Films 0.02× at the bottom (names not yet standardized).
8. **Experience:** directors on their first film 0.57× → 10+ earlier films 1.07×. Production companies with 10+ earlier films 0.93× vs 0.17–0.50× for less experienced ones.

**Tanner's source hypothesis: supported on returns.** Other adaptation (TV, games, toys) 1.06×, Comic 0.96×, Remake 0.95× vs Book 0.78×, Real life 0.67×, **Original 0.67×**. Adapted, known properties return roughly 40% more than original screenplays. (The winner-rate view hid this: the groups differ in how much they make, more than in how often they cross 1.25×.)

**Weak signals:** runtime (0.54× for 130+ min up to 0.92× under 85 min), release month (Sep–Oct and Apr–May 0.55–0.63×; Jun, Dec, Jul 0.82–0.84×), Wednesday openings 1.04× vs Friday 0.66×, quarter (0.69–0.81×).

**Not a model input: release year** (Tanner): a new film's year is always later than any training year.

**⚠️ Two warnings before these become model inputs**
- **Blank films return almost nothing (median 0.03–0.14×).** A blank usually means the databases never documented the film, and they document successful films better. So a blank partly records the outcome. The `_missing` flags (and "blank" groups) may leak it; test the model with and without them.
- **"First film of a franchise" is hindsight.** Those films return 1.72×, because a film only becomes the first of a franchise *after* it succeeds and gets sequels. Only "an earlier film was already released" (sequel) is known before release (matches Matt's A8 condition).


## 11. Deep dives (training years only)

Tanner's follow-up questions on §10 (2026-11-26). Same rows as §10: `in_population` films from 2000–2015 (3,708). Descriptive only: nothing here is a model input by itself.

| § | Question |
|---|---|
| 11a | What do the top three distributors' films look like (rating, genre, runtime, budget, return)? |
| 11b | Same for G-rated films |
| 11c | Does more budget guarantee a bigger return? What does the 1.25× line mean in dollars? |
| 11d | Why do documentaries return so much, with such a wide spread? |
| 11e | Release month sorted by median return |
| 11f | First films of a franchise (hindsight): what do they have in common with sequels? |

In [ ]:
# Shared helpers for §11
GEN = tr["genres_all"].fillna("").str.split(",").map(lambda l: [g.strip() for g in l if g.strip()])


def share_table(groups, col=None, values=None, top=8):
    """% of films in each group (columns) with each value of `col` (rows). `values` = list-valued Series instead of a column."""
    out = {}
    for name, m in groups.items():
        if values is not None:
            v = values[m].explode().dropna()
            out[name] = (100 * v.value_counts() / m.sum()).round(0)      # % of films with that value (films can have several)
        else:
            out[name] = (100 * tr.loc[m, col].fillna("(blank)").value_counts(normalize=True)).round(0)
    t = pd.DataFrame(out).fillna(0)
    return t.loc[t.iloc[:, 0].sort_values(ascending=False).index].head(top).astype(int)


def numbers_table(groups):
    rows = {}
    for name, m in groups.items():
        g = tr[m]
        rows[name] = {"films": len(g), "median budget ($M)": round(g["fin_budget"].median() / 1e6, 1),
                      "median US box office ($M)": round(g["domestic_gross"].median() / 1e6, 1),
                      "median return (x)": round(g["us_multiple"].median(), 2), "winners (>= 1.25x)": f"{(~g['loser']).mean():.0%}",
                      "median runtime (min)": g["runtime_min"].median()}
    return pd.DataFrame(rows)


ALL = {"all training films": pd.Series(True, index=tr.index)}

### 11a. The top three distributors

Distributor names aren't standardized yet, so the obvious variants of each studio's main label are merged first (e.g. "Warner Bros." + "Warner Bros. Pictures"; Disney + Buena Vista + Touchstone; Sony + Columbia). Specialty labels (Sony Pictures Classics, Fox Searchlight, Focus, New Line, Warner Independent) stay separate: they release a different kind of film. This map is a preview of the full standardization still owed (§6h).

In [ ]:
STUDIO = {"Warner Bros.": "Warner Bros.", "Warner Bros. Pictures": "Warner Bros.",
          "20th Century Fox": "20th Century Fox", "Fox": "20th Century Fox", "Twentieth Century Fox": "20th Century Fox",
          "Universal Pictures": "Universal", "Universal Studios": "Universal", "Universal": "Universal",
          "Walt Disney": "Disney", "Walt Disney Pictures": "Disney", "Buena Vista Pictures": "Disney", "Buena Vista": "Disney",
          "Touchstone Pictures": "Disney", "Walt Disney Studios Motion Pictures": "Disney",
          "Sony Pictures Entertainment": "Sony", "Columbia Pictures": "Sony", "Sony Pictures": "Sony", "Sony Pictures Releasing": "Sony",
          "Paramount Pictures": "Paramount", "Paramount": "Paramount",
          "Lionsgate Films": "Lionsgate", "Lionsgate": "Lionsgate", "Lions Gate Films": "Lionsgate"}
dist1 = tr["distributor"].str.split(",").str[0].str.strip()
studio = dist1.map(STUDIO).fillna(dist1)
print("Films per distributor after merging variants (training years):")
display(studio.value_counts().head(10).rename("films").to_frame().T)
TOP3 = studio.value_counts().head(3).index.tolist()
G3 = {s: studio.eq(s) for s in TOP3} | ALL
print(f"Top three: {TOP3}")
display(numbers_table(G3))
print("MPAA rating (% of each distributor's films):"); display(share_table(G3, "mpaa_rating"))
print("Genres (% of films tagged with each genre; films can have several):"); display(share_table(G3, values=GEN, top=10))
print("Source material group (%):"); display(share_table(G3, "source_group"))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
mult_plot(ax[0], studio.where(studio.isin(TOP3), "Everyone else"), "Return: top three distributors vs everyone else",
          order=TOP3 + ["Everyone else"])
rbin3 = pd.cut(tr["runtime_min"], [0, 95, 110, 125, np.inf], labels=["<95", "95-109", "110-124", "125+"], right=False)
rt_share = pd.DataFrame({s: (100 * rbin3[m].value_counts(normalize=True)).round(0) for s, m in G3.items()}).loc[rbin3.cat.categories]
rt_share.T.plot.barh(stacked=True, ax=ax[1], color=["#cde2fb", "#86b6ef", "#2a78d6", "#0d4a94"], width=0.6, edgecolor="white", linewidth=2)
ax[1].set(title="Runtime mix (% of films)", xlabel="% of films"); ax[1].legend(title="minutes", fontsize=8, ncols=4, loc="lower right", bbox_to_anchor=(1, 1.08))
ax[1].invert_yaxis(); ax[1].grid(False)
plt.tight_layout(); plt.show()

### 11b. G-rated films

In [ ]:
GG = {"G-rated": tr["mpaa_rating"].eq("G"), "PG": tr["mpaa_rating"].eq("PG")} | ALL
display(numbers_table(GG))
print("Genres (% of films tagged):"); display(share_table(GG, values=GEN, top=10))
print("Source material group (%):"); display(share_table(GG, "source_group"))
print("Creative type (%):"); display(share_table(GG, "creative_type", top=6))
print("Distributor (first listed, merged; %):")
display(pd.DataFrame({k: (100 * studio[m].value_counts(normalize=True)).round(0) for k, m in GG.items()}).fillna(0)
        .sort_values("G-rated", ascending=False).head(8).astype(int))
g_films = tr[tr["mpaa_rating"].eq("G")].sort_values("us_multiple", ascending=False)
print("Best and worst G-rated returns:")
display(pd.concat([g_films.head(6), g_films.tail(6)])[["title", "year", "genres_all", "fin_budget", "domestic_gross", "us_multiple"]])

### 11c. Budget vs return, and what 1.25× means in dollars

Does a bigger budget guarantee a bigger multiple? Left: every training film (log scales), with the median return in each budget tier. Right: the spread of returns (the "bell curve") for each tier.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].scatter(tr["fin_budget"] / 1e6, tr["us_multiple"].clip(lower=0.005), s=5, alpha=.25, color=BLUE)
med = tr.groupby(btier, observed=True).agg(b=("fin_budget", "median"), m=("us_multiple", "median"))
ax[0].plot(med["b"] / 1e6, med["m"], "o-", color="#0b0b0b", ms=6, label="median return per budget tier")
ax[0].axhline(1.25, color="#9a9890", ls="--", lw=1, label="1.25x winner line")
ax[0].set(xscale="log", yscale="log", xlabel="budget ($M, log)", ylabel="US box office / budget (log)", title="Budget vs return, every training film")
ax[0].set_yticks([0.01, 0.1, 0.25, 0.5, 1.25, 2, 4, 10], ["0.01x", "0.1x", "0.25x", "0.5x", "1.25x", "2x", "4x", "10x"]); ax[0].minorticks_off()
ax[0].set_xticks([1, 3, 10, 30, 100, 300], ["$1M", "$3M", "$10M", "$30M", "$100M", "$300M"])
ax[0].legend(fontsize=8)
bins = np.linspace(-3, 1.5, 46)
for k, (lab, m) in enumerate([("$1-10M", tr["fin_budget"] < 10e6), ("$10-40M", tr["fin_budget"].between(10e6, 40e6, inclusive="left")),
                              ("$40M+", tr["fin_budget"] >= 40e6)]):
    ax[1].hist(np.log10(tr.loc[m, "us_multiple"].clip(lower=10**-2.99)), bins=bins, histtype="step", lw=2,
               color=["#86b6ef", "#2a78d6", "#0d4a94"][k], label=f"{lab} (n={m.sum():,})", density=True)
ax[1].axvline(np.log10(1.25), color="#9a9890", ls="--", lw=1)
ax[1].set_xticks(np.log10([0.01, 0.1, 0.25, 0.5, 1.25, 2, 4, 10]), ["0.01x", "0.1x", "0.25x", "0.5x", "1.25x", "2x", "4x", "10x"])
ax[1].set(title="Spread of returns by budget (share of films)", xlabel="US box office / budget (log)", ylabel="density"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

d = tr.assign(profit=tr["domestic_gross"] - tr["fin_budget"], line=1.25 * tr["fin_budget"])
dollars = d.groupby(btier, observed=True).agg(films=("film_uid", "size"), median_budget=("fin_budget", "median"),
                                                 winner_line=("line", "median"), median_us_box_office=("domestic_gross", "median"),
                                                 median_profit=("profit", "median"), p75_profit=("profit", lambda s: s.quantile(.75)),
                                                 winners=("loser", lambda s: f"{(~s).mean():.0%}"))
for c in ["median_budget", "winner_line", "median_us_box_office", "median_profit", "p75_profit"]:
    dollars[c] = (dollars[c] / 1e6).round(1)
print("In dollars ($M). winner_line = 1.25 x the median budget (what a typical film in the tier must gross in the US).")
print("profit = US box office - budget (before marketing and the theaters' share).")
display(dollars.rename(columns={"median_budget": "median budget", "winner_line": "1.25x line", "median_us_box_office": "median US box office",
                                "median_profit": "median profit", "p75_profit": "75th pct profit"}))

### 11d. Documentaries

In [ ]:
doc = tr[GEN.map(lambda l: "Documentary" in l)].copy()
doc["also Music"] = doc["genres_all"].str.contains("Music")
GD = {"Documentary": GEN.map(lambda l: "Documentary" in l)} | ALL
display(numbers_table(GD))
print(f"Documentaries that are also tagged Music (concert films): {doc['also Music'].sum()} of {len(doc)}; "
      f"median return {doc.loc[doc['also Music'], 'us_multiple'].median():.2f}x vs {doc.loc[~doc['also Music'], 'us_multiple'].median():.2f}x for the rest")
print("Budget: documentaries are cheap, so a modest US gross is a big multiple:")
display(pd.cut(doc["fin_budget"], [0, 2e6, 5e6, 10e6, 30e6, np.inf], labels=["$1-2M", "$2-5M", "$5-10M", "$10-30M", "$30M+"])
        .value_counts().sort_index().rename("documentaries").to_frame().T)
cols_show = ["title", "year", "genres_all", "fin_budget", "domestic_gross", "us_multiple", "mpaa_rating"]
print("Top 12 and bottom 8 documentaries by return:")
display(pd.concat([doc.sort_values("us_multiple", ascending=False).head(12), doc.sort_values("us_multiple").head(8)])[cols_show])

### 11e. Release month, sorted by median return

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
mult_plot(ax, tr["release_month"].astype(int).map(lambda m: MONTHS[m - 1]), "Release month, best to worst median return", sort=True)
plt.tight_layout(); plt.show()

### 11f. First films of a franchise vs sequels vs everything else (hindsight, description only)

"First film" = the earliest film in a franchise that went on to have later films. **Nobody knew that at release**, so this can't be a model input; it describes what eventual franchise-starters had in common. Compared with sequels (an earlier film was already out, which *is* known before release) and with non-franchise films.

In [ ]:
GF = {"first film (hindsight)": known.eq("first film of a franchise"), "sequel / later film": known.eq("sequel / later film"),
      "no franchise": known.eq("no franchise")}
display(numbers_table(GF))
print("MPAA rating (%):"); display(share_table(GF, "mpaa_rating"))
print("Genres (% of films tagged):"); display(share_table(GF, values=GEN, top=10))
print("Source material group (%):"); display(share_table(GF, "source_group"))
print("Creative type (%):"); display(share_table(GF, "creative_type", top=6))
print("Distributor (merged; %):")
display(pd.DataFrame({k: (100 * studio[m].value_counts(normalize=True)).round(0) for k, m in GF.items()}).fillna(0)
        .sort_values("first film (hindsight)", ascending=False).head(8).astype(int))
print("Release month (%):")
display(pd.DataFrame({k: (100 * tr.loc[m, "release_month"].value_counts(normalize=True)).round(0) for k, m in GF.items()})
        .reindex(range(1, 13)).fillna(0).astype(int).set_axis(MONTHS).T)
print("Examples of first films:")
display(tr.loc[GF["first film (hindsight)"]].sort_values("domestic_gross", ascending=False)[["title", "year", "franchise", "fin_budget", "us_multiple"]].head(12))

### 11g. What the deep dives show (run 2026-11-26)

**Top three distributors (Warner Bros., 20th Century Fox, Sony; variants merged):** typical films return 0.99–1.17× vs 0.72× for all training films, and 41–46% are winners vs 33%. Their films are bigger (median budget $32–42M vs $20M), more PG-13 (43–55% vs 34%), almost never NR, more Action/Adventure and less Drama. Fox leans shorter and more family (24% PG); Warner leans longer, R and crime. Part of their edge is likely that they release wide in the US (not checked: we have no theater counts as an input).

**G-rated films:** as expected, 82% Family and 57% Animation, 49% released by Disney, shortest runtimes (median 88 min). Median return 1.31×, 52% winners. The best G returns are cheap documentaries and concert films (*March of the Penguins* 22.8×, *Hannah Montana Concert* 9.3×); the worst are cheap family films with no wide release (*Oogieloves* 0.05×).

**Budget vs return:** more money does **not** guarantee a multiple. Above about $10M the median return flattens near 0.8–1.0× however much is spent; what budget buys is a **narrower** spread (the $40M+ curve is tall and tight around 1×). Under $10M the spread is huge: most earn back little in US theaters (median 0.24–0.27×), but a few return 10×+.
**In dollars:** the median film **loses money in the US in every budget tier** (median profit −$0.6M to −$6.5M before marketing and the theaters' share). The 1.25× line means a typical $3M film must gross $3.8M in the US, a typical $125M film $156M. The upside grows with budget: the 75th-percentile profit is $1M for $1–5M films vs $62M for $80M+ films.

**Documentaries (76):** high median (1.05×) because they're cheap (median budget $2.5M), so a modest gross is a big multiple. The spread is wide because the genre mixes two kinds of film: wide-release crowd-pleasers (*Jackass* films, stand-up comedy concerts, *Fahrenheit 9/11*, *March of the Penguins*, music concerts: 13 concert films have a median of 2.89×) and limited-release issue documentaries that gross almost nothing (*Freakonomics* 0.03×, *Defamation* 0.01×). Note: the *Jackass* films and stand-up films are tagged Documentary in TMDB.

**Release month by median:** Jun, Dec, Jul, Mar, Jan at the top (0.81–0.84×); Sep, Oct, May, Apr at the bottom (0.55–0.63×). Fall is awards-season drama territory (low US returns); summer and the holidays are family and franchise season.

**First films of a franchise (hindsight):** they look like sequels, not like typical films: more PG/G, more Family, Action, Adventure, Fantasy and Animation-adjacent "Kids Fiction" (15% vs 4%), far less Drama (21% vs 58%), more "Other adaptation" (games, TV, toys) and Comic, bigger budgets ($35M vs $16M), released through the majors in summer. Median return 1.72× (vs 0.54× non-franchise).
- **Pre-release proxy for "franchise potential":** family / action-adventure / fantasy, PG or PG-13, an adapted property, a major studio and a summer date. Those are all inputs we already have, so the model can learn "franchise-starter-like" films without the hindsight tag.
- ⚠️ **Data starts in 2000, so some "first films" are really sequels to pre-2000 films** (*Toy Story 3*; *Mission: Impossible 2*). The sequel flag misses them. Fix: installment number from the title or Wikidata "follows" (P155); see the sequel idea in `FEATURE_IDEAS.md`.


## 12. Returns over time, distributors by return, big vs small budgets, franchise combos

Tanner's questions (2026-11-26):
- **12a** Has return gone down over the years? (all years 2000–2023, **trend charts only**; test years 2016–19 and COVID years 2020–23 are marked)
- **12b** Distributors ranked by median return, not film count
- **12c** In raw dollars, is a big budget at a low return better than a small budget at a high return?
- **12d** In franchises, are R-rated comedies the best bet? How much of the group are they, and how reliable?

Everything except 12a uses training years only (2000–2015).

**Inflation:** the return multiple needs no adjustment (box office and budget are in the same year's dollars). Dollar amounts are shown in **2023 dollars** using the US CPI-U annual average (BLS series CUUR0000SA0).

In [ ]:
# BLS CPI-U, US city average, all items, annual average (1982-84 = 100)
CPI = {2000: 172.2, 2001: 177.1, 2002: 179.9, 2003: 184.0, 2004: 188.9, 2005: 195.3, 2006: 201.6, 2007: 207.342,
       2008: 215.303, 2009: 214.537, 2010: 218.056, 2011: 224.939, 2012: 229.594, 2013: 232.957, 2014: 236.736,
       2015: 237.017, 2016: 240.007, 2017: 245.120, 2018: 251.107, 2019: 255.657, 2020: 258.811, 2021: 270.970,
       2022: 292.655, 2023: 304.702}
to2023 = lambda year: CPI[2023] / pd.Series(year).map(CPI).values
for df in (tr,):
    f = to2023(df["year"].astype(int))
    df["budget_23"], df["gross_23"] = df["fin_budget"].values * f, df["domestic_gross"].values * f
    df["profit_23"] = df["gross_23"] - df["budget_23"]
allp = clean_now[clean_now["in_population"]].copy()
f = to2023(allp["year"].astype(int))
allp["budget_23"], allp["gross_23"] = allp["fin_budget"].values * f, allp["domestic_gross"].values * f
print(f"Training films: {len(tr):,}; all population films for the trend: {len(allp):,}")

### 12a. Has return gone down over the years? (all years, trend only)

In [ ]:
yr = allp.groupby("year").agg(films=("film_uid", "size"), med=("us_multiple", "median"),
                               q25=("us_multiple", lambda s: s.quantile(.25)), q75=("us_multiple", lambda s: s.quantile(.75)),
                               winners=("loser", lambda s: 1 - s.astype(bool).mean()),
                               budget=("budget_23", "median"), gross=("gross_23", "median"))
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for a in ax:
    a.axvspan(2015.5, 2019.5, color="#f2c6ad", alpha=.35, lw=0); a.axvspan(2019.5, 2023.5, color="#bfe8d6", alpha=.45, lw=0)
ax[0].fill_between(yr.index, yr["q25"], yr["q75"], color=BLUE_LIGHT); ax[0].plot(yr.index, yr["med"], "o-", color=BLUE, ms=4)
ax[0].axhline(1.25, color="#9a9890", ls="--", lw=1); ax[0].set(yscale="log", title="Median return per year (band = middle half)", ylabel="US box office / budget")
ax[0].set_yticks([0.1, 0.25, 0.5, 1, 1.25, 2, 4], ["0.1x", "0.25x", "0.5x", "1x", "1.25x", "2x", "4x"]); ax[0].minorticks_off()
ax[1].plot(yr.index, 100 * yr["winners"], "o-", color=BLUE, ms=4); ax[1].set(title="Winners (>= 1.25x) per year, %", ylim=(0, 60))
ax[2].plot(yr.index, yr["budget"] / 1e6, "o-", color="#0d4a94", ms=4, label="median budget")
ax[2].plot(yr.index, yr["gross"] / 1e6, "o-", color="#86b6ef", ms=4, label="median US box office")
ax[2].set(title="Median budget and US box office (\\$M, 2023 dollars)", ylim=(0, None)); ax[2].legend(fontsize=8)
for a in ax:
    a.text(2017.5, a.get_ylim()[1], "test", ha="center", va="top", fontsize=8, color=INK2)
    a.text(2021.5, a.get_ylim()[1], "COVID", ha="center", va="top", fontsize=8, color=INK2)
plt.tight_layout(); plt.show()
period = pd.cut(allp["year"], [1999, 2005, 2010, 2015, 2019, 2023], labels=["2000-05", "2006-10", "2011-15", "2016-19 (test)", "2020-23 (COVID)"])
display(allp.groupby(period, observed=True).agg(films=("film_uid", "size"), median_return=("us_multiple", "median"),
        winners=("loser", lambda s: f"{1 - s.astype(bool).mean():.0%}"), median_budget_2023_M=("budget_23", lambda s: round(s.median() / 1e6, 1)),
        median_us_box_office_2023_M=("gross_23", lambda s: round(s.median() / 1e6, 1))).round(2))

### 12b. Distributors ranked by median return (training years, 30+ films)

In [ ]:
sd = pd.DataFrame({"studio": studio, "m": tr["us_multiple"], "win": ~tr["loser"], "b": tr["fin_budget"]})
rank = sd.groupby("studio").agg(films=("m", "size"), median_return=("m", "median"),
                                winners=("win", "mean"), median_budget_M=("b", lambda s: s.median() / 1e6))
rank = rank[rank["films"] >= 30].sort_values("median_return", ascending=False)
rank["winners"] = (100 * rank["winners"]).round(0).astype(int).astype(str) + "%"
display(rank.round({"median_return": 2, "median_budget_M": 1}))
fig, ax = plt.subplots(figsize=(9, 7))
mult_plot(ax, studio.where(studio.isin(rank.index)), "Distributors (30+ films), best to worst median return", sort=True)
plt.tight_layout(); plt.show()

### 12c. Big budget at a lower return, or small budget at a higher return: which makes more money?

All dollars in 2023 dollars, training years. **Profit = US box office − budget** (before marketing and the theaters' share, so real studio profit is lower). Three ways to look at it:
1. **Per film:** the average and typical (median) profit of one film in each budget tier.
2. **Per dollar invested:** if you spread $100M across many films in a tier, how much US box office comes back (total gross ÷ total budget)?
3. **Risk:** how often a film in the tier doesn't even earn its budget back.

In [ ]:
TIERS = ["$1-5M", "$5-10M", "$10-20M", "$20-40M", "$40-80M", "$80M+"]
bt = pd.cut(tr["budget_23"], [0, 5e6, 10e6, 20e6, 40e6, 80e6, np.inf], labels=TIERS)
g = tr.groupby(bt, observed=True)
money = pd.DataFrame({
    "films": g.size(),
    "median return": g["us_multiple"].median().round(2),
    "avg profit per film ($M)": (g["profit_23"].mean() / 1e6).round(1),
    "median profit per film ($M)": (g["profit_23"].median() / 1e6).round(1),
    "best 10% of films earn at least ($M)": (g["profit_23"].quantile(.9) / 1e6).round(1),
    "US gross back per $100 invested": (100 * g["gross_23"].sum() / g["budget_23"].sum()).round(0),
    "% that lost money": (100 * g.apply(lambda d: (d["profit_23"] < 0).mean())).round(0),
})
display(money)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
x = np.arange(len(money))
ax[0].bar(x - 0.2, money["avg profit per film ($M)"], 0.4, color=BLUE, label="average")
ax[0].bar(x + 0.2, money["median profit per film ($M)"], 0.4, color=BLUE_LIGHT, label="median (typical film)")
ax[0].axhline(0, color=INK2, lw=0.8); ax[0].set_xticks(x, [t.replace("$", "\\$") for t in money.index]); ax[0].legend(fontsize=8)
ax[0].set(title="Profit per film (\\$M, 2023 dollars)", ylabel="\\$M")
ax[1].bar(x, money["US gross back per $100 invested"], 0.6, color=BLUE); ax[1].axhline(100, color="#9a9890", ls="--", lw=1)
ax[1].axhline(125, color="#9a9890", ls=":", lw=1); ax[1].set_xticks(x, [t.replace("$", "\\$") for t in money.index])
ax[1].set(title="US gross back per \\$100 invested (dashed = \\$100, dotted = 1.25x)")
ax[2].bar(x, money["% that lost money"], 0.6, color=BLUE); ax[2].set_xticks(x, [t.replace("$", "\\$") for t in money.index]); ax[2].set(title="% of films that didn't earn back their budget", ylim=(0, 100))
for a in ax:
    a.grid(False); a.grid(axis="y", color="#e6e5e0", lw=0.6); a.set_axisbelow(True); a.tick_params(axis="x", labelsize=8)
plt.tight_layout(); plt.show()

ex = tr.assign(budget_M=tr["budget_23"] / 1e6, gross_M=tr["gross_23"] / 1e6, profit_M=tr["profit_23"] / 1e6)
print("Tanner's example in this data: a $100M+ film at a modest return vs a ~$1-2M film at a big return")
display(pd.concat([ex[(ex["budget_M"] >= 100) & ex["us_multiple"].between(1.0, 1.3)].sample(3, random_state=1),
                   ex[(ex["budget_M"] < 2.5) & ex["us_multiple"].between(3, 6)].sample(3, random_state=1)])
        [["title", "year", "budget_M", "gross_M", "us_multiple", "profit_M"]].round(1))

#### 12c (cont.). Quantity vs size: how many films, and where the dollars go

Tanner: are the smaller studios making money by releasing *more* small films, while the big studios put fewer, larger bets? First, each budget tier's share of films vs its share of the money; then each distributor's "portfolio": how many films, how big, and what came back on all of it together.

In [ ]:
share = pd.DataFrame({
    "films": g.size(),
    "% of films": (100 * g.size() / len(tr)).round(1),
    "total invested ($M)": (g["budget_23"].sum() / 1e6).round(0),
    "% of dollars invested": (100 * g["budget_23"].sum() / tr["budget_23"].sum()).round(1),
    "total US gross ($M)": (g["gross_23"].sum() / 1e6).round(0),
    "% of all US gross": (100 * g["gross_23"].sum() / tr["gross_23"].sum()).round(1),
    "total profit ($M)": (g["profit_23"].sum() / 1e6).round(0),
})
display(share)

port = pd.DataFrame({"studio": studio, "b": tr["budget_23"], "gr": tr["gross_23"], "pr": tr["profit_23"], "small": tr["budget_23"] < 10e6})
pg = port.groupby("studio")
pf = pd.DataFrame({"films": pg.size(), "% under $10M": (100 * pg["small"].mean()).round(0),
                   "median budget ($M)": (pg["b"].median() / 1e6).round(1), "total invested ($M)": (pg["b"].sum() / 1e6).round(0),
                   "total US gross ($M)": (pg["gr"].sum() / 1e6).round(0), "gross per $100 invested": (100 * pg["gr"].sum() / pg["b"].sum()).round(0),
                   "total profit ($M)": (pg["pr"].sum() / 1e6).round(0), "median return": pg.apply(lambda d: (d["gr"] / d["b"]).median()).round(2)})
pf = pf[pf["films"] >= 30].sort_values("gross per $100 invested", ascending=False)
print("Distributor portfolios (30+ training films, 2023 dollars), sorted by US gross per $100 invested:")
display(pf)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(pf["median budget ($M)"], pf["gross per $100 invested"], s=pf["films"] * 1.5, color=BLUE, alpha=.6)
for s, r in pf.iterrows():
    ax.annotate(f"{s} ({r['films']:.0f})", (r["median budget ($M)"], r["gross per $100 invested"]), fontsize=7, xytext=(5, 3), textcoords="offset points")
ax.axhline(100, color="#9a9890", ls="--", lw=1); ax.axhline(125, color="#9a9890", ls=":", lw=1)
ax.set(xlabel="median budget per film (\\$M, 2023)", ylabel="US gross per \\$100 invested (whole slate)",
       title="Distributor slates: film size vs what the whole slate earned back (dot size = number of films)")
plt.tight_layout(); plt.show()

**Quantity vs size (training years, 2023 dollars):**
- **$80M+ films are 20% of releases but take 60% of the money and bring back 57% of US box office and 44% of all profit.** Films under $10M are 24% of releases but only 2% of the money (and 3.5% of box office).
- **Per dollar, the best slates are mid-size specialty labels:** Fox Searchlight ($169 back per $100), Lionsgate ($142, 26% of its films under $10M) and New Line ($136). The majors cluster at $130–133; **Warner Bros. is lowest of the majors ($114)**.
- **The small-film labels lose money as a slate in US theaters:** Sony Pictures Classics, IFC, Roadside and Magnolia (44–61% of their films under $10M) earn back only $7–62 per $100. "Quantity" alone doesn't make money. These labels likely earn elsewhere (international sales, video on demand, streaming deals, awards), which our US-theatrical label doesn't see.
- So both strategies can work, but not by volume: Lionsgate and Fox Searchlight win with **cheap films that get a real US release**; the majors win with **a few big, wide releases**.

### 12d. Franchise films by rating × genre

Two franchise groups: **sequels** (an earlier franchise film was already out: known before release) and **all tagged franchise films** (adds first films, hindsight). For each rating × genre combo with 10+ films: its share of the group, median return, how often it's a winner, and how bad its weakest quarter is. A film counts in each of its genres.

In [ ]:
def combo_table(mask, min_n=10):
    sub = tr[mask].assign(genre=GEN[mask]).explode("genre").dropna(subset=["genre"])
    n_group = mask.sum()
    t = sub.groupby(["mpaa_rating", "genre"]).agg(films=("film_uid", "size"), median_return=("us_multiple", "median"),
                                                  weakest_quarter=("us_multiple", lambda s: s.quantile(.25)),
                                                  winners=("loser", lambda s: 1 - s.astype(bool).mean()))
    t = t[t["films"] >= min_n]
    t["% of group"] = (100 * t["films"] / n_group).round(0).astype(int)
    t["winners"] = (100 * t["winners"]).round(0).astype(int).astype(str) + "%"
    return t.sort_values("median_return", ascending=False).round(2)[["films", "% of group", "median_return", "winners", "weakest_quarter"]]

seq = known.eq("sequel / later film")
tagged = tr["franchise"].ne("(none)")
print(f"Sequels: {seq.sum()} films (median return {tr.loc[seq, 'us_multiple'].median():.2f}x)")
display(combo_table(seq).head(12))
print(f"All tagged franchise films (incl. first films, hindsight): {tagged.sum()} films (median {tr.loc[tagged, 'us_multiple'].median():.2f}x)")
display(combo_table(tagged).head(12))

In [ ]:
# Is an R-rated comedy a solid bet? Every R comedy, by franchise status
rc = tr["mpaa_rating"].eq("R") & GEN.map(lambda l: "Comedy" in l)
grp = pd.Series(np.where(seq, "sequel", np.where(tagged, "first film (hindsight)", "no franchise")), index=tr.index)
summary = tr[rc].groupby(grp[rc]).agg(films=("film_uid", "size"), median_return=("us_multiple", "median"),
                                      winners=("loser", lambda s: f"{1 - s.astype(bool).mean():.0%}"),
                                      weakest_quarter=("us_multiple", lambda s: s.quantile(.25)), best_quarter=("us_multiple", lambda s: s.quantile(.75)))
display(summary.round(2))
print("R-rated comedy sequels, best to worst:")
display(tr[rc & seq].sort_values("us_multiple", ascending=False)[["title", "year", "franchise", "fin_budget", "domestic_gross", "us_multiple"]])

### 12e. What these show (run 2026-11-26)

**Has return gone down?** Yes, then partly back. Median return: 0.84× in 2000–05 → 0.64–0.67× in 2006–15 → 0.76× in 2016–19 (test) → 0.59× in 2020–23 (COVID; 2020 alone 0.35×). Winners: 37% → 31–32% → 36% → 26%. In 2023 dollars, the typical film's budget and US box office both fell by about half from 2000 to 2015. Caution: part of this is **which films are in the data** (more small and foreign films get recorded in later years), not only the market. A model trained on 2000–15 will meet a somewhat different mix in 2016–19, and a very different one in 2020–23 (A5 stress test).

**Distributors by median return:** Disney is first (1.28×, 53% winners), then Paramount, Sony, Fox and Universal (1.16–1.19×). **Warner Bros. distributes the most films but ranks 11th (0.99×).** The specialty and indie labels sit at the bottom: Sony Pictures Classics 0.29×, Roadside 0.15×, IFC and Magnolia 0.02× (they release small films in few US theaters, often after a festival or alongside video on demand).

**Big budget at a lower return vs small budget at a higher return** (2023 dollars; profit = US box office − budget):
- **Per film, bigger makes more money.** Average profit per film rises from $3M ($1–5M films) to $21M ($80M+ films). Tanner's example holds: *Ocean's Twelve* at 1.1× made $25M; *The Squid and the Whale* at 4.9× made $9M.
- **Per dollar invested, small films return more on average.** $100 spread across $1–5M films brings back $212 in US box office, vs $114 across $80M+ films.
- **But small films are a lottery.** That average comes from rare breakouts: 69–74% of films under $10M don't earn back their budget, vs 52–55% above $40M. The *typical* film loses money in every tier.
- **Answer:** if the studio can make many small films, the portfolio returns more per dollar (a few hits pay for many flops). For any **single** film, the bigger budget makes more dollars and is slightly less likely to lose money. Big films also carry the biggest absolute losses, and marketing (not in our data) scales with budget.

**R-rated comedies in franchises:** Tanner's read is **partly right**.
- Among **sequels**, R comedy is 5th (1.85×, 70% winners). Ahead of it: PG-13 romance (2.52×: *Madea*, *Twilight*, *Think Like a Man*, *Step Up*), PG-13 drama and comedy, and R horror (2.00×). PG-13 action isn't near the top; PG-13 sci-fi and thriller sit around 1.3×.
- Among **all franchise films** (with first films, hindsight), R comedy is 4th (2.49×, 77% winners).
- **Share of the group:** R comedies are only **8% of sequels (23 films)** and 9% of all franchise films (64).
- **Always a solid investment? No.** The safety comes from being a sequel, not from R comedy. R comedies **outside a franchise** (443 films) have a median of 0.57× and only 30% winners. R comedy sequels are good (weakest quarter still 1.17×), but some still flop: *Sex and the City 2* 0.95×, *Hot Tub Time Machine 2* 0.88×. *Jackass* films (also tagged documentary) make up 3 of the top 3.


### 12f. Trends over time by rating, genre, source, budget and franchise status

All years 2000–2023 in **4-year periods** (trend description only, as in 12a). Each small panel is one group: the dot is the group's median return in that period, the gray line is the median of **all** films in that period (so you can see a group pulling ahead or falling behind the market). Hollow dots = fewer than 15 films in that period (treat with caution). Shaded: test (2016–19) and COVID (2020–23). Under each grid, a table of the same medians.

In [ ]:
PERIODS = ["2000-03", "2004-07", "2008-11", "2012-15", "2016-19", "2020-23"]
allp["period"] = pd.cut(allp["year"], [1999, 2003, 2007, 2011, 2015, 2019, 2023], labels=PERIODS)
allp["winner"] = ~allp["loser"].astype(bool)
GEN_ALL = allp["genres_all"].fillna("").str.split(",").map(lambda l: [g.strip() for g in l if g.strip()])
market = allp.groupby("period", observed=True)["us_multiple"].median()


def trend_grid(groups, title, order, ncols=4, min_dot=15):
    """Small multiples: median return per period for each group vs the whole market."""
    df = pd.DataFrame({"g": groups, "m": allp.loc[groups.index, "us_multiple"], "p": allp.loc[groups.index, "period"]})
    t = df.groupby(["g", "p"], observed=True)["m"].agg(["median", "size"])
    order = [o for o in order if o in t.index.get_level_values(0)]
    nrows = int(np.ceil(len(order) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 2.6 * nrows), sharey=True, squeeze=False)
    x = np.arange(len(PERIODS))
    for ax, g in zip(axes.flat, order):
        s = t.loc[g].reindex(PERIODS)
        ax.axvspan(3.5, 4.5, color="#f2c6ad", alpha=.35, lw=0); ax.axvspan(4.5, 5.5, color="#bfe8d6", alpha=.45, lw=0)
        ax.plot(x, market.reindex(PERIODS).values, color="#9a9890", lw=1.2)
        ax.plot(x, s["median"].values, color=BLUE, lw=1.8)
        small = s["size"].fillna(0) < min_dot
        ax.plot(x[~small.values], s["median"][~small].values, "o", color=BLUE, ms=5)
        ax.plot(x[small.values], s["median"][small].values, "o", mfc="white", mec=BLUE, ms=5)
        ax.axhline(1.25, color="#9a9890", ls="--", lw=0.8)
        ax.set_title(f"{g}  (n={int(s['size'].sum()):,})", fontsize=9)
        ax.set_yscale("log"); ax.set_ylim(0.05, 6)
        ax.set_yticks([0.1, 0.25, 0.5, 1.25, 3], ["0.1x", "0.25x", "0.5x", "1.25x", "3x"]); ax.minorticks_off()
        ax.set_xticks(x, ["00-03", "04-07", "08-11", "12-15", "16-19", "20-23"], fontsize=7)
        ax.grid(False); ax.grid(axis="y", color="#e6e5e0", lw=0.6)
    for ax in list(axes.flat)[len(order):]:
        ax.axis("off")
    fig.suptitle(title + "   (blue = group median, gray = all films, dashed = 1.25x)", x=0.01, ha="left", fontsize=10)
    plt.tight_layout(); plt.show()
    table = t["median"].unstack().reindex(index=order, columns=PERIODS).round(2)
    table["change 00-03 to 16-19"] = (table["2016-19"] / table["2000-03"]).round(2).astype(str) + "x"
    display(table)

In [ ]:
rating = allp["mpaa_rating"].where(~allp["mpaa_rating_missing"].astype(bool), "NR")
trend_grid(rating, "MPAA rating", ["G", "PG", "PG-13", "R", "NR"], ncols=5)

In [ ]:
gen_long = GEN_ALL.explode().dropna()
top_genres = gen_long.value_counts().head(12).index.tolist()
trend_grid(gen_long[gen_long.isin(top_genres)], "Genre (12 most common; a film counts in each of its genres)", top_genres, ncols=4)

In [ ]:
trend_grid(allp["source_group"].fillna("(blank)"), "Source material group",
           ["Comic", "Other adaptation", "Remake", "Book", "Real life", "Original"], ncols=6)

In [ ]:
btier_all = pd.cut(allp["budget_23"], [0, 10e6, 40e6, 100e6, np.inf], labels=["under $10M", "$10-40M", "$40-100M", "$100M+"])
trend_grid(btier_all.astype(object), "Budget tier (2023 dollars)", ["under $10M", "$10-40M", "$40-100M", "$100M+"], ncols=4)

In [ ]:
# Franchise status known at release, for every year (earlier film in the franchise already out)
prior_all = pd.Series([fs != "(none)" and any(first.get(f, pd.NaT) < d for f in str(fs).split("|"))
                       for fs, d in zip(allp["franchise"], pd.to_datetime(allp["release_date"], format="mixed", errors="coerce"))], index=allp.index)
fstat = pd.Series(np.select([prior_all, allp["franchise"].ne("(none)")], ["sequel / later film", "first film (hindsight)"], "no franchise"), index=allp.index)
trend_grid(fstat, "Franchise status", ["sequel / later film", "first film (hindsight)", "no franchise"], ncols=3)
print(f"Share of films that are sequels, by period:")
display((100 * (fstat == "sequel / later film").groupby(allp["period"], observed=True).mean()).round(1).rename("% sequels").to_frame().T)

### 12g. What the trends show (run 2026-11-26)

Median return per 4-year period; "market" = all films in that period (0.85× in 2000–03, 0.67–0.68× in 2004–15, 0.76× in 2016–19, 0.59× in 2020–23).

**Rising**
- **Horror** is the clearest riser: about 1× until 2015, then **2.03× in 2016–19** and 1.55× in 2020–23, far above the market. It held up best through COVID. **Mystery** rose with it (1.22× in 2016–19).
- **G-rated** films stay the best rating every period (1.24–1.84×), and recovered after the mid-2000s dip.
- **Comic** adaptations dipped in 2008–11 (0.67×) and came back to 1.32× in 2016–19.

**Falling**
- **Family** films lost their edge: 1.61× in 2000–03 → 0.98× in 2016–19 → 0.73× in 2020–23, converging on the market (likely streaming taking family viewing).
- **Adventure** (1.09× → 0.61×) and **Action** (0.82× → 0.56×) slid steadily.
- **Remakes** were strong early (1.24×) but swing a lot (0.46× in 2012–15).

**Steady**
- **Comedy** sits above the market in every period (0.79–1.12×). **Drama** sits below it in every period (0.32–0.67×).
- **R** is consistently below the market; **PG-13** roughly at it until COVID (0.68×).
- **Sequels** keep a large premium in every period (1.21–1.76× vs 0.43–0.68× for non-franchise films), though it narrowed. Their share grew from 3% of films (2000–03) to 21% (2020–23), partly because our data starts in 2000, so early sequels to pre-2000 films aren't recognized.
- **Budget:** films over $40M held near 0.85–1.03× until 2019 and then fell hardest in COVID (0.54–0.60×). Films under $10M swing the most (0.13–0.68×).

**For the model:** the 2016–19 test period looks a lot like 2012–15 except for the horror/mystery jump, so a model trained on 2000–15 may under-predict horror. 2020–23 differs on almost every group (A5 stress test is justified). Any trend here is description: none of it is a model input (release year is excluded).


## 13. Sequels hiding in titles (prototype feature)

The franchise check (§10b) only counts a film as a sequel when an **earlier franchise film is in our data**, and our data starts in 2000. So *Toy Story 3* (Toy Story 1995 and 2 1999 are pre-2000) looks like a "first film". Idea: read the **title** for an installment number: a trailing number or Roman numeral (*Shrek 2*, *Saw III*), "Part 2", "Chapter 3", "Episode II".

Checked on all usable films (it's data engineering, not peeking at outcomes): how many sequels it finds that the franchise check misses, and what it gets wrong.

In [ ]:
import re
ROMAN = {"II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6, "VII": 7, "VIII": 8, "IX": 9, "X": 10}
WORDS = {"two": 2, "three": 3, "four": 4, "five": 5, "second": 2, "third": 3, "fourth": 4}
TITLE_PATTERNS = [
    r"\b(?:part|chapter|episode|vol\.?|volume)\s+(\d+|[ivx]+|two|three|four|five)\b",   # Part 2, Chapter Three, Episode II
    r"(?<!^)\b(\d)\b(?!\d)(?=\s*(?::|-|–|$|\())",                                     # Shrek 2, Saw 3D excluded below, Toy Story 3: ...
    r"\b(II|III|IV|VI|VII|VIII|IX)\b(?=\s*(?::|-|–|$|\())",                             # Saw II, Rocky Balboa excluded, Mission: Impossible III
]


def installment_from_title(title):
    t = str(title).strip()
    for pat in TITLE_PATTERNS:
        m = re.search(pat, t, flags=re.IGNORECASE if "part" in pat else 0)
        if m:
            v = m.group(1)
            n = int(v) if v.isdigit() else ROMAN.get(v.upper(), WORDS.get(v.lower()))
            if n and 2 <= n <= 12:
                return n
    return np.nan


u = clean_now[clean_now["usable"] & clean_now["exclude_reason"].isna()].copy()
u["release_date"] = pd.to_datetime(u["release_date"], format="mixed", errors="coerce")
u["title_installment"] = u["title"].map(installment_from_title)
u["prior_in_data"] = [fs != "(none)" and any(first.get(f, pd.NaT) < d for f in str(fs).split("|")) for fs, d in zip(u["franchise"], u["release_date"])]
u["title_says_sequel"] = u["title_installment"].notna()
both = pd.crosstab(u["title_says_sequel"].map({True: "title has a number", False: "no number in title"}),
                   u["prior_in_data"].map({True: "earlier franchise film in data", False: "no earlier film in data"}), margins=True)
display(both)
for t in ["Toy Story 3", "Mission: Impossible II", "Shrek 2", "The Dark Knight", "Harry Potter and the Chamber of Secrets"]:
    r = u[u["title"].eq(t)]
    if len(r):
        r = r.iloc[0]
        print(f"{t:<42} title number: {r['title_installment']!s:<5} earlier film in data: {r['prior_in_data']}")

In [ ]:
print("Found ONLY by the title (sequels the franchise check misses):")
only_title = u[u["title_says_sequel"] & ~u["prior_in_data"]]
display(only_title[["title", "year", "franchise", "title_installment"]].head(30))
print(f"{len(only_title)} films. Of these, {only_title['franchise'].ne('(none)').sum()} have a franchise tag (so the franchise "
      f"exists but its earlier films are pre-2000 or missing), {only_title['franchise'].eq('(none)').sum()} have none.")
print("\nFound ONLY by the franchise check (sequels with no number in the title):")
display(u[~u["title_says_sequel"] & u["prior_in_data"]][["title", "year", "franchise"]].head(12))

In [ ]:
# Does adding title-detected sequels change the picture? (training films only)
trn = u[u["in_population"] & u["year"].le(2015)]
seq_new = trn["prior_in_data"] | trn["title_says_sequel"]
cmp = pd.DataFrame({
    "franchise check only": [trn["prior_in_data"].sum(), trn.loc[trn["prior_in_data"], "us_multiple"].median(), trn.loc[~trn["prior_in_data"], "us_multiple"].median()],
    "franchise check + title": [seq_new.sum(), trn.loc[seq_new, "us_multiple"].median(), trn.loc[~seq_new, "us_multiple"].median()],
}, index=["sequels found (training)", "median return: sequels", "median return: everything else"]).round(2)
display(cmp)
added = trn[trn["title_says_sequel"] & ~trn["prior_in_data"]]
print(f"Sequels added by the title rule in training years: {len(added)}; their median return {added['us_multiple'].median():.2f}x")

### 13b. Tightened rule (v2)

v1 flags titles where the number isn't an installment (*Big Hero 6*, *District 9*, *Super 8*, *Triple 9*, *The Trial of the Chicago 7*, *The Taking of Pelham 1 2 3*). v2 keeps a title number only if one of these is true:
- the title uses an explicit installment word or Roman numeral ("Part 2", "Chapter 3", "Episode II", "Men in Black II"), or
- the film has a franchise tag (*Toy Story 3*), or
- an **earlier** film exists under the title with the number removed (*Paddington 2* → *Paddington*), checked against all of TMDB, including pre-2000 films.

In [ ]:
tm = pd.read_csv("../data/raw/TMDB_movie_dataset_v11.csv", usecols=["title", "release_date", "vote_count"], low_memory=False)
tm = tm[tm["vote_count"].fillna(0) >= 20]                     # real releases, not stray entries
tm["base"] = tm["title"].str.lower().str.strip()
tm["rd"] = pd.to_datetime(tm["release_date"], errors="coerce")
earliest = tm.groupby("base")["rd"].min()


def base_title(t):
    t = re.sub(r"[:\-–(].*$", "", str(t))                          # drop subtitle
    t = re.sub(r"\b(part|chapter|episode|vol\.?|volume)\b.*$", "", t, flags=re.I)
    t = re.sub(r"\s+(\d+|II|III|IV|VI|VII|VIII|IX)\s*$", "", t.strip())
    return t.strip().lower()


explicit = u["title"].str.contains(r"\b(?:part|chapter|episode|vol\.?|volume)\s+(?:\d+|[ivx]+|two|three|four|five)\b", case=False, regex=True) \
           | u["title"].str.contains(r"\b(?:II|III|IV|VI|VII|VIII|IX)\b", regex=True)
base = u["title"].map(base_title)
earlier_base = [pd.notna(earliest.get(b, pd.NaT)) and earliest.get(b) < d for b, d in zip(base, u["release_date"])]
u["title_sequel_v2"] = u["title_says_sequel"] & (explicit | u["franchise"].ne("(none)") | pd.Series(earlier_base, index=u.index))
u["sequel_any"] = u["prior_in_data"] | u["title_sequel_v2"]

dropped = u[u["title_says_sequel"] & ~u["title_sequel_v2"]]
print(f"v1 title matches: {u['title_says_sequel'].sum()};  v2 keeps {u['title_sequel_v2'].sum()};  rejected {len(dropped)}:")
display(dropped[["title", "year"]].head(20))
added = u[u["title_sequel_v2"] & ~u["prior_in_data"]]
print(f"Sequels found only by the title (v2): {len(added)}")
display(added[["title", "year", "franchise", "title_installment"]].head(40))
trn = u[u["in_population"] & u["year"].le(2015)]
print(f"Training films: sequels by franchise check {trn['prior_in_data'].sum()} -> with title v2 {trn['sequel_any'].sum()}; "
      f"median return of the added ones {trn.loc[trn['title_sequel_v2'] & ~trn['prior_in_data'], 'us_multiple'].median():.2f}x "
      f"(sequels overall {trn.loc[trn['sequel_any'], 'us_multiple'].median():.2f}x, others {trn.loc[~trn['sequel_any'], 'us_multiple'].median():.2f}x)")

### 13c. What the title check shows (run 2026-11-26)

- **Toy Story 3 is not identified as a sequel today** (its earlier films are pre-2000). Same for *Mission: Impossible II*, *Terminator 3*, *Men in Black II*, *Jurassic Park III*, *Blade II*, *Bad Boys II*, *Scream 3* and most sequels from 2000–2005.
- The title rule (v2) adds **77 sequels** across all usable films (36 in training years: 286 → 322). Their median return is 1.41×, close to other sequels (1.45×), so they belong in the sequel group.
- The franchise check still catches sequels without numbers (*The Dark Knight*, *Avengers: Endgame*), so the two together work best: **`sequel` = earlier franchise film in data OR installment number in the title**.
- **Errors left in v2:** it still accepts *Super 8* (a different, earlier film called *Super* exists) and rejects some real foreign sequels whose base title doesn't match TMDB exactly (*Ong Bak 2*, *Ju-on: The Grudge 2*, *Torrente 3*). Small numbers; worth a hand-check list before promoting to `01_clean_dataset.ipynb`.
- Time-order: OK. A sequel's title and number are set before release.


## 14. Where Hollywood placed its bets vs what succeeded (all years, trend description)

For each period and group: the share of **films released** (quantity bets), the share of **budget dollars** (money bets), the share of **US box office** that came back, and the median return. If a group's share of box office is bigger than its share of dollars, it earned more than its weight. Sequel status here uses the franchise check **plus** the title rule from §13.

In [ ]:
ORANGE = "#eb6834"
ap = allp.copy()
ap["title_installment"] = ap["title"].map(installment_from_title)
ap["sequel"] = prior_all | ap["title_installment"].notna()
ap["budget_tier"] = pd.cut(ap["budget_23"], [0, 10e6, 40e6, 100e6, np.inf], labels=["under $10M", "$10-40M", "$40-100M", "$100M+"]).astype(str)
ap["sequel_status"] = np.where(ap["sequel"], "sequel", "not a sequel")
ap["rating"] = ap["mpaa_rating"].where(~ap["mpaa_rating_missing"].astype(bool), "NR")


def bets(col, groups=None, explode=False):
    d = ap[["period", "budget_23", "gross_23", "us_multiple", col]].copy()
    if explode:
        d = d.explode(col)
    if groups is not None:
        d = d[d[col].isin(groups)]
    tot = ap.groupby("period", observed=True).agg(n=("film_uid", "size"), b=("budget_23", "sum"), g=("gross_23", "sum"))
    t = d.groupby(["period", col], observed=True).agg(n=("us_multiple", "size"), b=("budget_23", "sum"), g=("gross_23", "sum"), med=("us_multiple", "median"))
    t = t.join(tot, rsuffix="_all")
    out = pd.DataFrame({"% of films": 100 * t["n"] / t["n_all"], "% of dollars": 100 * t["b"] / t["b_all"],
                        "% of US box office": 100 * t["g"] / t["g_all"], "median return": t["med"]}).round(1)
    out["median return"] = out["median return"].round(2)
    return out


def bets_plot(col, title, groups=None, explode=False, ncols=4):
    b = bets(col, groups, explode)
    gs = groups or sorted(b.index.get_level_values(1).unique())
    nrows = int(np.ceil(len(gs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.8 * ncols, 2.7 * nrows), sharey=True, squeeze=False)
    x = np.arange(len(PERIODS))
    for ax, gname in zip(axes.flat, gs):
        s = b.xs(gname, level=1).reindex(PERIODS)
        ax.axvspan(3.5, 4.5, color="#f2c6ad", alpha=.35, lw=0); ax.axvspan(4.5, 5.5, color="#bfe8d6", alpha=.45, lw=0)
        ax.plot(x, s["% of films"], "o-", color="#86b6ef", ms=3, lw=1.5, label="% of films")
        ax.plot(x, s["% of dollars"], "o-", color="#0d4a94", ms=3, lw=1.5, label="% of budget dollars")
        ax.plot(x, s["% of US box office"], "o--", color=ORANGE, ms=3, lw=1.5, label="% of US box office")
        ax.set_title(gname, fontsize=9); ax.set_xticks(x, ["00-03", "04-07", "08-11", "12-15", "16-19", "20-23"], fontsize=7)
        ax.grid(False); ax.grid(axis="y", color="#e6e5e0", lw=0.6)
    for ax in list(axes.flat)[len(gs):]:
        ax.axis("off")
    axes.flat[0].legend(fontsize=7, loc="upper left")
    fig.suptitle(title + "   (orange above dark blue = earned more than its share of the money)", x=0.01, ha="left", fontsize=10)
    plt.tight_layout(); plt.show()
    return b


b_seq = bets_plot("sequel_status", "Sequels vs everything else", ["sequel", "not a sequel"], ncols=2)
display(b_seq.unstack()["% of films"].round(1).T.add_prefix("% films ").join(b_seq.unstack()["% of US box office"].T.add_prefix("% box office ")))

In [ ]:
b_bud = bets_plot("budget_tier", "Budget tier (2023 dollars)", ["under $10M", "$10-40M", "$40-100M", "$100M+"], ncols=4)
b_rat = bets_plot("rating", "MPAA rating", ["G", "PG", "PG-13", "R", "NR"], ncols=5)
ap["genre"] = GEN_ALL
b_gen = bets_plot("genre", "Genre (a film counts in each of its genres)", ["Drama", "Comedy", "Action", "Adventure", "Horror", "Family", "Science Fiction", "Fantasy"], explode=True, ncols=4)

In [ ]:
# What happened in the 2010s? Checks against the data
yearly = ap.groupby("year").agg(films=("film_uid", "size"), median_return=("us_multiple", "median"),
                                pct_sequels=("sequel", "mean"), pct_horror=("genre", lambda s: np.mean(["Horror" in l for l in s])),
                                pct_comic=("source_group", lambda s: (s == "Comic").mean()), pct_under10M=("budget_23", lambda s: (s < 10e6).mean()),
                                us_box_office_2023_B=("gross_23", lambda s: s.sum() / 1e9))
yearly[["pct_sequels", "pct_horror", "pct_comic", "pct_under10M"]] = (100 * yearly[["pct_sequels", "pct_horror", "pct_comic", "pct_under10M"]]).round(1)
display(yearly.round(2).T)
mcu = ap["franchise"].str.contains("Marvel Cinematic Universe", na=False)
print(f"MCU films in the population: {mcu.sum()}; their share of all US box office in our data by period:")
display((100 * ap.loc[mcu].groupby("period", observed=True)["gross_23"].sum() / ap.groupby("period", observed=True)["gross_23"].sum()).round(1).rename("% of US box office").to_frame().T)
ex_mcu = ap[~mcu].groupby("period", observed=True)["us_multiple"].median()
print("Median return by period with and without MCU films:")
display(pd.DataFrame({"all films": market, "without MCU": ex_mcu}).round(2).T)

### 14b. Bets vs success, and what changed in the 2010s (run 2026-11-26)

**Where the bets went**
- **Sequels:** 6% of releases in 2000–03 → 18% in 2016–19 → 23% in 2020–23, while their share of US box office went **15% → 51% → 55%**. Hollywood shifted toward sequels, and sequels earned far more than their share of releases.
- **Budget:** $100M+ films are 14–20% of releases but take 47–57% of the money and earn about the same share back. The **$40–100M middle shrank** (29% of releases → 19% by 2012–19). Small films under $10M grew from 17% to 30% of releases by 2012–15 while staying about 2–5% of the money.
- Genre and rating detail is in the small multiples above.

**What happened in the 2010s?** Checked against the data:
- **The median film recovered in 2014 and 2017–19** (0.81–0.89×) after a low in 2007–13 (0.56–0.68×). The total US box office in our data stayed flat at about $13–15B a year (2023 dollars), and the number of films fell (about 250 a year through 2016 → 183 in 2019). **But the films that dropped out were almost all small and mid-budget** (under $10M: 100 → 59; $10–40M: 89 → 70), while $40M+ films stayed at about 50–55 a year and films grossing $10M+ barely changed. Those small films had captured little of the pie, so this is mainly a **composition effect**: fewer low-return small films in the mix raises the median. Part of it may be our sources recording fewer small recent films (industry-wide release counts didn't fall), not fewer being made.
- **2008 recession:** no drop in film count afterwards (249 → 251 → 245 in 2008–10), so no bigger share per film. Mid-budget films did fall (119 → 94), replaced by more small films. 2009 box office ticked up a little (about $14.9B vs $13.7B), but median returns in 2009–10 were among the lowest (0.57–0.62×).
- **Marvel (MCU):** the MCU grew to 11% of US box office in our data in 2016–19, but **removing MCU films barely changes the median** (0.76× → 0.73×). Marvel explains the big totals, not the typical film's recovery.
- **Horror:** horror rose from about 9% of releases (2015) to 15% (2019), with median returns near 2×. Cheap horror (the Blumhouse model) lifted the typical return.
- **Sequels:** their share of releases climbed steadily (8% in 2010 → 21% in 2019) and they return about 1.4×, which pulls the median up.
- Caution: industry-wide ticket sales have fallen since 2000 (Statista: about −46%), with prices rising. Our data covers only films with a recorded budget and US gross, so totals are not the whole market.


## 15. Exploring the not-yet-tested feature ideas (training years only)

The 9 ideas from `FEATURE_IDEAS.md` not covered in §10–13, built as columns on the training films (2000–2015, 3,708). Each gets a return chart (as in §10) and two scores:
- **AUC alone:** how well the feature by itself separates winners from losers (0.5 = coin flip, 1.0 = perfect), from a simple logistic regression with 5-fold cross-validation **inside the training years**. The test years stay untouched.
- **Gain over budget:** AUC of budget + the feature, minus AUC of budget alone. Shows whether the feature adds anything beyond "how big is the film."

These are screening scores for ranking ideas, not the final model.

| § | Idea | Columns built |
|---|---|---|
| 15a | Budget scale | `log_budget`, `budget_tier`, `budget_pct_in_year` |
| 15b | Genre grouping and count | `primary_genre`, `n_genres`, genre flags |
| 15c | Release season and holiday window | `release_season`, `summer_window`, `holiday_window` |
| 15d | Runtime bins and unknown flags | `runtime_bin`, `runtime_missing` |
| 15e | Multi-column combinations | `budget_vs_genre_year`, `tentpole`, `budget_per_minute` |
| 15f | Release-window competition | `big_films_in_window`, `max_competitor_budget` |
| 15g | Franchise × source material | `franchise_source` |
| 15h | Writer + director | `director_writes`, `pair_prior_films` |
| 15i | Studio and distributor grouping | `distributor_group`, `major_producer` |

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

F = pd.DataFrame(index=tr.index)          # feature columns for the training films
y = (~tr["loser"]).astype(int)            # 1 = winner
SCORES = []
cv = StratifiedKFold(5, shuffle=True, random_state=560)


def design(cols):
    X = []
    for c in cols:
        s = F[c]
        if s.dtype == object or str(s.dtype) in ("category", "string", "str"):
            X.append(pd.get_dummies(s.astype(str).fillna("(blank)"), prefix=c, drop_first=True, dtype=float))
        else:
            v = s.astype(float)
            X.append(pd.DataFrame({c: v.fillna(v.median()), c + "_na": v.isna().astype(float)}).loc[:, lambda d: d.std() > 0])
    X = pd.concat(X, axis=1)
    return (X - X.mean()) / X.std().replace(0, 1)


def cv_auc(cols):
    p = cross_val_predict(LogisticRegression(max_iter=2000, C=1.0), design(cols), y, cv=cv, method="predict_proba")[:, 1]
    return roc_auc_score(y, p)


def score(idea, cols):
    for c in cols:
        a = cv_auc([c]); g = cv_auc(["log_budget", c]) - BASE_AUC if c != "log_budget" else np.nan
        SCORES.append({"idea": idea, "feature": c, "AUC alone": round(a, 3), "gain over budget": round(g, 3) if pd.notna(g) else np.nan})
    display(pd.DataFrame(SCORES)[lambda d: d["idea"].eq(idea)].set_index("feature").drop(columns="idea"))


def num_plot(ax, col, title, q=5):
    b = pd.qcut(F[col], q, duplicates="drop")
    labels = {iv: f"{iv.left:.3g}–{iv.right:.3g}" for iv in b.cat.categories}
    mult_plot(ax, b.map(labels).astype(object).fillna("(blank)"), title, order=list(labels.values()) + ["(blank)"])

### 15a. Budget scale

In [ ]:
F["log_budget"] = np.log10(tr["fin_budget"])
F["budget_tier"] = pd.cut(tr["fin_budget"], [0, 5e6, 20e6, 60e6, np.inf], labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"]).astype(str)
F["budget_pct_in_year"] = tr.groupby("year")["fin_budget"].rank(pct=True)
BASE_AUC = cv_auc(["log_budget"])
print(f"Budget alone (log): AUC {BASE_AUC:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(14, 3.4))
mult_plot(ax[0], F["budget_tier"], "Budget tier", order=["$1-5M", "$5-20M", "$20-60M", "$60M+"])
num_plot(ax[1], "budget_pct_in_year", "Budget percentile within its release year")
plt.tight_layout(); plt.show()
score("budget scale", ["log_budget", "budget_tier", "budget_pct_in_year"])

### 15b. Genre grouping and genre count

Primary genre = the first genre TMDB lists; genres with fewer than 40 training films go to "Other".

In [ ]:
prim = GEN.map(lambda l: l[0] if l else "(blank)")
F["primary_genre"] = prim.where(prim.map(prim.value_counts()) >= 40, "Other")
F["n_genres"] = GEN.map(len)
for gname in ["Horror", "Animation", "Family", "Drama", "Comedy"]:
    F[f"is_{gname.lower()}"] = GEN.map(lambda l, g=gname: g in l).astype(int)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
mult_plot(ax[0], F["primary_genre"], "Primary genre (first listed)", sort=True)
mult_plot(ax[1], F["n_genres"].clip(upper=5).map(lambda n: f"{n}" if n < 5 else "5+"), "Number of genres listed", order=["1", "2", "3", "4", "5+"])
plt.tight_layout(); plt.show()
score("genre grouping", ["primary_genre", "n_genres", "is_horror", "is_animation", "is_family", "is_drama", "is_comedy"])

### 15c. Release season and holiday window

Seasons: Winter Jan–Feb, Spring Mar–Apr, Summer May–Aug, Fall Sep–Oct, Holiday Nov–Dec.

In [ ]:
SEASON = {1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Summer", 6: "Summer", 7: "Summer", 8: "Summer",
          9: "Fall", 10: "Fall", 11: "Holiday", 12: "Holiday"}
F["release_season"] = tr["release_month"].astype(int).map(SEASON)
F["summer_window"] = tr["release_month"].isin([5, 6, 7, 8]).astype(int)
F["holiday_window"] = tr["release_month"].isin([11, 12]).astype(int)
F["release_weekday"] = tr["release_date"].dt.day_name().where(lambda s: s.isin(["Wednesday", "Friday"]), "Other day")
fig, ax = plt.subplots(figsize=(8, 3.4))
mult_plot(ax, F["release_season"], "Release season", order=["Winter", "Spring", "Summer", "Fall", "Holiday"])
plt.tight_layout(); plt.show()
score("release season", ["release_season", "summer_window", "holiday_window", "release_weekday"])

### 15d. Runtime bins and "unknown" flags

In [ ]:
F["runtime_bin"] = pd.cut(tr["runtime_min"], [0, 90, 120, np.inf], labels=["under 90", "90-120", "over 120"], right=False).astype(object).fillna("(unknown)")
F["runtime_min"] = tr["runtime_min"]
F["runtime_missing"] = tr["runtime_min_missing"].astype(int)
fig, ax = plt.subplots(figsize=(8, 2.8))
mult_plot(ax, F["runtime_bin"], "Runtime", order=["under 90", "90-120", "over 120", "(unknown)"], min_n=10)
plt.tight_layout(); plt.show()
print(f"Runtime unknown: {F['runtime_missing'].sum()} training films (too few to judge)")
score("runtime bins", ["runtime_bin", "runtime_min"])

### 15e. Multi-column combinations

- `budget_vs_genre_year`: budget ÷ the median budget of training films with the same primary genre and release year (is this a big bet *for its kind of film*?).
- `tentpole`: budget $80M+ **and** a sequel (franchise check or title, §13) **and** a summer or holiday release.
- `budget_per_minute`: budget ÷ runtime.

In [ ]:
med_gy = tr.groupby([F["primary_genre"], tr["year"]])["fin_budget"].transform("median")
F["budget_vs_genre_year"] = tr["fin_budget"] / med_gy
F["sequel"] = u["sequel_any"].reindex(tr.index).fillna(False).astype(int)
F["tentpole"] = ((tr["fin_budget"] >= 80e6) & F["sequel"].astype(bool) & (F["summer_window"].astype(bool) | F["holiday_window"].astype(bool))).astype(int)
F["budget_per_minute"] = tr["fin_budget"] / tr["runtime_min"]
fig, ax = plt.subplots(1, 3, figsize=(18, 3.4))
num_plot(ax[0], "budget_vs_genre_year", "Budget / median budget for same genre and year")
mult_plot(ax[1], F["tentpole"].map({1: "tentpole", 0: "not a tentpole"}), "Tentpole: $80M+, sequel, summer or holiday")
num_plot(ax[2], "budget_per_minute", "Budget per minute ($)")
plt.tight_layout(); plt.show()
print(f"Tentpoles in training: {F['tentpole'].sum()}")
score("multi-column", ["budget_vs_genre_year", "tentpole", "budget_per_minute", "sequel"])

### 15f. Release-window competition

For each training film: how many **big** films (budget at or above the training 80th percentile) opened within ±21 days, and the largest competitor's budget. Competitors come from all real films with a budget and a US release date (any population status, junk excluded). Only competitors' budgets and dates are used, never their box office.

In [ ]:
pool = clean_now[clean_now["usable"] & clean_now["exclude_reason"].isna() & clean_now["release_date_src"].eq("us")][["film_uid", "release_date", "fin_budget"]].dropna()
BIG = tr["fin_budget"].quantile(.8)
pool = pool.sort_values("release_date").reset_index(drop=True)
dates = pool["release_date"].values.astype("datetime64[D]").astype(np.int64)
big_counts, max_comp = [], []
for uid, d in zip(tr["film_uid"], tr["release_date"].values.astype("datetime64[D]").astype(np.int64)):
    lo, hi = np.searchsorted(dates, d - 21), np.searchsorted(dates, d + 21, side="right")
    w = pool.iloc[lo:hi]
    w = w[w["film_uid"] != uid]
    big_counts.append(int((w["fin_budget"] >= BIG).sum())); max_comp.append(w["fin_budget"].max() if len(w) else np.nan)
F["big_films_in_window"] = big_counts
F["max_competitor_budget"] = np.log10(pd.Series(max_comp, index=tr.index))
print(f"'Big' = budget >= ${BIG / 1e6:.0f}M (training 80th percentile)")
fig, ax = plt.subplots(1, 2, figsize=(14, 3.4))
mult_plot(ax[0], F["big_films_in_window"].clip(upper=6).map(lambda n: str(n) if n < 6 else "6+"), "Big films opening within ±3 weeks",
          order=["0", "1", "2", "3", "4", "5", "6+"])
num_plot(ax[1], "max_competitor_budget", "Largest competitor's budget (log10 $)")
plt.tight_layout(); plt.show()
score("competition", ["big_films_in_window", "max_competitor_budget"])
print("Within budget tiers (does competition matter for big films specifically?):")
display(pd.crosstab(F["budget_tier"], F["big_films_in_window"].clip(upper=4), values=tr["us_multiple"], aggfunc="median").round(2))

### 15g. Franchise × source material

Uses the sequel flag known before release (§13) crossed with `source_group`.

In [ ]:
F["franchise_source"] = np.where(F["sequel"].astype(bool), "sequel: " + tr["source_group"].fillna("(blank)"), "not a sequel")
fig, ax = plt.subplots(figsize=(9, 4))
mult_plot(ax, pd.Series(F["franchise_source"], index=tr.index), "Sequels by source material (vs all non-sequels)", sort=True, min_n=15)
plt.tight_layout(); plt.show()
score("franchise x source", ["franchise_source"])

### 15h. Writer + director

`director_writes` = the director is also a credited writer. `pair_prior_films` = earlier films the same director and writer made together (earlier release dates only).

In [ ]:
def names_set(s):
    return {norm_n for n in str(s).split(",") for norm_n in [n.strip().lower()] if norm_n and str(s) != "nan"}

F["director_writes"] = [int(bool(names_set(a) & names_set(b))) for a, b in zip(tr["director"], tr["writer"])]
pairs = clean_now[clean_now["usable"] & clean_now["exclude_reason"].isna()][["film_uid", "release_date", "director", "writer"]].dropna()
pairs = pairs.assign(release_date=pd.to_datetime(pairs["release_date"], format="mixed", errors="coerce"))
pairs["key"] = [tuple(sorted(names_set(a))) + ("|",) + tuple(sorted(names_set(b) - names_set(a))) for a, b in zip(pairs["director"], pairs["writer"])]
pairs = pairs.sort_values("release_date")
pairs["prior"] = pairs.groupby("key").cumcount()
F["pair_prior_films"] = tr["film_uid"].map(pairs.set_index("film_uid")["prior"]).fillna(0).astype(int)
fig, ax = plt.subplots(1, 2, figsize=(14, 3))
mult_plot(ax[0], F["director_writes"].map({1: "director also wrote it", 0: "separate writer"}), "Director also a writer")
mult_plot(ax[1], F["pair_prior_films"].clip(upper=2).map({0: "first time together", 1: "1 earlier film", 2: "2+ earlier films"}),
          "Same director + writer team before", order=["first time together", "1 earlier film", "2+ earlier films"], min_n=20)
plt.tight_layout(); plt.show()
score("writer + director", ["director_writes", "pair_prior_films"])

### 15i. Studio and distributor grouping

Distributor (first listed, variants merged as in §11a) grouped into: **Major** (Disney, Warner Bros., Universal, Sony, Paramount, 20th Century Fox), **Mini-major** (Lionsgate, Weinstein, Miramax, New Line, DreamWorks, Summit, Relativity, MGM, STX, Open Road), **Specialty / indie** (Sony Pictures Classics, Fox Searchlight, Focus, IFC, Magnolia, Roadside, A24, Samuel Goldwyn, ThinkFilm, Warner Independent…), Other, Blank. `major_producer` = a major studio name appears among the production companies.

In [ ]:
MAJORS = {"Disney", "Warner Bros.", "Universal", "Sony", "Paramount", "20th Century Fox"}
MINI = {"Lionsgate", "Weinstein Co.", "Miramax Films", "New Line Cinema", "DreamWorks SKG", "Summit Entertainment", "Relativity Media",
        "Metro-Goldwyn-Mayer", "STX Entertainment", "Open Road", "FilmDistrict", "CBS Films", "Screen Gems", "Dimension Films"}
SPECIALTY = {"Sony Pictures Classics", "Fox Searchlight", "Focus Features", "IFC Films", "Magnolia Pictures", "Roadside Attractions", "A24",
             "Samuel Goldwyn Company", "ThinkFilm", "Warner Independent", "Paramount Vantage", "First Look", "Anchor Bay", "Freestyle Releasing",
             "Overture Films", "Artisan Entertainment", "Lionsgate Premiere"}
F["distributor_group"] = np.select([studio.isin(MAJORS), studio.isin(MINI), studio.isin(SPECIALTY), studio.isna()],
                                   ["Major", "Mini-major", "Specialty / indie", "Blank"], "Other")
MAJOR_PAT = r"warner|universal|walt disney|disney|pixar|columbia|sony pictures|paramount|20th century|twentieth century|fox"
F["major_producer"] = tr["production_companies"].fillna("").str.lower().str.contains(MAJOR_PAT).astype(int)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.2))
mult_plot(ax[0], pd.Series(F["distributor_group"], index=tr.index), "Distributor group", order=["Major", "Mini-major", "Specialty / indie", "Other", "Blank"])
mult_plot(ax[1], F["major_producer"].map({1: "a major among producers", 0: "no major producer"}), "Production company")
plt.tight_layout(); plt.show()
score("studio grouping", ["distributor_group", "major_producer"])

### 15k. Niche crowding (Tanner's idea, 2026-11-26)

G-rated family films have the best returns but are few. Maybe they win partly *because* they're few: each gets its audience to itself. Feature: how many films **like this one** open within ±6 weeks, from all real films with a US release date (only their dates, ratings and genres, never their box office):
- `same_rating_in_window`: films with the same MPAA rating
- `same_genre_in_window`: films with the same primary genre
- `niche_share`: this film's primary genre's share of all releases in the window (how crowded the niche is relative to the market)

In [ ]:
pool2 = clean_now[clean_now["usable"] & clean_now["exclude_reason"].isna() & clean_now["release_date_src"].eq("us")].copy()
pool2["release_date"] = pd.to_datetime(pool2["release_date"], format="mixed", errors="coerce")
pool2["pg"] = pool2["genres_all"].fillna("").str.split(",").str[0].str.strip()
pool2 = pool2.dropna(subset=["release_date"]).sort_values("release_date").reset_index(drop=True)
d2 = pool2["release_date"].values.astype("datetime64[D]").astype(np.int64)
same_r, same_g, share_g = [], [], []
for uid, d, r, g in zip(tr["film_uid"], tr["release_date"].values.astype("datetime64[D]").astype(np.int64), tr["mpaa_rating"], prim):
    lo, hi = np.searchsorted(d2, d - 42), np.searchsorted(d2, d + 42, side="right")
    w = pool2.iloc[lo:hi]
    w = w[w["film_uid"] != uid]
    same_r.append(int((w["mpaa_rating"] == r).sum())); same_g.append(int((w["pg"] == g).sum()))
    share_g.append((w["pg"] == g).mean() if len(w) else np.nan)
F["same_rating_in_window"], F["same_genre_in_window"], F["niche_share"] = same_r, same_g, share_g
fig, ax = plt.subplots(1, 2, figsize=(14, 3.4))
num_plot(ax[0], "same_rating_in_window", "Films with the same MPAA rating within ±6 weeks", q=4)
num_plot(ax[1], "niche_share", "Primary genre's share of releases within ±6 weeks", q=4)
plt.tight_layout(); plt.show()
print("Inside each rating: median return by how many same-rated films open within ±6 weeks")
crowd = pd.cut(F["same_rating_in_window"], [-1, 2, 6, 15, 1000], labels=["0-2", "3-6", "7-15", "16+"])
t = pd.DataFrame({"r": tr["mpaa_rating"], "c": crowd, "m": tr["us_multiple"]})
display(t.groupby(["r", "c"], observed=True)["m"].agg(["size", "median"]).round(2).unstack())
score("niche crowding", ["same_rating_in_window", "same_genre_in_window", "niche_share"])

### 15j. Ranking all features

In [ ]:
rank_all = pd.DataFrame(SCORES).drop_duplicates("feature", keep="last").sort_values("gain over budget", ascending=False).set_index("feature")
display(rank_all)
print(f"Reference: budget alone AUC {BASE_AUC:.3f}. Gains under ~0.005 are within noise for 5-fold CV on 3,708 films.")

### 15l. What the feature screen shows (run 2026-11-26)

Scores: 5-fold cross-validated AUC inside 2000–15. Budget alone scores only 0.536: its link to winning is curved (small films lose, mid and big films are similar), which a straight-line screen can't capture. The final model needs to handle curves (tiers or a tree model).

**Worth keeping (clear gain over budget)**
1. **Distributor group (+0.169, AUC 0.659), by far the strongest.** Major / mini-major / specialty / blank. ⚠️ Part of this may be the "blank" leak from §10e (blank distributor = undocumented film) and the specialty labels' limited US releases. Re-test without the Blank group before trusting it.
2. **Primary genre (+0.077).** One category beats separate genre flags; drama, comedy, horror and family flags each add a little on their own.
3. **A major studio among the producers (+0.040).**
4. **Release weekday (+0.020), sequel × source (+0.020), sequel (+0.015), runtime bin (+0.014).** Small but real.

**Not worth it (gain at or below noise, about 0.005)**
- Budget tier and budget percentile within the year (log budget already covers it), budget per minute, budget vs genre-year median, tentpole flag.
- Season and summer/holiday flags (weekday captures the useful part).
- Release-window competition: no overall effect. Inside budget tiers there's no consistent pattern either.
- Writer-director and repeat pairs, genre count.
- **Niche crowding (Tanner's idea): weak overall (+0.004 to +0.007).** It can't explain the G-rated advantage because **G films almost never open near each other**: 71 of 79 G films had 0–2 other G films within ±6 weeks. Studios already space them out, so the data never shows a crowded G window. For PG films, crowding makes little difference (1.11× with 0–2 others vs 1.00× with 7–15).

**About "fewer G films" (Tanner):** the count matters most as **certainty**, not as a film property. A group rate from 79 films is shakier than one from 1,600 R films. Two standard ways to use it:
- **Smoothed group rates** (target encoding with shrinkage): each group's historical return is pulled toward the overall average in proportion to how few films it has. A rare group needs consistent evidence to get an extreme score.
- **Merge thin groups** (e.g. G with PG as "family-rated") so each group has enough films.
Both are modeling choices for the next phase, not new columns.


### 15m. Competition from overlapping ratings (Tanner's idea, 2026-11-26)

**Theory:** audiences move between neighboring ratings. A family that would see a G film could just as easily pick a PG film; a PG-13 audience overlaps with R. So a film's real competitors are films in its **own and neighboring ratings** opening within ±6 weeks:

| Film's rating | Competes with |
|---|---|
| G | G, PG |
| PG | G, PG, PG-13 |
| PG-13 | PG, PG-13, R |
| R | PG-13, R, NC-17 |
| NC-17 | R, NC-17 |
| NR | NR only (mostly foreign and limited releases) |

Counted from **all US releases** (real films with a US release date; only their dates and ratings are used, never their box office). Because PG-13 and R films have many more neighbors than G films, each rating is shown on its own scale (its own quartiles of competitor counts).

In [ ]:
ADJ = {"G": {"G", "PG"}, "PG": {"G", "PG", "PG-13"}, "PG-13": {"PG", "PG-13", "R"}, "R": {"PG-13", "R", "NC-17"},
       "NC-17": {"R", "NC-17"}, "NR": {"NR"}}
pr = pool2["mpaa_rating"].fillna("NR").values
adj_count = []
for uid, d, rt in zip(tr["film_uid"], tr["release_date"].values.astype("datetime64[D]").astype(np.int64), tr["mpaa_rating"].fillna("NR")):
    lo, hi = np.searchsorted(d2, d - 42), np.searchsorted(d2, d + 42, side="right")
    w_uid, w_r = pool2["film_uid"].values[lo:hi], pr[lo:hi]
    adj_count.append(int(sum((u_ != uid) and (r_ in ADJ[rt]) for u_, r_ in zip(w_uid, w_r))))
F["overlap_rating_competitors"] = adj_count
print("Competitors in own + neighboring ratings within ±6 weeks, by the film's rating:")
display(F.groupby(tr["mpaa_rating"].fillna("NR"))["overlap_rating_competitors"].describe(percentiles=[.25, .5, .75])[["count", "min", "25%", "50%", "75%", "max"]].round(0))

fig, axes = plt.subplots(1, 4, figsize=(19, 3.4))
for ax, rt in zip(axes, ["G", "PG", "PG-13", "R"]):
    m = tr["mpaa_rating"].eq(rt)
    s = F.loc[m, "overlap_rating_competitors"]
    qb = pd.qcut(s, 4, duplicates="drop")
    lab = qb.map({iv: f"{int(np.floor(iv.left)) + 1}-{int(iv.right)}" for iv in qb.cat.categories}).astype(object)
    mult_plot(ax, lab, f"{rt}: competitors in G/PG/PG-13/R overlap", order=[f"{int(np.floor(iv.left)) + 1}-{int(iv.right)}" for iv in qb.cat.categories], min_n=10)
plt.tight_layout(); plt.show()

rows = []
for rt in ["G", "PG", "PG-13", "R"]:
    m = tr["mpaa_rating"].eq(rt)
    s, ret = F.loc[m, "overlap_rating_competitors"], tr.loc[m, "us_multiple"]
    lo_q, hi_q = s.quantile(.25), s.quantile(.75)
    rows.append({"rating": rt, "films": int(m.sum()), "fewest competitors (bottom quarter): median return": round(ret[s <= lo_q].median(), 2),
                 "most competitors (top quarter): median return": round(ret[s >= hi_q].median(), 2),
                 "Spearman (competitors vs return)": round(s.corr(ret, method="spearman"), 2)})
display(pd.DataFrame(rows).set_index("rating"))

# Compare with the exact-same-rating count (§15k); also relative to the rating's typical crowding
F["overlap_competitors_vs_typical"] = F["overlap_rating_competitors"] / F.groupby(tr["mpaa_rating"].fillna("NR"))["overlap_rating_competitors"].transform("median").replace(0, 1)
score("rating overlap", ["overlap_rating_competitors", "overlap_competitors_vs_typical"])
print("Same idea inside budget tiers (median return by quartile of the rating-relative count):")
display(pd.crosstab(F["budget_tier"], pd.qcut(F["overlap_competitors_vs_typical"], 4, labels=["least crowded", "2", "3", "most crowded"]),
                    values=tr["us_multiple"], aggfunc="median").round(2))

**What 15m shows (run 2026-11-26): Tanner's overlap theory holds.** Inside every rating, films facing more competitors in their own and neighboring ratings return less:

| Rating | Fewest competitors (bottom quarter) | Most competitors (top quarter) |
|---|---|---|
| G | 1.83× | 0.92× |
| PG | 1.22× | 0.79× |
| PG-13 | 1.10× | 0.87× |
| R | 0.77× | 0.41× |

- It also holds **inside budget tiers**: the most crowded quarter is lowest in each (e.g. $20–60M films 1.03× → 0.64×), so it isn't just "crowded windows hold smaller films."
- The raw count says little on its own (gain −0.013) because PG-13 and R films always have far more neighbors than G films. **Measured relative to what's normal for the film's rating** (`overlap_competitors_vs_typical`), it adds **+0.012** over budget, better than counting only the exact same rating (+0.004, §15k). That's the version to keep.
- Caveat: crowded windows may overlap with weak months (Sep–Oct is crowded with R dramas). Season on its own showed no gain (§15c), but the final model should include both and see whether crowding survives.
- Time-order: OK under A6. Competitors' release dates and ratings are set before release; some films move dates late.

### 15n. Repeat director–writer teams: track record or familiarity? (Tanner's question, 2026-11-26)

When a director and writer have worked together before, is a new film's success explained by
- **track record**: they already made hits together (or the director did), or
- **familiarity**: they know how to work together, whatever their past results?

**How it's tested** (training films, earlier release dates only):
- A **team** = the director plus a credited writer who is a different person. Films a director wrote alone are left out (they always "work with themselves").
- **Familiarity** = the team made at least one earlier film together.
- **Track record** = share of the director's earlier films that were winners (US box office ≥ 1.25× budget). Only earlier films with a known US box office count.
- If **familiarity** matters, repeat teams should beat new teams *among directors with the same track record*. If **track record** matters, directors with a good record should beat those with a poor one *whether or not the team is new*.

In [ ]:
H = clean_now[clean_now["usable"] & clean_now["exclude_reason"].isna()][["film_uid", "release_date", "director", "writer", "in_population", "us_multiple"]].copy()
H["release_date"] = pd.to_datetime(H["release_date"], format="mixed", errors="coerce")
H = H.dropna(subset=["release_date", "director"]).sort_values("release_date")
H["win"] = np.where(H["in_population"], (H["us_multiple"] >= 1.25).astype(float), np.nan)   # outcome known only for population films
H["dirs"] = H["director"].map(names_set)
H["wris"] = [names_set(w) - d for w, d in zip(H["writer"], H["dirs"])]                   # writers who aren't the director

# For each training film: director's earlier films and hit rate; team's earlier films together and hit rate
dir_hist, team_hist = {}, {}
rows = []
for uid, dirs, wris, win in zip(H["film_uid"], H["dirs"], H["wris"], H["win"]):
    d_prev = [x for d in dirs for x in dir_hist.get(d, [])]
    t_prev = [x for d in dirs for w in wris for x in team_hist.get((d, w), [])]
    rows.append({"film_uid": uid, "dir_prior_n": len(d_prev), "dir_prior_scored": int(np.sum(~np.isnan(d_prev))) if d_prev else 0,
                 "dir_prior_hit_rate": np.nanmean(d_prev) if d_prev and np.any(~np.isnan(d_prev)) else np.nan,
                 "team_prior_n": len(set(t_prev)) and len(t_prev), "team_prior_hit_rate": np.nanmean(t_prev) if t_prev and np.any(~np.isnan(t_prev)) else np.nan,
                 "has_cowriter": len(wris) > 0})
    for d in dirs:
        dir_hist.setdefault(d, []).append(win)
        for w in wris:
            team_hist.setdefault((d, w), []).append(win)
hist_feats = pd.DataFrame(rows).drop_duplicates("film_uid").set_index("film_uid")
T = tr.join(hist_feats, on="film_uid")
T = T[T["has_cowriter"].fillna(False).astype(bool)]
T["repeat_team"] = T["team_prior_n"].fillna(0) > 0
T["winner"] = ~T["loser"].astype(bool)
print(f"Training films with a director and a separate writer: {len(T):,}; repeat teams: {T['repeat_team'].sum()}; "
      f"directors with at least one earlier scored film: {T['dir_prior_hit_rate'].notna().sum():,}")

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n; mid = (p + z**2 / (2 * n)) / (1 + z**2 / n); half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return f"{p:.0%} ({mid - half:.0%}–{mid + half:.0%})"

S = T[T["dir_prior_hit_rate"].notna()].copy()
S["track_record"] = np.where(S["dir_prior_hit_rate"] >= 0.5, "director's earlier films mostly hits (≥ 50%)", "director's earlier films mostly flops (< 50%)")
S["team"] = np.where(S["repeat_team"], "repeat team", "new team")
print("Winner rate (95% range) and median return, by the director's track record and whether the team is new:")
tab = S.groupby(["track_record", "team"]).agg(films=("film_uid", "size"), winners=("winner", "sum"), median_return=("us_multiple", "median"))
tab["winner rate"] = [wilson(k, n) for k, n in zip(tab["winners"], tab["films"])]
display(tab[["films", "winner rate", "median_return"]].round(2))

R = T[T["repeat_team"] & T["team_prior_hit_rate"].notna()].copy()
R["team_record"] = pd.cut(R["team_prior_hit_rate"], [-0.01, 0.0, 0.99, 1.0], labels=["all earlier team films flopped", "mixed", "all earlier team films were hits"])
print("Repeat teams only: does what the team did before predict this film?")
tab2 = R.groupby("team_record", observed=True).agg(films=("film_uid", "size"), winners=("winner", "sum"), median_return=("us_multiple", "median"))
tab2["winner rate"] = [wilson(k, n) for k, n in zip(tab2["winners"], tab2["films"])]
display(tab2[["films", "winner rate", "median_return"]].round(2))

# Same-budget check: compare inside budget tiers
S["budget_tier"] = pd.cut(S["fin_budget"], [0, 20e6, 60e6, np.inf], labels=["under $20M", "$20-60M", "$60M+"])
print("Winner rate by track record x team, inside budget tiers (films in brackets):")
bt_tab = S.groupby(["budget_tier", "track_record", "team"], observed=True)["winner"].agg(["size", "mean"])
display(bt_tab.assign(cell=lambda d: (100 * d["mean"]).round(0).astype(int).astype(str) + "% (" + d["size"].astype(str) + ")")["cell"].unstack())

In [ ]:
# Which explains more, holding the other (and budget) fixed? Simple logistic screens, 5-fold CV inside training years
from sklearn.linear_model import LogisticRegression
Z = S.assign(log_budget=np.log10(S["fin_budget"]), record=S["dir_prior_hit_rate"], repeat=S["repeat_team"].astype(float))
yz = Z["winner"].astype(int)
def auc_of(cols):
    X = Z[cols]; X = (X - X.mean()) / X.std()
    p_ = cross_val_predict(LogisticRegression(max_iter=1000), X, yz, cv=StratifiedKFold(5, shuffle=True, random_state=560), method="predict_proba")[:, 1]
    return roc_auc_score(yz, p_)
res = {tuple(c): round(auc_of(c), 3) for c in [["log_budget"], ["log_budget", "repeat"], ["log_budget", "record"], ["log_budget", "record", "repeat"]]}
display(pd.Series({" + ".join(k): v for k, v in res.items()}, name="CV AUC (films with a director history)").to_frame())
m = LogisticRegression(max_iter=1000).fit((Z[["log_budget", "record", "repeat"]] - Z[["log_budget", "record", "repeat"]].mean()) / Z[["log_budget", "record", "repeat"]].std(), yz)
print("Standardized coefficients (bigger = stronger push toward winning):", dict(zip(["log_budget", "track record", "repeat team"], m.coef_[0].round(2))))

**What 15n shows (run 2026-11-26): it's the track record, not the familiarity.**

| Director's earlier films | New team | Repeat team |
|---|---|---|
| Mostly flops | 31% winners (556 films) | 27% (98) |
| Mostly hits | 44% (603) | 48% (155) |

- **Track record is a big step:** directors whose earlier films were mostly hits win 44–48% of the time vs 27–31%, whether or not the team is new. The intervals don't overlap.
- **Familiarity adds nothing on its own:** inside each track-record row, repeat and new teams overlap (27% vs 31%; 48% vs 44%). A repeat team after flops does no better, and on small budgets worse (16% vs 29%).
- **Repeat teams' own record says the same thing:** teams whose earlier films together were all hits win 50%; teams whose earlier films all flopped win 29%.
- **Model screen** (films whose director has a history, 5-fold CV): budget alone 0.469; + repeat team 0.474; + track record **0.586**; + both 0.585. Coefficients: track record 0.34, repeat team 0.01.
- **Why repeat teams *look* good in raw numbers:** teams that had hits get hired together again; teams that flopped mostly don't. So "worked together before" is a proxy for "succeeded together before." Use the **director's (and writer's) earlier hit rate** as the feature, not the pairing.
- Time-order: OK. Earlier films' results are known before this film's release. Directors with no earlier film in our data (most first films, plus careers before 2000) have no record: keep that as its own "no history" group.

## 16. Tanner's follow-ups on the winners (2026-11-26)

- **16a** Is the sequel premium real, or an artifact of sequels growing as a share of releases over time?
- **16b** Newer studios: A24 (founded 2012), Neon (2017), Blumhouse (producer). Are they in the data, and how do they do?
- **16c** Originals vs adaptations: do originals swing wider, so the originals that land, land bigger?

### 16a. Sequel premium: real, or a time-mix artifact?

If the premium came from timing (more sequels in later years), it would shrink or vanish when sequels are compared only with **non-sequels released in the same years** and with **the same budget**. Training years, sequel = franchise check or title rule (§13).

In [ ]:
SQ = tr.assign(sequel=F["sequel"].astype(bool), period4=pd.cut(tr["year"], [1999, 2003, 2007, 2011, 2015], labels=["2000-03", "2004-07", "2008-11", "2012-15"]),
              btier3=pd.cut(tr["fin_budget"], [0, 20e6, 60e6, np.inf], labels=["under $20M", "$20-60M", "$60M+"]))
def prem(d):
    s, o = d.loc[d["sequel"], "us_multiple"], d.loc[~d["sequel"], "us_multiple"]
    return pd.Series({"sequels": len(s), "sequel median": s.median(), "others median": o.median(), "ratio": s.median() / o.median(),
                      "sequel winners": (s >= 1.25).mean(), "others winners": (o >= 1.25).mean()})
print("Same period:")
display(SQ.groupby("period4", observed=True).apply(prem).round(2))
print("Same period AND same budget tier (sequels / ratio of medians):")
cell = SQ.groupby(["btier3", "period4"], observed=True).apply(prem)
display(cell.assign(v=lambda d: d["ratio"].round(2).astype(str) + "x  (" + d["sequels"].astype(int).astype(str) + ")")["v"].unstack())
# One number: premium after accounting for period and budget (logistic on winning)
from sklearn.linear_model import LogisticRegression
Xs = pd.get_dummies(SQ[["period4", "btier3"]].astype(str), drop_first=True, dtype=float).assign(sequel=SQ["sequel"].astype(float))
lr = LogisticRegression(max_iter=1000, C=1e6).fit(Xs, (~SQ["loser"].astype(bool)).astype(int))
print(f"Odds of being a winner for a sequel vs a non-sequel of the same period and budget tier: {np.exp(lr.coef_[0][-1]):.2f}x")

### 16b. Newer studios: A24, Neon, Blumhouse (all years, for counts; returns shown by split)

In [ ]:
allp2 = clean_now[clean_now["in_population"]].copy()
dist_txt = allp2["distributor"].fillna("").str.lower(); prod_txt = allp2["production_companies"].fillna("").str.lower()
STUDIOS = {"A24 (distributor)": dist_txt.str.contains(r"\ba24\b"), "Neon (distributor)": dist_txt.str.contains(r"\bneon\b"),
           "Blumhouse (producer)": prod_txt.str.contains("blumhouse")}
split = pd.cut(allp2["year"], [1999, 2015, 2019, 2023], labels=["train 2000-15", "test 2016-19", "stress 2020-23"])
rows = []
for name, m in STUDIOS.items():
    for sp in split.cat.categories:
        d = allp2[m & split.eq(sp)]
        rows.append({"studio": name, "split": sp, "films": len(d), "median return": round(d["us_multiple"].median(), 2) if len(d) else np.nan,
                     "winners": f"{(~d['loser'].astype(bool)).mean():.0%}" if len(d) else "", "median budget ($M)": round(d["fin_budget"].median() / 1e6, 1) if len(d) else np.nan})
display(pd.DataFrame(rows).set_index(["studio", "split"]))
print("Training-year films only (the rest stay unseen):")
for name, m in STUDIOS.items():
    d = allp2[m & split.eq("train 2000-15")]
    if len(d):
        print(f"\n{name}:"); display(d.sort_values("us_multiple", ascending=False)[["title", "year", "genres_all", "fin_budget", "domestic_gross", "us_multiple"]].head(10))

### 16c. Do originals swing wider than adaptations?

Tanner's hypothesis: adaptations do better on average, but originals spread wider, and the originals that land, land bigger. Compared by source group on training films: the middle half, the top 10% and 1% of returns, the share of big hits (4×+ and 10×+), and the dollars the top 10% earned.

In [ ]:
SG = tr.assign(sg=tr["source_group"].fillna("(blank)"), adapted=np.where(tr["source_group"].eq("Original"), "Original", "Adaptation (any)"))
def spread(d):
    m = d["us_multiple"]; prof = d["domestic_gross"] - d["fin_budget"]
    return pd.Series({"films": len(d), "median": m.median(), "25th pct": m.quantile(.25), "75th pct": m.quantile(.75),
                      "90th pct": m.quantile(.9), "99th pct": m.quantile(.99), "% at 4x+": 100 * (m >= 4).mean(), "% at 10x+": 100 * (m >= 10).mean(),
                      "% under 0.25x": 100 * (m < 0.25).mean(), "spread (90th / 10th pct)": m.quantile(.9) / m.quantile(.1),
                      "median profit of top 10% ($M)": prof[m >= m.quantile(.9)].median() / 1e6})
tab = SG[SG["sg"].ne("(blank)")].groupby("sg").apply(spread)
tab.loc["— all adaptations —"] = spread(SG[SG["adapted"].eq("Adaptation (any)") & SG["sg"].ne("(blank)")])
display(tab.round(2).sort_values("90th pct", ascending=False))
print("Same check inside budget tiers (originals vs all adaptations):")
SG["btier3"] = pd.cut(SG["fin_budget"], [0, 20e6, 60e6, np.inf], labels=["under $20M", "$20-60M", "$60M+"])
display(SG[SG["sg"].ne("(blank)")].groupby(["btier3", "adapted"], observed=True).apply(spread)[["films", "median", "90th pct", "% at 4x+", "% under 0.25x"]].round(2))

fig, ax = plt.subplots(figsize=(10, 4.5))
bins = np.linspace(-3, 1.6, 47)
for lab, col in [("Original", ORANGE), ("Adaptation (any)", BLUE)]:
    v = SG.loc[SG["adapted"].eq(lab) & SG["sg"].ne("(blank)"), "us_multiple"].clip(lower=10**-2.99)
    ax.hist(np.log10(v), bins=bins, histtype="step", lw=2, color=col, density=True, label=f"{lab} (n={len(v):,})")
ax.axvline(np.log10(1.25), color="#9a9890", ls="--", lw=1)
ax.set_xticks(np.log10([0.01, 0.1, 0.25, 0.5, 1.25, 2, 4, 10, 30]), ["0.01x", "0.1x", "0.25x", "0.5x", "1.25x", "2x", "4x", "10x", "30x"])
ax.set(title="Return distribution: originals vs adaptations (training films)", xlabel="US box office / budget (log)", ylabel="share of films (density)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print("Biggest original hits (training):")
display(SG[SG["adapted"].eq("Original")].nlargest(12, "us_multiple")[["title", "year", "genres_all", "fin_budget", "domestic_gross", "us_multiple"]])

### 16d. What the follow-ups show (run 2026-11-26)

**16a. The sequel premium is real, not a time-mix artifact.** Compared only with non-sequels from the **same period**, sequels' median return is 2.0–2.6× higher in every period (winners 52–74% vs 28–37%). Inside the **same period and budget tier** it still holds in every cell (1.3× to 12×, largest for cheap horror sequels). After accounting for period and budget, a sequel's odds of being a winner are **3.25×** a non-sequel's. Sequels did get more common, and their own median drifted down (1.72× → 1.29×), but non-sequels fell just as much, so the gap stayed.

**16b. Newer studios**
- **A24** (distributor): 22 training films (2013–15), median 0.50×, 27% winners, typical budget $5M. Hits: *The Witch* 7.2×, *Obvious Child*, *Spring Breakers*. It performs better in later years, which the model won't see during training.
- **Neon:** no training films (founded 2017).
- **Blumhouse** (producer): the standout. 28 training films, median **7.16×**, **82% winners** (*Insidious* 36×, *Paranormal Activity 2* 28×, *The Purge* 21×). It held up in later years too. This is the cheap-horror model behind the horror rise in §12f. A single-company flag is too narrow; a **production company track record** (share of earlier films that were hits, as for directors in §15n) would capture Blumhouse and others like it.

**16c. Do originals swing wider? Yes, but mostly downward.**
- Originals have the widest spread of any group (90th ÷ 10th percentile: 247 vs 64 for all adaptations), mostly from **more flops**: 31% return under 0.25× vs 24% of adaptations.
- At the very top, originals are a little more extreme: 99th percentile 15.0× vs 10.0×, and 1.5% reach 10×+ vs 1.1%. Those are nearly all **micro-budget horror and faith films** (*Saw*, *Insidious*, *God's Not Dead*, *War Room*).
- But in **dollars**, adaptations' hits are bigger: the top 10% of adaptations earned a median $56M profit vs $43M for originals. Comic films never reach 4× but their top 10% earn $169M.
- Inside budget tiers, originals don't land bigger: their 90th percentile matches adaptations (under $20M: 3.88× vs 3.80×), and above $60M adaptations reach 4×+ more often (1.9% vs 0.8%).
- **So:** originals are higher-variance, mainly because more of them flop. Their biggest *multiples* come from tiny budgets; adaptations make the bigger *dollar* hits.

**⚠️ Data bug found:** *Frozen* (2010, $5M indie horror) shows $400M US gross, and *The Little Mermaid* (2018, indie) shows $298M. Both grosses come from Box Office Mojo, which appears to add up grosses by **title across different films** (Disney's *Frozen* 2013 and *The Little Mermaid* 2023). They are the only two such cases among the 34 films at 15×+, but the Box Office Mojo join in `00_build_dataset.ipynb` should match on title **and release year** before summing. *Frozen* (2010) was the top "original" above; the conclusions don't change without it.

## 17. Summary and next steps (2026-11-26)

**Feature winners** (training years, return = US box office ÷ budget; gains are cross-validated AUC over budget alone)

| Feature | Evidence | § |
|---|---|---|
| Distributor group | +0.169; majors ≈ 1.2× vs specialty 0.02–0.3× (re-test without "blank") | 15i |
| Director track record | 0.469 → 0.586; mostly-hits directors win 44–48% vs 27–31% | 15n |
| MPAA rating | G 1.31× → PG 1.08× → PG-13 0.96× → R 0.56× → NR 0.05× | 10b |
| US vs non-US | 1.02× vs 0.11× | 10c |
| Primary genre | +0.077; Family / Horror / Comedy ≈ 1× vs Drama / War / Western 0.3–0.5× | 15b |
| Sequel (franchise + title) | 3.25× odds of winning after period and budget | 13, 16a |
| Major producer | +0.040 | 15i |
| Source group | adaptations ≈ 1.0× vs originals 0.67× | 10b, 16c |
| Rating-overlap competition | +0.012; lower return in every rating | 15m |
| Release weekday, runtime bin | +0.020, +0.014 | 15c, 15d |

**Rejected:** budget tiers / percentiles, season flags, release-window competition, tentpole flag, budget per minute, writer–director familiarity, same-rating crowding, genre count.

**Warnings:** `_missing` flags likely leak the outcome; budget's effect is curved; the winners overlap (several mean "wide US studio release"), so their gains don't add up.

**Data bugs found:** Box Office Mojo gross summed across same-title films (*Frozen* 2010, *The Little Mermaid* 2018); a few big films missed by the title matcher (*F9*, *The Butler*).

**Next steps**
1. Fix the Box Office Mojo title+year join in `00_build_dataset.ipynb`; rerun the pipeline.
2. Promote the winning features into `01_clean_dataset.ipynb`.
3. Re-test distributor without "blank"; model with and without `_missing` flags.
4. Baseline rule and first model (smoothed rates for thin groups, curved budget).
5. Promote the agreed findings into `A10-master.ipynb` §2 via PR; settle A7 vs A8 with Matt.

## 18. Baseline and first model (cross-validation inside the training years)

**Framing: winners** (Tanner, 2026-11-26). A **winner** = US box office ≥ 1.25× budget (the opposite of the `loser` label column). The model ranks films by their chance of being a winner, and the screen picks the **top 20%** (proposed amendment A9).

**The 2016–19 test set stays locked.** Everything here is 5-fold cross-validation on the 3,708 training films (2000–2015).

**Measures**
- **Winners caught @20%** = share of all winners that land in the top 20% picks. With 33% winners, random picks catch 20%, and the best possible is 60%.
- **Hit rate @20%** = share of the picks that are winners (random: 33%).
- **AUC** = ranking quality over all films (0.5 = random).

**Baseline (b), the bar to beat:** pick the genre × budget-tier groups with the highest historical winner rate, learned on the training folds only.

**First model:** gradient-boosted trees (handles curves like budget, and blanks natively) on the engineered features from `01_clean_dataset.ipynb` §12b plus the main inputs. Three versions:
1. **Full** (with `_missing` flags)
2. **No `_missing` flags** (do the blanks leak the outcome?)
3. **No `_missing` flags and distributor "Blank" treated as unknown**

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

FC = pd.read_csv("../data/clean/films_clean.csv", low_memory=False)
TRN = FC[FC["in_population"] & FC["year"].le(2015)].reset_index(drop=True)
yl = (~TRN["loser"].astype(bool)).astype(int).values         # 1 = winner (US box office >= 1.25x budget)
print(f"Training films: {len(TRN):,}; winners {yl.mean():.1%}; random picks catch 20% of winners, best possible {min(1, 0.2 / yl.mean()):.0%}")

folds = list(StratifiedKFold(5, shuffle=True, random_state=560).split(TRN, yl))


def capture_at(score, y, k=0.2):
    n = int(round(k * len(score)))
    top = np.argsort(-score)[:n]
    return y[top].sum() / y.sum(), y[top].mean()


def evaluate(name, oof):
    cap, prec = capture_at(oof, yl)
    return {"model": name, "AUC": round(roc_auc_score(yl, oof), 3), "winners caught @20%": f"{cap:.1%}", "hit rate @20%": f"{prec:.1%}"}


RESULTS = []
# Random
rng = np.random.default_rng(560)
RESULTS.append(evaluate("random 20%", rng.random(len(TRN))))
# Baseline (b): genre x budget-tier winner rate, learned on training folds only
g_b = TRN["primary_genre"].fillna("(blank)") + " | " + pd.cut(TRN["fin_budget"], [0, 5e6, 20e6, 60e6, np.inf], labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"]).astype(str)
oof_b = np.zeros(len(TRN))
for tr_i, te_i in folds:
    rates = pd.Series(yl[tr_i]).groupby(g_b.iloc[tr_i].values).agg(["sum", "size"])
    smooth = (rates["sum"] + 20 * yl[tr_i].mean()) / (rates["size"] + 20)          # thin groups pulled toward the average
    oof_b[te_i] = g_b.iloc[te_i].map(smooth).fillna(yl[tr_i].mean()).values
RESULTS.append(evaluate("baseline (b): genre x budget tier", oof_b))
display(pd.DataFrame(RESULTS).set_index("model"))

In [ ]:
CAT = ["mpaa_rating", "primary_genre", "origin_group", "distributor_group", "release_weekday", "source_group", "original_language"]
NUM = ["log_budget", "runtime_min", "release_month", "sequel", "major_producer", "director_prior_films", "director_prior_hit_rate",
       "writer_prior_films", "writer_prior_hit_rate", "company_prior_films", "company_prior_hit_rate", "rating_overlap_competition"]
MISSING = [c for c in TRN.columns if c.endswith("_missing")]


def make_X(df, use_missing=True, blank_distributor_unknown=False):
    X = df[NUM].astype(float).copy()
    for c in CAT:
        s = df[c].astype(object)
        if c == "distributor_group" and blank_distributor_unknown:
            s = s.where(s.ne("Blank"))
        if c == "original_language":
            s = s.where(s.isin(s.value_counts().head(8).index), "other")
        X[c] = s.astype("category")
    if use_missing:
        for c in MISSING:
            X[c] = df[c].astype(float)
    return X


def cv_model(X, name, seed=560):
    oof = np.zeros(len(X)); imps = []
    for tr_i, te_i in folds:
        m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=40,
                                           l2_regularization=1.0, categorical_features="from_dtype", random_state=seed)
        m.fit(X.iloc[tr_i], yl[tr_i])
        oof[te_i] = m.predict_proba(X.iloc[te_i])[:, 1]
    RESULTS.append(evaluate(name, oof))
    return oof


X_full = make_X(TRN, True)
X_nomiss = make_X(TRN, False)
X_noblank = make_X(TRN, False, True)
oof_full = cv_model(X_full, "model: full (with _missing flags)")
oof_nomiss = cv_model(X_nomiss, "model: no _missing flags")
oof_noblank = cv_model(X_noblank, "model: no _missing flags, distributor Blank = unknown")
display(pd.DataFrame(RESULTS).set_index("model"))

### 18b. Which inputs drive the model? (drop-one test on the cleanest version)

Each input is removed in turn and the model re-cross-validated; the drop in AUC is how much that input adds **given all the others** (overlapping inputs share credit, so each one's drop is small if another covers it).

In [ ]:
base_auc = roc_auc_score(yl, oof_noblank)
drops = []
groups = {"budget": ["log_budget"], "MPAA rating": ["mpaa_rating"], "primary genre": ["primary_genre"], "origin (US / non-US)": ["origin_group"],
          "distributor group": ["distributor_group"], "major producer": ["major_producer"], "sequel": ["sequel"],
          "director track record": ["director_prior_films", "director_prior_hit_rate"], "writer track record": ["writer_prior_films", "writer_prior_hit_rate"],
          "company track record": ["company_prior_films", "company_prior_hit_rate"], "rating-overlap competition": ["rating_overlap_competition"],
          "release weekday": ["release_weekday"], "release month": ["release_month"], "runtime": ["runtime_min"], "source group": ["source_group"],
          "original language": ["original_language"]}
for gname, cols in groups.items():
    Xd = X_noblank.drop(columns=cols)
    oof = np.zeros(len(Xd))
    for tr_i, te_i in folds:
        m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=1.0,
                                           categorical_features="from_dtype", random_state=560).fit(Xd.iloc[tr_i], yl[tr_i])
        oof[te_i] = m.predict_proba(Xd.iloc[te_i])[:, 1]
    drops.append({"input removed": gname, "AUC without it": round(roc_auc_score(yl, oof), 3), "AUC lost": round(base_auc - roc_auc_score(yl, oof), 3)})
display(pd.DataFrame(drops).sort_values("AUC lost", ascending=False).set_index("input removed"))
print(f"Cleanest model AUC with everything: {base_auc:.3f}")

In [ ]:
# What do the top 20% picks look like? (cleanest model, out-of-fold)
pick = oof_noblank >= np.quantile(oof_noblank, 0.8)
look = TRN.assign(picked=pick)
def prof_col(m):
    d = look[m]
    return [1 - d["loser"].astype(bool).mean(), d["fin_budget"].median() / 1e6, (d["origin_group"] == "non-US").mean(), (d["mpaa_rating"] == "R").mean(),
            (d["distributor_group"] == "Major").mean(), d["sequel"].mean(), d["us_multiple"].median()]
prof = pd.DataFrame({"top 20% picks": prof_col(pick), "everything else": prof_col(~pick)},
                    index=["share winners", "median budget ($M)", "share non-US", "share R-rated", "share major distributor", "share sequels", "median return"]).round(2)
display(prof)
print("Top picks that turned out to lose (the screen's costly misses):")
miss = pick & look["loser"].astype(bool).values
display(look[miss].assign(chance=oof_noblank[miss]).nlargest(8, "chance")[["title", "year", "fin_budget", "us_multiple", "distributor_group", "mpaa_rating"]])

### 18c. What the first model shows (run 2026-11-26, cross-validated inside 2000–15; test set untouched)

| | AUC | Winners caught @20% | Hit rate @20% |
|---|---|---|---|
| Random 20% | 0.500 | ≈ 20% | 34% |
| Baseline (b): genre × budget tier | 0.609 | 28.0% | 46% |
| Model, with `_missing` flags | 0.799 | 41.4% | 69% |
| **Model, no `_missing` flags (cleanest)** | **0.758** | **37.2%** | **62%** |

- **The model clearly beats the baseline:** of its top 20% picks, 62% are winners vs 46% for the baseline, and it catches **37.2% of all winners vs 28.0% (+9.2 points)**. That clears the contract's ≥ 5-point bar in cross-validation (the final check is on 2016–19, with a bootstrap CI).
- **Why winners, not losers:** with 67% losers, flagging the riskiest 20% can catch at most 30% of losers, so a 5-point bar left almost no room (the same model scored 28.1% vs 23.3%). Winners are 33% of films, so picking 20% can catch up to 60% of them. Proposed amendment **A9** makes "pick the top 20% most likely winners" the decision.
- **The `_missing` flags add 0.041 AUC** (0.799 vs 0.758). That confirms the leak suspicion (blanks mark undocumented flops). Use the model **without** them; report "with flags" only as a sensitivity check.
- **Distributor "Blank" as unknown changes nothing** once the missing flags are gone.
- **What drives the model** (AUC lost when removed, given all the others): budget 0.038, distributor group 0.020, MPAA rating 0.009, company track record 0.009, sequel 0.007, then genre, origin, rating-overlap competition and weekday (0.004–0.005 each). **Director and writer track records add nothing once company track record and the rest are in.**
- **What the picks look like:** the same median budget as everything else ($20M), but 67% from major distributors (vs 35%), 23% sequels (vs 5%), almost no non-US films (1% vs 22%), fewer R-rated (35% vs 45%). Their median return is 1.67× vs 0.55×. The costly misses are family sequels and mid-budget studio films that fell short (*Dolphin Tale 2* 1.17×, *Planes: Fire & Rescue* 1.18×, *Agent Cody Banks 2*). The harder question for a studio is picking winners **among wide releases**. Next: evaluate inside budget tiers, or only among major and mini-major releases.